# OCR lại trang bị lặp hoặc lỗi

Trên Kaggle, bật GPU và **Internet Off**. Gắn Dataset chứa `original_ocr_readable/`
và Dataset **models**. Gắn **packages** nếu chọn PaddleOCR-VL, PP-OCR hoặc TongGu.
Mặc định chạy **Hunyuan, sách 4 trang PDF 10**, theo nửa phải rồi nửa trái.
Ô xem trước cho thấy ảnh gốc và các vùng đem đi OCR. Chỉnh đường cắt nếu đi qua chữ.

Chọn trang/model bằng `TARGETS`. Dùng `TARGETS=[]` để lấy ca từ báo cáo, với
`PAGE_LIMIT=1` giữ đúng một trang nguồn. Chỉ đặt `PAGE_LIMIT=None` sau khi xem thử.
Mỗi model được nạp một lần cho các trang đã chọn. Các file kết quả giữ nguyên chữ lặp, ảnh
cắt, tọa độ và thông tin gốc để duyệt. Ô cuối hiển thị văn bản trước/sau và tải ZIP.

In [ ]:
DATA_DIR = ''  # Thư mục chứa original_ocr_readable/ hoặc chính thư mục handoff.
MODELS_DIR = ''  # Để trống để tự tìm Dataset phù hợp với model đã chọn.
PACKAGES_DIR = ''  # Cần cho PaddleOCR-VL, PP-OCR và TongGu.
TARGETS = [('4', 10, 'hunyuan')]  # (sách, trang PDF từ 1, model)
# Model: hunyuan, paddle (PaddleOCR-VL), ppocr, tonggu.
# Ví dụ bỏ sót cột: [('kinh', 123, 'paddle')].
PAGE_LIMIT = 1
RETRY_MODE = 'auto'  # auto | whole | split_rtl | split_ltr
CROP_BOXES = []  # Cặp (x0,y0,x1,y1) theo thứ tự đọc, dùng với một trang đã xem.
MAX_NEW_TOKENS = 2048  # Mỗi vùng. Có thể tăng khi đủ bằng chứng đầu ra bị cắt ngắn.
REPETITION_PENALTY = 1.12  # Hunyuan
GPU_ID = None  # None = worker tự dùng GPU đầu tiên nhìn thấy.
PAGE_TIMEOUT_SECONDS = 300
LOAD_TIMEOUT_MINUTES = 12
SETUP_TIMEOUT_MINUTES = 45  # TongGu có thể cần build native từ gói offline.

In [ ]:
# Các ca từ báo cáo và lần dò lặp trên checkpoint gốc.
ISSUE_QUEUE = [
    ('4', 10, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 10, 'paddle', 'lặp mạnh'),
    ('4', 10, 'tonggu', 'lặp mạnh'),
    ('1', 4, 'tonggu', 'lặp mạnh'),
    ('1', 26, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('1', 42, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('1', 87, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('1', 91, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('1', 97, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('1', 98, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('1', 99, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('1', 100, 'hunyuan', 'nghi lặp'),
    ('2', 5, 'hunyuan', 'nghi lặp'),
    ('2', 12, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('2', 19, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('2', 23, 'hunyuan', 'sai thứ tự hai lá'),
    ('2', 37, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('2', 45, 'hunyuan', 'nghi lặp'),
    ('2', 65, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('2', 77, 'hunyuan', 'nghi lặp'),
    ('4', 1, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 2, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 4, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 5, 'hunyuan', 'lặp mạnh'),
    ('4', 6, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 7, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 8, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 9, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 11, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 11, 'tonggu', 'lặp mạnh'),
    ('4', 12, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 12, 'tonggu', 'lặp mạnh'),
    ('4', 13, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 13, 'tonggu', 'lặp mạnh'),
    ('4', 14, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 15, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 16, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 17, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 17, 'tonggu', 'lặp mạnh'),
    ('4', 18, 'paddle', 'lặp mạnh'),
    ('4', 19, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 20, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 22, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 23, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 24, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 25, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 27, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 28, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 29, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 33, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 33, 'tonggu', 'lặp mạnh'),
    ('4', 34, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 37, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 38, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 38, 'tonggu', 'lặp mạnh'),
    ('4', 39, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 39, 'tonggu', 'lặp mạnh'),
    ('4', 40, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 41, 'hunyuan', 'lặp mạnh'),
    ('4', 44, 'hunyuan', 'nghi lặp'),
    ('4', 46, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 46, 'paddle', 'lặp mạnh'),
    ('4', 47, 'hunyuan', 'lặp mạnh'),
    ('4', 48, 'hunyuan', 'lặp mạnh'),
    ('4', 49, 'hunyuan', 'nghi lặp'),
    ('4', 50, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 51, 'hunyuan', 'lặp mạnh'),
    ('4', 53, 'hunyuan', 'nghi lặp'),
    ('4', 54, 'hunyuan', 'lặp mạnh'),
    ('4', 55, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 56, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 57, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 57, 'tonggu', 'lặp mạnh'),
    ('4', 58, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 58, 'tonggu', 'lặp mạnh'),
    ('4', 59, 'hunyuan', 'lặp mạnh'),
    ('4', 60, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 61, 'hunyuan', 'nghi lặp'),
    ('4', 62, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 62, 'tonggu', 'lặp mạnh'),
    ('4', 64, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 65, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 66, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('4', 68, 'hunyuan', 'nghi thiếu chữ'),
    ('4', 68, 'tonggu', 'lặp mạnh'),
    ('6', 76, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('6', 88, 'hunyuan', 'lặp mạnh, chạm trần token'),
    ('6', 90, 'hunyuan', 'lặp mạnh'),
    ('bien', 263, 'hunyuan', 'nghi lặp'),
    ('bien', 408, 'paddle', 'sai thứ tự đoạn'),
    ('bien', 418, 'ppocr', 'ghép xen hai cột'),
    ('kinh', 100, 'ppocr', 'bỏ sót 木通'),
    ('kinh', 123, 'paddle', 'rỗng dù ảnh có cột chữ'),
    ('kinh', 124, 'ppocr', 'đảo cột'),
]

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import queue
import shutil
import signal
import subprocess
import sys
import tempfile
import threading
import time

from IPython.display import display
from PIL import Image, ImageDraw

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/ocr_page_retry')
CHECKPOINT_NAMES = dict(hunyuan='hunyuan', paddle='paddle_vl', ppocr='ppocr', tonggu='tonggu')
BOOKS = ('1', '2', '3', '4', '5', '6', 'bien', 'kinh')
REASONS = {(book, page, model): reason for book, page, model, reason in ISSUE_QUEUE}


def read_json(path):
    return json.loads(path.read_text(encoding='utf-8-sig'))


def digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def find_readable(value):
    if value:
        root = Path(value)
        options = [root, root / 'original_ocr_readable']
    else:
        options = list(INPUT_ROOT.rglob('original_ocr_readable'))
    matches = [root for root in options if (root / 'images/metadata.json').is_file()
               and (root / 'manifest.json').is_file()]
    if len(matches) != 1:
        raise ValueError(f'Tìm thấy {len(matches)} thư mục handoff. Điền DATA_DIR.')
    return matches[0].resolve()


def choose_boxes(size, book, page, engine, mode, manual):
    w, h = size
    if manual:
        return list(map(tuple, manual)), 'custom'
    if mode == 'auto':
        mode = 'split_rtl' if (book, page) in (('4', 10), ('5', 10), ('2', 23), ('3', 23)) else 'whole'
    left, right = (0, 0, w // 2, h), (w // 2, 0, w, h)
    if mode == 'split_rtl':
        return [right, left], mode
    if mode == 'split_ltr':
        return [left, right], mode
    return [(0, 0, w, h)], mode


def plan_tasks(readable, output, selected, page_limit, mode, manual, show=True):
    images_root = (readable / 'images').resolve()
    indexes = {book: {row['page']: row for row in read_json(images_root / book / 'index.json')['pages']}
               for book in BOOKS if (images_root / book / 'index.json').is_file()}
    checkpoints, pages, tasks = {}, {}, {}
    for raw_book, page, engine in selected:
        book = str(raw_book)
        if engine not in CHECKPOINT_NAMES or type(page) is not int or page < 1:
            raise ValueError(f'Target không hợp lệ: {(book, page, engine)}')
        if book == 'bien' and engine == 'tonggu':
            raise ValueError('TongGu nằm ngoài phạm vi sách bien.')
        if page not in indexes.get(book, {}):
            raise ValueError(f'Handoff thiếu trang {book} / {page}.')
        index = indexes[book][page]
        source_key = (index['source_sha256'], page)
        if source_key not in pages and page_limit is not None and len(pages) >= page_limit:
            continue
        job_key = (*source_key, engine)
        if job_key in tasks:
            continue
        # The selected alias is retained in the report. The source hash prevents a duplicate retry.
        identity = f'{book}__p{page}'
        png = (images_root / index['image']).resolve()
        if not png.is_relative_to(images_root):
            raise ValueError(f'Đường dẫn ảnh ngoài handoff: {png}')
        mapping_key = (book, engine)
        if mapping_key not in checkpoints:
            model_book = read_json(readable / CHECKPOINT_NAMES[engine] / f'{book}.json')
            records = {item['page'] + 1: item for item in model_book['records']}
            if model_book['book'] != book or len(records) != len(model_book['records']):
                raise ValueError(f'Checkpoint sai sách hoặc trùng trang: {book}, {engine}')
            checkpoints[mapping_key] = records
        previous = checkpoints[mapping_key].get(page)
        image_sha256 = digest(png)
        if previous is not None:
            prep = previous['image_preparation']
            if prep['dataset']['source_sha256'] != source_key[0]:
                raise ValueError(f'SHA-256 PDF khác checkpoint ở {identity} / {engine}')
            region, = [r for r in prep['regions'] if r['canonical_region_id'] == 'page_000']
            if image_sha256 != region['variant_sha256']:
                raise ValueError(f'PNG khác ảnh OCR gốc ở {identity} / {engine}')
        aliases = [f'{other}__p{page}' for other, entries in indexes.items()
                   if entries.get(page, {}).get('source_sha256') == source_key[0]]
        pages[source_key] = True
        destination = output / identity / engine
        destination.mkdir(parents=True, exist_ok=True)
        original = destination / 'original.png'
        shutil.copyfile(png, original)
        with Image.open(original) as raw:
            image = raw.convert('RGB')
        boxes, effective_mode = choose_boxes(image.size, book, page, engine, mode, manual)
        overview = image.copy()
        draw = ImageDraw.Draw(overview)
        parts = []
        for number, box in enumerate(boxes, 1):
            x0, y0, x1, y1 = box
            if (any(type(n) is not int for n in box)
                    or not (0 <= x0 < x1 <= image.width and 0 <= y0 < y1 <= image.height)):
                raise ValueError(f'Vùng {box} ngoài ảnh {image.size} / {identity}')
            if box == (0, 0, *image.size):
                filename = original
            else:
                filename = destination / f'input_{number:02d}.png'
                image.crop(box).save(filename)
            parts.append(dict(number=number, source_box=list(box), image=str(filename),
                              image_sha256=digest(filename)))
            draw.rectangle((x0, y0, x1 - 1, y1 - 1), outline='red', width=max(2, image.width // 240))
            draw.text((x0 + 6, y0 + 6), str(number), fill='red', stroke_width=1, stroke_fill='white')
        reason = next((REASONS[(alias.rsplit('__p', 1)[0], page, engine)] for alias in aliases
                       if (alias.rsplit('__p', 1)[0], page, engine) in REASONS), 'chọn thủ công')
        paddle_no_layout = engine == 'paddle' and (book, page) == ('kinh', 123) and mode == 'auto'
        before = dict(id=identity, status='missing') if previous is None else {
            'id': identity, 'status': 'checkpoint', 'text': previous['text'],
            'hit_max': previous.get('hit_max'), 'quality_issues': previous.get('quality_issues', []),
        }
        (destination / 'before.json').write_text(json.dumps(before, ensure_ascii=False, indent=2), encoding='utf8')
        if previous is not None:
            (destination / 'before.txt').write_text(previous['text'], encoding='utf8')
        overview.thumbnail((900, 650))
        overview.save(destination / 'crop_preview.png')
        if show:
            print(f'{identity} · {engine} · {reason} · {effective_mode} · {len(parts)} vùng')
            display(overview)
        tasks[job_key] = dict(id=identity, engine=engine, aliases=aliases, reason=reason,
                              parts=parts, mode=effective_mode, paddle_no_layout=paddle_no_layout,
                              page_size=list(image.size), source_pdf_sha256=source_key[0],
                              original_image_sha256=image_sha256)
    if not tasks:
        raise ValueError('Không có ca OCR phù hợp.')
    if manual and len(pages) != 1:
        raise ValueError('CROP_BOXES chỉ áp dụng cho một trang nguồn.')
    return list(tasks.values())


if sys.platform != 'linux' or not INPUT_ROOT.is_dir() or not OUTPUT_ROOT.parent.is_dir():
    raise RuntimeError('Mở notebook trên Kaggle với GPU và Internet Off.')
if RETRY_MODE not in {'auto', 'whole', 'split_rtl', 'split_ltr'}:
    raise ValueError('RETRY_MODE không hợp lệ.')
if PAGE_LIMIT is not None and (type(PAGE_LIMIT) is not int or PAGE_LIMIT < 1):
    raise ValueError('PAGE_LIMIT cần là số nguyên dương hoặc None.')
if min(MAX_NEW_TOKENS, PAGE_TIMEOUT_SECONDS, LOAD_TIMEOUT_MINUTES, SETUP_TIMEOUT_MINUTES) <= 0:
    raise ValueError('Giới hạn token và thời gian phải dương.')

readable = find_readable(DATA_DIR)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix='run_', dir=OUTPUT_ROOT))
selected = TARGETS or list(REASONS)
tasks = plan_tasks(readable, run_dir, selected, PAGE_LIMIT, RETRY_MODE, CROP_BOXES,
                   show=PAGE_LIMIT is not None and PAGE_LIMIT <= 3)
engines = list(dict.fromkeys(task['engine'] for task in tasks))
print(f"{len(set(task['id'] for task in tasks))} ID trang, {len(tasks)} ca OCR · {engines}")
print(run_dir)

In [ ]:
import base64
import zlib

PAYLOAD_SHA256 = 'a7b28a5a1a849c03ec947694717e4d81bbf1ab5e65a59042873773d48037ef55'
PAYLOAD = (
    'c-qXpX_F&Kb|CgwqI*pOq6&!w@~SLUqnFuL)jh?jt}10!H#tOsu>l~Eh)Ey<g9zj?$uuLiq_|^78M!8DO}kvlc5Jjpnu#>7X{oE5B0tQU{zu#H9Q?$;pm+Hd'
    '5je8CMpBd0o$&DEx%>0y?>&FV&4-I9Ut9;5>Eb+EH0ReJcDxVAi*)7%!FZW37f}#+@ob(haxYAhG!OGQO|rMi+eAu7VIB^rVU|UiE}MkeWEv0j@9$?RG@x#+'
    '>Rz#jhBuC*=}2nM!yG$NwO+wb`I2AH<K#?zI7zPGCg1tu#rM8Ac@exiefjyTH(uAP{rr#r<Y#~UFMj?n{_y8N{>jh&^lyLhqd)vl|Hohc{4ak3CBOJrzyI?e'
    '{qE2I^^gA3-~IUKzxP+a_;3Gw7B0?5>1FcsfAcp#`<p-h<&XdT7yr{&|L`||^s_(s-=NXo|M-9X#lQa1fBfCwgFk-x`~T|i|LLFq?4SKtKl{7?>lgpszlAzb'
    '8CpRflYBPS?SJ{Ve+%Dubm-z|fAAMj4wa^19$ghi`S~CH=of$RXMg|S|G_W+>g&J%@mD|l>W_c%_3!`U>p%SE-~H(?fBZ+k{Ij3@{D1tD|M<KAW$kV9{N#Ht'
    'zj$L$vecBT!-Qc}nCCBk^v_@@o+tP5HW@`@FC8v|SvrcQL3A~oE=SR*9wldS5;eR*nx6Zew}}V;y!{aW?@gMVeYW&s*s&y!62yRTI=%LmNq7;)({M12ym7kl'
    '@=4@P(nb8kG|9uMH;AKzdTVkw+$G@Ka*=pEn%9L%<k0=uvIh7yqFt_M`JzrxG#Al)8V;j+?JfLMYk0NYn(sHWJeoC!VHS<k>8S2gCpeg%U4RG|z+d5Vn%C0~'
    ';)t43JcfzHNtTBRbeuN4dKTpkFJI25k*_LQfKKBKy6_#~5kM^B(S~U<j?V%_`ucEuMx!-6S=WvS-S?nD-}iPt#+EV!Vm)l(&3VVWpw&BXz#nm9pO9vljpAXB'
    'z52}<W}U%oVFKqKoJ6cEgWL*q;U)Blb8n88vw2p>t^krG!*d*F!#M7K8cs9V<P1(ua2{P}-8YLR{Fb%qIxFW+S`KwHn}qE;4AqR1Az)Cww#>&n$G9((=xP+7'
    'MX+A9HovB4@;0IWcrPEHKBmQT(ChIxU;W#qH~HybBolA&fB(C0zW&o8)@*v8fAjT!9eeo#%l-#A@c-eTfAjV4ExqAS|2?4EH(&o#FZ<@}zYM+no3H<8T#%@Q'
    'Ym8(aTy;DxlVx(f3=vHT5t8{b4@U7qe@A%bBX*T_X!B|F;4{&tx-3mF4u^TVxb9*NStq2Sc6az&x?3!hAPHv%6e*vFuw2E$(LC<pNrR%jR*TDK;Z=~uh)(>W'
    'y>~2Mvgm9EXOif$?kP!=NDUo@BcPTz(P-#Yx`-BeJPfD7BA?dEodx0I3}8b9TseFEsIlY>8zG?c_$r!a3e;BVyFC&2>vEm}K=5%qO+%VzQ&-5+>5||A@L|{v'
    '?$H#g5Jt-?5k;HDi37s#crQ{UHS`~+Vgc5QieWAUFBpWw^C%hV)eDz-sxWB=+ZZ7Bz*#Turt=YxTy=$ua1n=zX$@o)a5tPfm4=IS9>DEq@aSxjF6Wgaje$fX'
    '%vI|oy`@CWt6+k*>1A*hCD8&dtdW8)t6X$g#U#26@)Xub@A^UO;gO)T1Mqm>KB`%ppg#a8Lx5)YJA6oi0NB-nMRWmV+rnP?iWgZpJ%i=SC$j)h+Nm^$3nW}b'
    'qaCQWbJ13(4w#5}X``TOl*hE!^9WWxzjn~@SY`uo45KP6n1`d$G%A5`(QY2OHSTt<_;!T(&(k8r@`>-C;>T$`i}RIujf5x2<5`q0bG2prir6H<*?d_zpv)i!'
    '^Xq(4-UR0(@@;pdoP*`)3{ZArEqJSWC?pA32|Cl7^c@<!1FDdRi(F&fp_;%nytZa0bKt_DVzj1P!$lB>>ynG*3RzEm8t@L~4`YC6Qguq8d^w8(<X`WwI11;0'
    'r3vO~mIrYX=Rp80%XCcq1n#KCvjA^^pPD7Ir5J`A##rE_ve7)m|Bm5io&=`B9u`J`9*f$o)id)KaTa-BhSO#Abg@Vm_1YJ}TFh}NqEVSt*8DPH!uAO?b$unn'
    'n0z|uuQ?o7{6jcH_zU=%*>%m|1kku91DsEO`q#iQ0GIHSxi|gh>p$#xHx>Z5Rj?7EGN6s;`?MBBvEaXhG5mK62)LV-0MO^^@#nGk5g@Av%peu0yRuwr=q-KQ'
    'xKGe^L-P=g0?GE~lW)HIN$h>>wW`3fc>;q*#9=f%jZQ<joo7k51<pAA&6Zj24RF;_^uW$=o~pA7XiDsi0m<QK03;)d1k-#QY%ce$z_YtX01I7L!dYO4ypOCs'
    'EHFdrTOH@p#x1pL2~&i{2mTh5z((FMoM#&Rj{!#>6?iQD5!SBPZg>a%!ZgC<x?aCPsSlx|8w4e^3&uv2SV&x{Gg1~fuZCwA>qYSW%@d$D9@UqN!At~bAl%Rw'
    '5iS-&RJHO&P)akXgsfDndR}lyswcMHwCtW}PB<h8aS_nZVwZ)v5t<vpwRALzt!NNi;os>DQbWDMA@zU~{^Gi7bqkEBIeo}c;HhJdR8s`}D+zJWwm5*r&h@rK'
    'al@mY8$uiq1Ek-TPzu+oU<8&jc=q|pXHSDKPfnkmym<5aPWPMB=jK6aQI<(Oi5JACqzhW@+AU)j!o*kg3RZjbCVr;c><B0CskU&`f3O51wvvx0slNijta*qW'
    'VVwncejnIEiY9C&X=n2^o?#OW`fip0<(lA*jgXzluk~b{zH1J5Gz^E62!}JAk<qj;!$yh{RJhc%;E%W|LOcGcgmnCyxM$`{45LiAjJ&aGsCnK}0W!hFGzhOr'
    '9GBu`@rTX5a=*-b(%>$iN(>n>oWzbGhe_ZXj<R;U4GY*+@`~OqVU|9ybR&iBl67kWZ&t5u;_wO>WXqRpdM#D9dj&cet!TWQ11d!Io0^Lkurehitld@|*GkEe'
    '7O5yTu7kd^msRVs>NpmTPl0Y#laaL>wPKvDPW*dVZe6ITr7qC4z1&bKES${}-J`8yMToU})u94;8j!kF(*XQc*63C!Sg8eerJpQd$x*wBb&Z~P251qZf;5k='
    '@;d&DWW)yD-SlU-hH8od$T0;X1j~(;%$`nI>QUA0{j6pl^lb<&5n=Q|?f{6u>Qi~{wOaH0X`IySL*n3u2n)_bp~J8lXW1N8?!c-!>fd!O)5WMh^pVYM@9($w'
    '=+3Yfu!&T)K7`rUP<f9e?-Gtny<U6#>)-Xhc=7DhXHTDa&z?Me`Pu2otMBaKy5N_`AaI|&dG^V(=g;1J&;ITcaBtO{(g2ogn5*|7{uRL0G|8pH45+dC9A{$`'
    'q9gf~Arq7VbFZJHX3+?6R&Dwe2xNfbK$x6Oqxw9ZUZ16j^_WoL%cMfXtD^|N=3T+)VxW_0l<2N;2M=fo;Xu^uR^J!UYn(u*q_^;W!-T80_BdC-)L+B@uJ+)+'
    'dvNnfCDCBGgtZ96M*sqJPBK7ohQ+yIMQZHhO|nN&%~wHHxdOJtQ5dZ>XXN`*vyJr&zciq=F58XGO9e|*pCN|Nd#!%OEZ~^)*LKxci)iT;ufr|Bt~>s^t~)_6'
    '!31~8EPlD$?DGm;wec`s?cv|o^!N4NIuw+GVrTlba4|y82Gufw$maM0)bQ>x>FY9I;l3K4D$=c}rgs*lGkk4+L@$MkdGvsF8=lWoRMjn_SqNJX#3N6aP(ErZ'
    '`hJ3{0pSEQFU$esLLi6vW$GbY1@u2>nLVSU^Pu&gdFfb6fq5TJ!bJ#wEi(1^Q*74szDUlK^fK9<0SUzSWd*Kx9z`Hb>#DpQP472=hQ$J2=w%aR8ALh(bX|lb'
    'VgWe7x}s(QuBSQt&l{w9p3a)yQ(zXZ4PXH77_LD94+OIG0=h#w>BYn$$Z8GJE4ckGqQtwLz`t-MGa#?t+4Ooo@zUXNxyVSDjW_@Fx@m98imOQJaEb~PJ>w0x'
    'lhT{ex)@0kdTo?6C#Fw@;mJ59wj5!qb?i}5h=j}TEE+*M=~8c%ZaV=a=;DKC6IkSW4KJdAN}*A&UvVXxUKItQ>g7Yhh>cJw1w8bPgY}B{KXzb-uhI3!ClkNm'
    '($Wxp4UD=%sYfD~#V(x*Nsp^Qyt#J|p(IHlgdTA1=(eK803?Q6Z#dE16yOc5S+8)TnOO+!*)Rzr($E>=C@D*@F8xk;jJrV`C%6Vv*d5xxKpi>_w?ZHUJ*L6p'
    'Mu||-TX+KE5c0cq1_OZwD!d~Eu-!cL?m<H=xoZ1;^@NgB*C0v(v7XoA9&7N9`xQvU(i2I<c;P9~@VeYnXvIAuLl0e2HR`Sx_jy@kgjKi1rmf}!fQ;Jt+u&n5'
    'g=xawhz}dh1DqAJEVM1Xy*^A3|By?^-5TzKG%17(vJ3<mkb0_`k-h`Qg_B1UAPm3@3wZnPG@K1ap%-_&9bRad8IE<|)4%p$s=MBvJsEu3i|{c;wZXyMc3dKD'
    '9lk)oC6;KA)%j;{r%mm5q;DVfl6q|lx4zY+-EW5TIS8+H8gad>$?a5wI*$YthJjVf$#kzP#k81#sH<>21^lWt7L}H}u6iBYeTAmXG&kgqz-8?98(zydyT`0J'
    'v*b2a<Cba$+}@hAba=n(DNL1*_&iV^_2$+cRnLc(z#7&xm$ib{_61ixwyG>6m`l5Ws;~`kb;7WkUh1j=tsdV37Ch;61bt9DZoq@D*Ret)Jb5{>RM>;4>G98x'
    'yL@5`2!m>jI9b|!EiF;k5liZNWv3&M6TP;nK24d{;TSufvxE+)_Hy;8QHPt}_kf@5I&-DVq{s-H#Z|}w<e3@)eU{bfRP5t@22qh`%_Mb^yc4bB@qo6!eVZtu'
    'kw^&QG=J<HQb?GC#KpeEJA6%Hi$cvlGX>6Ochqq*b8^lU&8oxy>8ZK({acs!RWoT=yhxEFGVBPSdAP#KKCDD5{f1n|R8Q5vwWU_z6sL-1718r?x<mq_l@N<<'
    'rBr0hf3OUvaef^D_qvQSI!_HeAX-+Xa!*4nC&z8Z2gh7}y4obp@omCEBW|M5r$(=d0c%dx<;{u+uUn}+v4)A7SrjgYlX|V*ylFQMTDSk$N1MO)O}pK=1*oJa'
    'dwAk+plcGNW=(_v57UfQ40XP4XGA8VCWuZWkewDA2k?zCT+5{nJRThU<KlyBVIh<y$pvixGM_JVOGiU|gA7&bOYyaptftGAVR18?!`4xwx^Iz(EIvy@wB4n?'
    'aBY`^dac*mdH4^1^Cvt1@HhXme;<*#1^+5{%rFR~{295YJ(v7gL*TWHTOi0=H=q3G1xg739QASZj;lsfx}FWvX(pUnU8$<VDKLGq)1;BoyXZnIt!cga4NyvU'
    '3mXC))!}+Dxn(d)XCRKDa@7S^S%@-00zy;g_P!RT=|!P%Sb-e}0+E%D>J1wW@`<H7YYr`~0ycnSPjwmfsb|}IRDCoLo=U8WlCykbjmtHATA$c|_aV%y7sK4c'
    'UVzu5;_^7K5sE&WAsIeGQifCwY3<lN5_W-5j~b^SBLmaM1?9%1z{Dy@%?YZqS`0hYTs|VZlza?g#St>#v8uF@A3)=EzT$;z=XUPpE_DpP%G6O<*hh|EYE+^s'
    'FRek?Wi}@zNMOlY<r5Z7d9pan$27-%GfqQ;bRTU}{cXnOzHp>z94!jxLlDe5CM*(W1vI8I+%!3b4vx{sTkA@-^}OS0n!>ItrCGzvez3^bxfby0<TnX^dLz5*'
    'j(%UjJAn>|c`Wk|+7A!*TL;E`spGX<2gipGj*R(|UBsqMc00Cg(j|iQVj}?e=J>3_mV^{33{Vei@I#&v(y^>$;edE#Dw9B4@#?2XCY%;yj;j9L4Rxv(%v%iU'
    'TcqD-mIuCN=XF|a;DYDG1-a_OlO&v8XK}WZCR60~&pmAN2slz;%DiP3q3R3%E%4F}5GHAYSi1Z2@u`PTO_^g3PJ93A*>kn;&v<zH&*KTymcKuZrzGalkC*dI'
    '3C$sLxdd|)&NcFe_1g2AZ$3Bk$#OPG!gyM*w~tz_hKK+5rKjd;l!b8q;^eIP{N%;s-+K94tzkHdmzl6Kd#q=s{<8Ro5z52IiWR{%fO-aGyxVhRdbz=vK(AZV'
    'XnqzhvkVB$C)4Gk&Lbg{gVo(FRMTyunga>fB7|!PNA#Pdo~;W%ebMpu4i6E}E`U_X?(xM0`xN98e4+*c2JWSUmQQjuNsT2WI1A^QLW1xr&Q_qpr@*_B>&XHY'
    'pwskn2hrhdnIotF$U_qi7%_t{$(grUP9vfip&*cl2;|^_EaC=i6w}s*$2?|sPP^eaZ%}ET4z6=#a?~X{M!p-xJoN(5CMmLh3G}qpZ0)O;giDnC>`xCLXuW@)'
    '&KsgBF7z@f^fEEMOp3h_MIYyQk1#-}C`jk{oBJ-So0OQnNH6IcC3Xw9*)sXU+4rttuuGH%VStYML8B;EL?m>SjI8M>Mis8(coe8%uDJzT_bl0X=7;gTUT)kV'
    'z=-~HO{G4o^8|MY9>4Fro|vIl&v(e&CC}HH;8kDGo@QivlX+g5d0LzA&Qo>I+M%n;dD@N1%&BUg#I70$swRppS5FjGxNy{n{9oCGpLz9nScBanP{mcbw1pFp'
    'TOxeV6IKI|n<?sLJjy2x@6xY)pWG6-D+3&@p9>r>X#q7S+W<F-a94m7_<B<X95{+Z2#L8wJ4@IlB6x=X)Ql&$QkrK)B~jU9lTaXv0g}-%Qgwa?fk9eF>IMLI'
    'X11J}%VGieF{%s1_uRhF+z|UK^`$681bmKkB^{SKga$`iaU9|$N%FN)LLzf-YXr^8V&8fxcX+7#UfX6K2h%V?QXB(*yA{dQl-3J%K&aYP>*a9D+yO~+1e82V'
    '6!=pGa9q;%RkT+UOOeL~-cV`eaHD0p3blxoEO9JeMb`Aev}00h!*Vi;F<!!a{c`(k9wzRdK(qa-ec3<sE7}ht*a+NJSwHNE_kw}hp%MHJ{py$7hhtXNtL5+D'
    'HRJ3KTsJT!>ZPK+m`utPDgiebRajzu-I(KXPfQBv>fyPuK0%E`ySJeo&B<w2v2hB+l?VeH=~p9C>E#@oY46iHRANV>L3&lE2V3YWQ#_6~FIOj_WT^p7lmp|4'
    'sZ7JW$6d(#Re_8%$&uHtzVSGjBCx$W)W7eAG6?=X;8!zT9w;8DdkBYX(4f6`r|<VGN9oDq8+S=CQ<s~&b5@MO>vcLi?b0~7qs9ooTl7_AIJg7NRl-3ISnkj)'
    'C5R$qqJSEgZmZHd0nME=R8i**PYK7K-aj-Uzz6mP%pt<pee8)<9^pvL0Bpt*Wgpj3M^VP*6ea#T>?8cYbK%(+OzBBH$|osmn`@VV(ovvntS`ew0#XF&$w?ue'
    '!usI14!Sa-{vLi$!+GJ^Tm(bM0i&zDMJrw&dkA*ld3==HMFi858M);>7>#HOY!4A2@@jM1LYY{U1oLPiJ7^4KX+OzUupn4eU6bYO&P*It42rXdQz9U)Giw;v'
    '#301+id8Fs?mR|$QSRB-E<QNT;vHYH3^%tveM8@JLNmN78*VeZg(9GA1YfZsg%@i}@)vu3<!&Wv3NcJ4aOE1cjF4!QH8^74_6^c<Bg?Sa!P{Hk_TxJfv_d(?'
    'WnmmVI683Nu&WwWqSL<poNo>Ik=I6{h&zKH>#Z=YhP0%mk!+8fcCBc6K}%(hqAYX;zfpdQWa3X=fvfEDwy~5`K1MB=>jlo&vLwSHxD`S6LD$yK@yCs}{1$@h'
    'rC28M4~YLAqQ(vltJLu%F&$}Q+qX55q|piF*z}lrxPt54BAb?of%VE7TZEZzP1Bh}cTq<DfFGV|YN*$|)9mo|4(|L2P0{iRhQ=T_2IHn~&q!fFfipmSXm~b+'
    'aQP_>TEP{~RuIEE>#1Xg59@%K$rA2H<u<1zs8#e{)gyFRp%K(HqwEw{x{qzV7~^)i$`xD7Llbm0EY{7m#aKnoKAN(DpL*6r?O+B->;59Be80!%d<V{XhyLvQ'
    '>+L_R?x~A%2c<czA;E}k3<*6Y-1AItD~IhB7;N5rO-TFJ>uVZ}qsu_lVgVcwcB;6_1GYeU<ef+J+#^<kIGL?cRKarx1Xcb5jHX>+V1NXXWw}PvhBnVsKzDg{'
    'il|;O+ex%s(1W<#E{_1=qKi1XWa5?b1L>{_S;V0>lx&321<Ix$G<s*t^-fZDyhkr3b(QlpR<5&ljf}|?sr1y~`ZA=!AU;Fs!9HH1jx1VwHR!+ls-WF3yHKEt'
    '5np!93qqaXK-GzQ!o}Y4QA?Ee{GOKJJ~uRiumP84WeGcr^W=IR2cwV<aCvi{&g+Hu^c7Pio@tTJjjkPjP$jocDDd<MCu`R=5+ks(aMO4RAHb}sh^W2@Fwmgy'
    '-+|_+X5`ocZHtC9@pJ?df+H3ha=2j024&DI$KF)8ht-eqtYj5r6Jamv#w#T5_=m&+nu%5kZr0LlfTCylcvv5x(^j*M&yO_VM=&Be12>OVS-(=KH(jg{x>;j_'
    'Y6er)vEaGtkmcVM>Mn3-v@Y~l^kufAN)YeCSbepkn+TbsRe_4<C6$mKP0@F;J-sjQ#)SODUEoSt3}nShNbO#$)52R_bzQMnrAOtplxmrfok)&xx@7o(eU3As'
    'KBbF5wKb<s_$ry+$1A2%=9x~JFGlfP9Gx?}?IXn(Mitz&Z6dDj6KU5qz@XsH;%}FPUy*$|(pBM+QfQY5wCX*Iy!~WM#i+2>MidAv$ViW!HIIoR@vWg=+Kyk2'
    't9et}d^>EC#h6%t%#BWV*l|rvN7SG`3DW5Oy6P+Nukc0N+oRVi1`7zT@QJ!d>kI$t8_(8kn9d_h*c(jK;d!R5w>i{+oNe<mEHMc9QC^;o;ZWBt3U<$?SN4k*'
    'MUS#9gRBu$r?`T<bff7twmM)q3gM(!i^y>n){BAT+GAoX5OK0-aS=&i1@8uS`L^TT=rM0K#?uUU2`=}<<>BHivL6b`Y{6Xzf=c-dxmDz%({;=wrDKqB)b`re'
    '*o>gm1@CyB>u-Lk{&uLwqqvAxh7_$L(#wQEu&SW7_eZ)C{=(->{?v9pwfhCcH685Q9W<=wyLR)wD4-+~Lq{LZnzCs?qf?5dQaT9Y8y(Q>Yq!3o7Ctln4Rv5o'
    '7opG@vrx*|x}l<Vu#vi>y8JwxduLIyM3Iel((--6!O_VGwU#*EBZH+-6fcm$xQG%|7?e#TU<TJZC}$MS@wbY11i%8us8WXBU|UU&J;x-3Pmb$7{O+k5WC*?>'
    'Z7MGam)8rm6#0O~$<XSe<lBJHDQ4MdFY84f*sVfLvBh97!XyJOXI3ZM2FG%%%B>UYtnuo!4hPq*@U5g_-p##}nywhH$Da|qfe>7=M+L2jd}z~$Yx&SNANE$Y'
    '*e|r`m!Fe+SH-s1g*NJ```S#Q%7&kIemg6EvEWvYN{CBl8!dx(<H=NNtE|7#?V#w8AXV&>M`PFp>nBk3LwmolZwVMh6yq3Ha-5J#$%58E2G3&@Y(%H(DsL*M'
    'MI9FBW5ES|ubvNnEb{8x&qqazMOZ8YqG6O}=^{`eLh3?u8xrx;rI#-UOaQZVoL{0hCLTWBiNqElPs?<vyvX6^+l5nz{5S{ZsjwvDnwA|Q5cZG*25^YI;W_CX'
    'y57yL($&vcW)<zbggN6M{O=tJFT%fI>5zO>>+RW=@Ro%;Q~)i86c|KrL$shBpUs9BJE+LvA0v+1Fq<t<H4-@^*x`8BcI&&gxd<9f&O#9Mxp22#tq2>6k_9X;'
    'SX1I=)bWGwL~TJ$imvvGJ9v@ipQ5UU__4JHx46qN4j^KfV^Cp*CN=y}bBq`CqnnCt(m|WoNVV~s0R~K7PB9?%eXrK6!M}?Y&@3CL5CpV?JX_=*<WGE=BV^<l'
    '9)@eSP+H8~vS68TUEqLlhiEFw=aTlOroWqKE>)}QQGw(`JrbW-oB$TBo}T(@=4N$CGKx7$W~H%ANBA&fIxAj{Bu>civV2?Yv9BT?<7v(6R`BCO&8CVDVmWvK'
    '>+hG*t$qSChzHA@8UBX%2G&9TG;)AnsVKC!x6cts)y_%`6Kh)?5{K;-g~LchDhLg6Nd|L8JeA3X_=4;{b2l^}8?ae@mzLap{9Y!=eR%3Ii;gzG$zWjbNBjaF'
    'stW8}$cDEA=o5)pNW(cqQFbnmX7h18jkb7mwwwY;>>xp(Z$UMC9qy^ZHC7g-3Cl&#re;ypu9Ew_0;u^WLI}gf^%K(dq`-y4IOF)LTf4|-JF*%(KoP@C><>~0'
    'KV{s&*&k(eP~0Ng7TOaIU#2_kQjl}~(=YE>;kOYXwjWsV!5VfwVyIrV-eTQeSmcI3b;=J3J>rq;i(V~Dmy2N(5SPGA?SeQ$X{-1E;QB2#lhyj{RcLC;Q*z&Z'
    'OD55>T*+33`Od?%o)J<x)6fJb>fcscA&UA)!?x1sL~pxUs^Y4$O^NmJ@5sD1w)1e6sy&jRxTnBs9`G|qV4u22qRG$#Y}NByF~;_$7tvSz=85Bbmw0Fl4C<+6'
    '8yLicNIdTwaN&bgJ}SC9<Ns_2WNrCSxr0n)I6~mI1`rlT^A)R>-aNK;i)Ib3T2D@=&^fPOIC7}5(EF_}DBXaOq0pf*jksBel%(7olrlF)e1Suyk5&~l3hR_5'
    'MNd1xnk$Mt9O?9~?xD<m*GQ&a$CaC<w=G2*DmFnea)T?0SI`(OLpYnS$q4xw)9T_6Q$a-|>ldCWRJ^sWr(%&IGZ9b^7F}7SDUlu;*)Km>HHc9)ASsQqbl*_s'
    'U-DZSfT^FN=@begDrf#rVK5v<>BFUUptN0G{Ma+fuf86@CJ<DsOm!Np^+hXtR?ks%K^4VKx>lOI{u-#XTCm&O^ipX7NDYXXt2H6bZNt-(>sgA?-O?*4uAQY7'
    '-7RZ`U@&_XtlmEIior8bG_-pAs0g@Tq^?%@q7rlS)2yCww*ob-)tH2PheutJmH8KbPv;SzNA*b;;MWS`&|P^%M%cn=1iQ8xLYI=#fUj_yyS0y|1wT<p6Bm>P'
    '6;x0z(3Y8A?6n}t9l?62o6QD5-*!Z*8WJ8?(H6U=`=v$?LO^TDJr_vY9FqkF(FIm;=eVXC*(D+p!_6UQOGl>hmBJPz3SO9QuSz{>u#1j;t6D>8;aI~#88{m2'
    'jZ@aqoYE`SDtdz4G7>(;B?3farI@*iJFWfEZF8QS)!fH}W)@xmX(aK3cd}c1_2M&!rx0l%o>}jd7^i@2sEN_UZ_QZ>{6C^>-G&N{Int$b6AA9p)*13uc^Udj'
    '6n3vQLPqzEei5N3VV45-Yf{hl5PAr6zAR>qO=8GdT>+>ASo)%2k~xr&qkeH~_l)xeY6dKR^{cvI<vA+XurX0hd}XYxYdypihO)o->*ew>FIL6FwQRL{F0lXM'
    'I27suK?9}&fi$RDQ$H-2?~CxV)~|Zy=W7U2i2#yIO4FRalaQgJq1rD(Mqta55kE_~Yi55fsxPON%izwmD4g12DX~B?$=W3qSw3kllWCls*Ch^Jke*YxW~;4P'
    '0ULg{lq?=9AU4*iYj{U?^dHegU}zkp#1?u=&sP@4%XxzJMs&%7%ZVLp&3foiy%MSwP20j@j;4y24fPjNAJqbAUzw+{bXJV0IM-gepG{3QM`SlIkZBE*Lzg%7'
    'Y;#c9x$gTT+ad=IfmH7wt+vHM`{Z@~SK!@h`Ou=|w;ivtM|^UkNV*x~-f*&WwuncJN@%%Kt9H}bMp|6+fc{Er3~fl%GkRQjMBn+?yRk6jwnB2X)Yq&PH<~4)'
    '5k+O4yg)W2vZd&LtHNB~sAxufy6@eigK{fzFY996x!(4ZnIP9>_Ri*KwNm6no9>ubRkE~`kWa-(&P<tZ?%mV#R|w*Sa8%)nXkBjY%aQ|qsCZE|e!1zOC}-}Y'
    'j=79*y%3vCOekJOqU{|9$d2}mTo_TKHE-n_G0anW?7L4`)&p4iMbUkuyXy;?;G7<TA?{S&HK@V>jL20QLi38&Bp<+0og4S)Lw{vZw#vv!_En8K6SXH+SS2N6'
    ';{;tPTVUhoIW>QbAmeyuLOEWRh!1_^JE=p{h;zUV6|m|d8d~9CczpWEg>s{>4q&4&0YgoVr{S4IFLE2|I8+rw_Dy@JB5Z)0%7jYvySg69_YHICHqaq1s~gx!'
    ')ub5!Q=!kR0N3AA$>$i9*t8ps=P0&|R^-$BE#>1ZQD;kQ>w;OK9(XU{R`hEC))wwc_YzWw2bf^NK*|z1>2;0>gKbj8dR0*rD6MK|t-7`0RcWz0+p4g1h6V(I'
    '{?K$3wb5FcSxWf=AFV(tR<4{{DfdgKGUSQ_c;O{6i?ksc25qyLxJOD;szYc1^?@GH5c8TnbaWn9=P9wIce94<)zoW1U$U5LWxzRGew;G}R{1<oFf*D25p8q<'
    'g&#J_0-Q7f;qV8`IE(q}D+KMcVhnQBR`Nts`eRKkI*KSjS%LI)grt!pqSgnD7nlg7qiom+n`+jBrqS7U>{a2=w(f;4ds!+smoHzoaDJvSnY$s8y0z_7(XqiE'
    'w4N=7>U~GO_Haa}ipRds5fgUC3&0QMBe5hmtoF_THgM#<dh#hdw!#H5pol8`)W>ljDHZF+C^}C7=d9e4WXpjEjzk}T)@N`zRtYW#Wh5gUH$j5l?TH2Z^z|ba'
    '&Cf*X0f|z+jW|25hPH|Ss!I#s4SLeQqn0|js~TpDcLnQ)N6?!lVsqWq5;ctGvTZ3F$$4~1(E+=FXn6QY5b3H|GR7Is+uijrPEm14VmD+o?>>eg;1Tg})XL(M'
    'tGKC-<LQ)#u{}><ISsQYHr*YH9njvuG2WWniV@pdZF)ZCQoo6-o#f3KCh@8B50%q6x`JLSq-WW4)?rCMg;-Wx0}Wh-JW+BFL1a#K)sge}zHTJGT!dJtT-@s~'
    '8d%<2H*3j^BALN~R*9~jp1ywe@&y?pjK=YDwj&|-I2o!Cn9-Q;7gT7HwE?r&147${I8eDn<#l}Jh2t@qpr#T0F^nV14|`SXhWD=S5AR*yhc2#b-2wa>!LP9n'
    '<^PPXc`^E)z$SPhMkU&zP(`erc_h1z9N02pLnS6u!!WrBu`mXvm-~Zam>GzYQ6~A+<|(jdyI%lV{vyF7sH2o4zK4Tos=|g%t}_Z~_vF<xiF8pBDqq~KjFJAd'
    'fw2=74G(G4K*wo7<ydOuzK5-f_&cco#o!wQ)cQil^#w*N!XK<3#;*I7-L;Ra`r9!bTxtcg9$-RHuIO;@!3qF(90+%cAmY5;C=R^}Aa)!mrrgG$nqZ(jTi5v%'
    'iRFGdp`Ppj8Jm!qH5#ic>tJ?!>LJ!QhfzGGI9hB9-|$SSF`_riLGA(+T}@oT#f(@SnUKnTY=N`hA;5kHzi}=9>3U<HhkNSYm|kx&DI?uj7LS9W3ef&A^XlA?'
    '?78?xjzJ*Ir<f9Lv>Zkwt)3agIS7+#oz&HfQ|`*jB0&$xZy~ekVP0k5!o8uv)ErjNH^J^sKl?;m{*~01(rP$`!&leb3r!3zMBI6KP^AqAjs;kN?j8ooE3Z+z'
    'CI8SZ1Bd_YdR>kYkT8He0Cb)e>QFu9aCmfd(I&c`Icq(NhLl!X6Q_RyQ2i5JaTE80T-HUmPlhWz#vC@}!Csjtd7q-HPX+eNfaT^8hG>DgLktUSNkS0}F|mj6'
    '<^U>ybd66-sE)%iDZdADEdV6&aI)^esAnAFiXY{@&jFm#?gc&Hh?nb<{n8~x3Qgn(K;UvxvMIT+Dl-V*BJ*p$EcVb2ivL{0`cW7{<DDbhRn9IV_uVZ8ue4I#'
    'mUh3G(zYUHU_uK(Yj5wScu3Z?^%p61<pFmE5pi!he4=1;B2^c^*L~q%sYKPS>-hfz^x4#*wRJ@)*Y%7yGP!!v!Zwl7`V=ODG$i;e%1^Ko#Zs=@21GNq9~Csw'
    '833AgG=3P5tUq{I*NOrPo0L=YgVM;sD_|8~e)8+gGi(fCs{@2$mpIM&m9wA{ntKsnS#`L;u7Vwhz&|Xcmi}!yJCkRzskk~gmmZJ*XwH_?Jf2Ul>$<T6g<qbd'
    '0>w0_07zVp`NEl8&r=MKNdtxrHS|DTJ<j)`{ywrpW)0ItBJqmlGiINsSexiL;X5uAb+R9U?vZ_0|3&7TLulCj4p4v3-BQXnl?;SEdU;Tt&}yhMP?f%7VD;5)'
    '`D<lc3ls@`gtc2d;DF_ohCoRvKX$0Xz(_XqrG;TS?v>i^n;QFW+qU`A?sK@WR`VFcLk`RXy?UWnrTe<n@tfD?Wfbq3ke(gL5DVB@h6~%eu+47e2xusNvMmXJ'
    'YKd0*2xG^Z?w4NX`N$jjNyI~2(hydOl-D?Eyk%=0SpjlZL!6S*UKv&OUlw}mzastD80{^mz=2Fj+wuJ-{f)UuVQ*Pgk5?Le<Zux+hKoFTAM6EuI|g(+LY)T~'
    '`GO~NtK}|^^K3-#EJlT)*OdoFB53@cNO!xGz%DRR>uw-wnocvZk+$qjN&^*415|QM`sgr71gI4R0;oKuTuSEwEei8E#_KLX?R~10S0!eAP1kJNDfJxXt+_-#'
    'C|s0PU0C&lbOF0SF|BI0KW$->v$b@4g#=_>Gqe!&-Z{wXGa!f<COz6|9htJBQw9YgUHZQK8kt|DtHwql9MTJp(BEpcRjj!S6>Clb%(#z6{sS_HxKk&+8DV{M'
    'jQ_%O(j1}PsZ1FK9h^20i+)8S^Wx%9!(lX|;2@~XR6B`#u!zRf@>AY$8qcfQf|rBuBWLg!DzNTj@cqIQC^jXV6SuKy5fDFte`R&7ODnJee#zGgwaKEsW-}tB'
    '?wX{dhQ){?OM(ZeL8^v8Yd73DVwycHq|@S;T8;p3rx612FPW_8HSnyW+?Pn)dY75pXu|wLbmOG9Tq(6VxhRy!Io|R~bd^(ZNlG1=ph1DKoe5eIXrs<T>=Q&r'
    'QyEaot}XZp2+DWgs$MtjV3@>NngjC1!$2+%W5c%AvP_GU`czRtoj28qtgzZ($cZXK36M5RmFtaTWbGzl#@<}ip|}D784gJFEIrp2<Wr|IfMvblRn*qftYa^%'
    'v)XK<&v5Xl_#}^y5y*9_bz(D8I2_U||HdO?g?t~y<8iczC@!38XcE}qhOyA1-Y<*R&eoB&V@&dGPKv-tJ(9p~x$~oB<oFfe@A1D{vU4~S8+>0m<D(W8|9e|{'
    'Pp6(LEz2xwgA~S^uVRQ~KCbF{jhR;I4i^_Z=2>`c#qL>VZpUy^o*WgChFRDTNZ1#PLgwLo<L;d@zk-7CT85>ulArnX<>}|QEbdvtp0fLMM9i*3brjiB{3m61'
    'G7wYDRry5R5OJp<r|$yCrUlO;fBG%XKw*0ncm_Siie1H9k{6bnno6?X@k)wr+sj8?QMJ{&d9AuvEV839=<7EMI8BPRCT&5DQnv)A*p>U~t5O^ATjr{&_)*u?'
    '++IgBh~zPfJ4~;k6gX2*sE7dwt+o45)B1&nb~cDcqe$EuGW8&l(fJkaPyqW4ALwmPpKAN$mQEc*Ykr_mAgZ!J4jMg)(_iNK*A4HXL(T9<ydh|*$`2D}`%2Ff'
    '98*NuJfN^ao?eb5f*sn@$Rqs>5U9t3z<vxbge<L1(@S-enz0`7yZStxGJ`jybti|P3v|l@w2h}6oQzjohg+kVFLBS2?CdO|LOm$6<%2_!BzyhB6S$#~p*#kC'
    'd2;&f<i(rd^{c0kU(=BprI@;M0rz+kF5(}meM1F8Es2&3vUbYyRNFbPD1DL+pC)fkVQkbbqLUZ=d-vG;&d!r(pFDoK*J>YoU&e391|k7oZ>dA`Lj6U8-mHrl'
    'jWLOu#1~?(UmHbdVe;*vd%E-4$qNJA7dtPa+-tY?4jy^mjeyvBZ_t@tndUySf`vC99&QWeC>l(c3)Bj~Ul={^xDwQDVY>{`(IQ;n;hUuwbW6V{5D(jXp6=e0'
    '?jPY}+cX}=xwbu__l(W(YIAJ?u)@Q{F%Hpo{+61bwaUT@QLn>kIEhr3YrRfN--qc`B4*;Ph~FPSKl#p6?|C@zo^zP$L||iw{Q(a-h=)t~UvRaD<L+%St}x;7'
    'jct-0@_`jz#TY}{lb${3d5hr>aQK5?#qbAfh6lDF1&M03{hDKaw1dHX>A-;DF@o|J&p$USO{vUF7=sCm7~<)27QU}dMr?xqVsdq|!Y^(E$Ko#Y-VDZRVR++W'
    'f<^<DyOagO*&>RP_X)&j?*Iglt>}Hi@X6LNz%?CA-`jbA_LNX?D=ff~!}mOL=nW?zcVJF}@GMEO^5*Fe-ZKz`y8^MG%!23d69Lc%ixX_(p~Ng(>p{R<@{D{I'
    'f&@QICA8do0Z8Rf^UNr)HW&14yoqLI;qn3~>$adc3=sT&&CNYg7{aP>j^|m5ip6a8xCuyO=!y1@<G$eLz*C0F4zyclJLfxD4jhwG450XH)?+mw1K^}#>~Q*R'
    'F9_YyrX^n{)bQJmr&&W-bm?-_$iR@tNpuy=vc>ztJ%l3KVbOS+o_VJ{1W6nuk3T{)1mx}UrBs|)w}Nmu|5d>F`RmiiuO4A0l^GlkJj!Pw2r?9gCBvk^talo$'
    'N1DrYI(}Ew{S9cl^XusphdG<3gK)ZINP-+6H;>4Q1N1FU&$GwhrK5BI{2s+OecF5W+k5@{Y~MVL{MEzq>duGnmAwH5JQ>DwSP6QJN-+JDx#AvK><^}HUQcOx'
    'gtOV~eF2EK3`QNI!}ZzcpL<U~f8-4)2_8r#o`IO`cjl~sTn!&CJDlWroPS#a@Dl!f{G}((Ksy9E?W#xJH@qbE=GTjG7K0RgtwKtz-dbB5k4na6W=Xm43goM3'
    '0gJcuIiUABiehNKFhZgGf@s5mm!KwVkdje%x-*_GuQpLJ>n@$8i#Z&V^bF;$_qE}gZS+vLj&@Gjaxo5v(IdK5N0AAUygj66OKAAsZu|mZ=&Ul{;UPRuu=;Ia'
    'P4jH|uB3%*N+wYjXPx5mp_`;;OW^O9ILw-x0Cpf-pw7TZj(1TKoI(dXPhX#+rr`AS$tS2HxH_0mm)U07KMzN7ws^;Z-<&*&U!Qu9)3Yz*=<>irKpsE!D2JDu'
    'sfo?z7MtN&Ch;U*I0MQ4oJ9%w{*%J5sM1>d@rxw>&a+eRlatS$KKDL<4&v?F4od2rbO_*}PY6IxQ8?aBTXz~|utHcmix%FKH?Lm-Z$HmHtK-e8C5Gi&>b+LO'
    'Vs*SdEI<cG>E*8i7}N8%(7Xq&>MC%G1YjdPc(r`{t$G5zz+Dn9GIAQixQS#&0UQ)Ckl|vd-D*+zn-RWWn(J&MgbOuCxLinE{_Q|DOZNGbHz&KFJ^A#7#~p(d'
    '8x7CNl<qE2!8M)^m)WlXETq-bXop|TWSDmZ5{Y-9S%UWk;Ol2kp1dPKXCflMf1n0a%>AuJ@U6kSeVe@g;`Ga>-wR&9{NnWS)8MnymtVZ%$B_MpqxRrv|3MUu'
    '9zG0@qV~h%_HcjiC^~*{bh!UuI2=Db8ie8T!v|6OXxttR!-Mfr7!6zPXn*gx)xvkB&)NvUe)KTh8$`!pyLAwbS_cQiQS0zv6h#k)t?=RT!QNgN9zGcD4IhRF'
    '!zhfP>~L^AdKexag{@I*Y<e1m$L&2RZ;uZ4;m>d{92_1UwGa0XA09=6=-_y7_+StEYK5)g!|{F_;C?uI@NoF>a5Rb@3=U0CN5_Msa0q}5Tl)`=4-cY)c64+Y'
    'jSddSQ9Bw8M|%&Bj~~K6;e*34+8>Sg2IJx3!v}j04%@8<(F3ceqw#3}Vf%P*@8Ga?9EJy@@i;sjM#JG@IEWq`v|A4khKEP{herTS8zvV$d~h^Aez<?!ZV&d4'
    '$$j!O(p?`6_Kt9&+M~fffIJ?I4q%?g55}Vhd&7hFgTuW;nDqV_z}y=ig+~vLA4ILauyx!%+Hdc*L+WWCeN(@9aq{KKv*#zDJbxOTK7IY><>|?rXD?p_pFe%`'
    'otIB|f2Wr@hbe|;H?@p{9Al_m^g#u=`D{VP0?Y(u>^%FDJeVlpDOu}bAf!o{dEsJ+d8>vXxkkGpkWdbX9NGViXc3QNGOCxS5}=j9#Kq{1;G*hs4mW=kqS#cM'
    '<oP`7?Czd{RJ<HCLB!d8_To35e{c8sZ@dw}Z;5dWN6RG2x1Ys})PtRmGPX`h)to>^;wKW1Ed30FBvFdW5bK$(0{!yxvYDVf9})N%Ep`a}<ZgD*I(T@n)7m@S'
    'X|*379qt_SRKh&X0_1kiuJ1IRd7LD?zA<^V>lCdDsKS&Uc?Cur#l{fWLh_uo763Xmyfnec^f{2UE7OPqtd5q-g?D?HHCBx9-XzDhiggF{0d#^%#n}YkQE|YX'
    'd;*Z4P1sS5az>%^1~B*+(KT)m$Ya1?-`*<c8XYKV9t}BmcE`#RGn07U%xzQH3sR+)9eyY-GTTc8V~}X4*WGFw9Zt_c;>jnouHwUmI{@e$-dl9GAg{v!19=#q'
    'hFR9F8ND(Yp46NObk)PMC)0tQQ^s!9;6!>Bcnx`zUqsri!+8|v2;|N?dc_(Pd+l2NSIo>E(wUYGtL2te$U!7|bkPrd7!G38v(m1jO?!H-9KQLXwIOlqU4N*j'
    '>NPe5ne>{-RWgd%90(U%xpv3N1?p5nz)`^Bk&>5kaw2lPy%$Te%HhG-an%9PiV(%1>IkM;9-$>M`pqKvev{seqihLu>8f69qK$`T5e?(0gaPS!R`YFd4`U;G'
    'qq}R+0JlwA^^A2{b?l`lJ@l(WMzhJbngN@5V0H92rSv@dJzH$f+{sqKt78>Z0nM9Ls0j7MJBs18dgyNlGNpYXGoU-ct=;2oeA5wt@LRo%X(~>n;9JECp#4xx'
    'PVe;g&!%WZf7CSD;aXu29tIsBq?{>Z&Aer#tYmh;y`n*ajnT_QUt|RjZwk|JZOoI)$8giK1o5bN2DxrlI4)dj;&t0meswN%YRJhpYBxC8ZRbX1frdZzkhisl'
    'Tl?*YH|tN>s*c`%NO1V(E@3mu0{wAksPa8Dm@bgZ&w9cD+|w+p7nAE)xZxTr>72znD-I6SAnHt}{FvAmi_q=oMozjNFeJ0kL?PtuMk8>Q>6;JFt}_wwDy$WP'
    'Or1y9l4GR;hKi6;QlHk%guQp?*@H^`&1s%7ZGnkNq#Krf5~?SA;>=ET>3Bn^jLkSc+!ti6PquHa?Yi2668YmGQnWEDQ$1awCdvC2TcLbA-950^)&Zut*UlPK'
    '0t=?Gg=&3Gs?kn&BT#N@y-?UT3JEpv|0IbP<=nN>!br&s?&LG{CZuW@J%`u^%&RFWw^UKXlUQM4hIp@|JAgzofqwPxIMsi0x(p`@1Pz0(8x|<9C}Qg?;M-q;'
    'm~-`tD$2L)I@5o1)YhlJ`R4Q_c>Mh2_3LMkPoAUwHO!O^Zl9dIIeGo`4VfdKyuknym_mWL=N%Ops94uTqi7dDJ%0J*Y4G~lXD?3Pd~x~|yK6@eLEs+jA4Ct@'
    'VSDiK;1ER4aD23X@bLJ--~i;v2c!1!FnVw}ZnsCr2m6QngJY=II@$+${UD_97Qd#~^ld`_k@znR;A5g)viYNV+(E7p#@=g*yPoA+OaVYV<+Nfl*SpRp(t&ng'
    'SAM_9jKx`?V%_L^dZbcxAU&QF3Q+P(TA4EGfT{lW4O~)3eK|U?n^LxMHbw4Hva!DRJIu4#Ic&-H*=c~-h^ep<JL~e|1<W#MwIC9$a_O80`jJ(Ea3E7X3+HGr'
    '5~wWj9nN>=b;;@FNTv`|0^IeL6bB(Qfd%WYgpL!*q7knN6ze<P1mZou{WJ|xx*G(mh@+dJE(qozCBag~c@Wg`4XPqV9TqnY9C&G<$0U;I!Y7mPzW701tEKYy'
    'L&%Ngrsi79Hwuhw#{{5)iogIIs)Me?eafgy6-(y}zqB!;5vRGPc@@tLc4mFI&J2CodU|%khY^s?1Qp@gE{`#dXXx}=F<U!q!OBnx8-cKUQI`^S;!*TI>ZD@q'
    'a?DW;fkSQZ!4Xc&577~R8cg%W9d^AO2CbZLy#*^x06Fvm3il2V6(`uzyIiCj4mRknii88oTg}6}Y^O4eFlHiI1?(0_M&l}4WN~j+7<^e@r)KG_st<*es*9a-'
    'YD)2x>X9~YkRD!m8uNrr8E}{LIR*Cg=(?n1ONrg8@Tf>QHx2!o!`YzE06rs;hBZk&_u0D`#DmD8q~*{u3SLL{xu${wgD&Cm;DcWJ;*C}zo!GQBXHO;&Vt@}j'
    'i}N~dL98Nk6cPomo_BFrjL(wk@Ovbk0smeKGhRJwnJEixstU4y!QeWNvO2}GG7>3^yH;8>`i@CVGT;J(FdN2kR~&FKq-%hXNB}iao2HAbTdy^c9CYyEC!De$'
    'Od^h;m@f{)F#63XLG^gOw#>&n$F&la$iG&?CyZ^Aqe9kK&dP62qAS>nD9h`m-Z8}&4?&^U6<bHO;ZiP)VJ8S9r)p}{Oy_XyYKsBq0An3<Awi}Ns2ii#ue#N_'
    'Dv{PYd2`$&rw;T(ua(3>21+GMp$`3mGLkB+nJq4Kfc9FXg|AzR-*t?2S<AAnwhD@v;`-4d1-#0Eio53mZEy1pQjJc<=1_OLL{xkyf249bmnO~mb!EU!DcA9&'
    'GAXb$#-wS<FwMstp8`v37bO?*B27?RZ5B=hq-W2!3xx#30dvP8X~|y#-4UNXZw7(5Cvpx*)m#Dyic{yS@4b5R>9a3B51zgF^rhcW!TAc63RSN23B56ZoTl<+'
    'HZzcsVGbA-y`WG+=!)}36sPq^9nzHJjp&0oRpCw$1pV1$czz52uyn1e;uY6%iq=O-6RFG7N<3U)305p!B80$9->3o|>wQk4NV~4+S^*~219}~V1M<&{m}b|S'
    'O#+lZb7zWmu*&@P6Hq*YtS3QR1+0By@Nb`E<zHw=a!I;^-zau1ZOlCz5GM=?C56yk;Ox|pL7wsm+pRo}8)d@~MFnhE3B6>oCf$IGK$i=kc?Aq=&{v&CP}0e)'
    '!8lX#uE9HVmA{c<c|m)m-T~4NqTWrSB$&K3pUrE<HcbLvt`h>eSOI7Z#_Zsmz*!3vgYt<SPdyQQG~_VGovItmxvXxBOGy`nS{Zqt)pu4%p3aqvC{=q5QQxN<'
    'M`mTPW)nEbU|M~t<Wg&~=<SCJjp(5`Th2gA&nUMJp-};{+qWP3rR$-%3P`f$%~fVde;cCAF6JtlZ#iIugb9TzY4a0Ygl8}7cBB$J$g7*Gs6mg(aH=wn+|yNw'
    'QDpVG4eB_F_s}sXCt;bHD6MaasE);rJKMd1HoeYX|Cau;lPhg~c5Jdh=Okk)w}KPG&4!^mWqx|id9KEPQ^}w{{ITB2o;<i$ofNK{r-H%j4~><UZ^hB6j8JH?'
    'di0yQJPTvi9!j5BK^)qNg|2XuKU5+o=EEthNAcisJ{#D(J1k3~4`+S1J;Gv!X7m-JVHr-s<P7Gy&4G@2ZQmxpR@892dp10G?kGvh{@NoxT1;m$kAmp%v;AT@'
    'XJZZ!4nf8S2I1ASC%agEOaTE==f0eKNfhDAR|K$3B_5@918;lB5SG+ULts@^y4WT{P*XBOcz>0Uz+sM18pxyRv`%5miqV#&0w2IZtyPr$<okHKgi~w>@!;|-'
    'iq7k;a<;YcFv+KuxPm{S6WjuB*^sj5FCTw5c>0^(@7TXze6ltJ*fpv(HzlxI1>hFoeDR{hsaJ@;M$lo=zRF6#QqEwd;)|O5nud=!e#OHhe5%Kowv#l^97v3M'
    'r?^jXHb%`XRI|RI7V3f(i~4zvfTk)0?*}?_3aWDW<67>mW-AVqj&GU7KcK`aLo_|20JWm;B4tOt)oA-=`J1wOlYsPtVO?IM8y-i~(0aO}K?!v0X_s+Ncfm{m'
    '+S!aEjH83*B<UEF1?*BjB4w$Szy;e}8&W&EnzL^s>V9Ec-AqUY1tiD7xU}N)V+tbmChvynml@itd)01u*YLlqJ^1e)Whr122Rlb~N!Le_dtA%_#IW|b?)EED'
    '=d}9rstq`CMf&PX!WFCv)hJzP3)X`%NWtRfV(Mcpnx+4)DtC^Fx6A!G!~}K|RY4S5>5Aa@x7>m{PW7r?&<bLQ2(Q}HjSxT9kQ&S)!%KW7YIqnyWg5;|?S}3X'
    'aE=kvWQn}elq_U|X*ukYog+up!VS3Ga>lLo$FyoDDQf>&!ADnQl2;r`OmM-`{dIl|dY9BW+`BFJOujNIYNbjgZ!}8z!WX$D(RqZZYh^=aO(<(&`NH3v*`rZP'
    '@G(|KAJ)4QteU)Q*&K+Rb;JLZk}y=J>adeE<dfxWkc2UuKpD04+M^b%{KXgGo2L;)8v>MRK0kT!__toZwuTBZiQq&ffW*WQ4x~C2iDe<!tzAx16&bAIsiZcP'
    '@E9lEZ<t44F6BY&aw~f*#`g<#{Cte?7fQ}p$}(JdN$JXTD>BNM9qFp=P_z+BXaX?+PYUT)t5vgFbk$RzLw>p~Jv*R{-5x_IQ<T#wYT!OA+q)Q-I=Z5qg4*VZ'
    '3YS1T7Z_66DK+6pC|(lcd%M}<RPd&#W|?wX`GT<n^Gp-u6vc{4C20;C?JhGkD~;g$MVu|kxRmYG$|h$j%LZdj5?wMs-*8X^J(STZ9ChJt{us5ll~}Dl{;E-0'
    '-cKkiQ*_;d1<om4qM|BL=(`q7&hr~R_7<Mnu&ns7rc*JYsV^NKMU}|PxUL{?D1dzD{x8x5eF*h3nQtIhNSH)kl+rT}8X~FJ8c9f_NL1;02;(zzf7Ri{iO<5s'
    'bCgZ>!n@~PS~op6?K`PhIW%F7KdDFLnhJ7f$mJHP9?Ni8%wpXFi3o-Tm$mw^Ntw0!wX2%h0w>3>0iDn}n}<UR?W1C!REFtee%b_N#It4*!@yjfwiGPSoJc<I'
    'aSxY$^UO)R1N*kO{(_~2<#mBY2Evm?)_q3c>&X8?AaEmVlW4hM2U|Ygk1A+R07neXO&zLJ>pL@0V79%F)t1_pHV!AW7R9RP_c=F_>&A{mt5C|?J2gQa#WGJ*'
    'ICsf)aJI~IO5%lv=;+rGSXY%_#`C_zB$1uZnqiXEND+@bYv0vqTJNf<vWvt$Nk1hE=)U~#c*lPACyxHUcX+U(<$Y+WlSPjyMhnbs!>PbTsB;w`(ItW%FTdsE'
    'm9ra$noVC@I8g-P?A$16Wa5~VeURpPiq`KaLDpN%);=8Ca3WFj+ay0g@xupP?CbnxnzIAwv*irLj^w;|jkPXej#qt(D34&VF)1o*>eh%TdF*1Ec^ZBPfsq^n'
    'raI6KnS{=e;-)+sE^{Tg+M*NA7X#a0a-uN_%Fi?jn-Gk<KrL+=Gfjcq$0S~WxZoFaJ4{oW5IZ8Nx%Z$HhUlZbq|A$pHe>E#!Ukki==Kf+f8;T|#UUn?gzOGO'
    'U;}4}98jMes3nm9o?43#gHe-}n(xY5V#*4~0XJ61Bp(8NR}|yXbZIgwpzeTv5${fzO?!+4iGSHNa}+PLRq*T|IvW#V42y+iMyXp%BO7}KU~z|j=c?bZ7w-z!'
    'uEWTH74NH&@#?ixC;-_#w~36-ENQyxbFk~ZB6-A?n(6?0VWd<vmJKvV1e^7UOnoV?JzW3kIsBQh<5rQuW8qYG;3IDjR&amqkjsR;v!WoRfdhv;R2c+G_A(YV'
    'X<0E|2&@sVCw`Q}Pv1DE=L*b&?OMMamr;1mH<g&{U_5I~Lx2pTBMr*|H%TOfm&crYLrN#7mlKOHv@PEs0Y$>qi%khMG5&ojx%(XMu;cEi9X%<~t4^<H^0ymT'
    'l_sY=A|zc%-3S`KWAw_&p_LvBL+5hN3Faz@tgm9Xq$9S>>$)7GWf*AHDkg(u{f4Hbn6!-&G*b`0xS`o<g5hLI7ll;mQ3oCF<~~=qjs}mZHj4tv*9R*7%7DlL'
    '_J^^8TatfefH_UdpC`Un)t|*~W)0%}V@zpFX)HOhjFSC&xR`xRDWeTO87L|e)6a&Qj$bfH#cq|UTS-N*F|UxN&D)%D)0&%=jvVjpaqR}-xD9TovrcQDoMtTV'
    'tB%aSR!n*>qua_rCpBeF1YQ-g__o4C&C#*;7Mw?!Z6tUn&~wUi`QqH42vEP_+2M^hV6oAkXo2aj*-kmdv(lSx4Xd_RwsXIwR~wb7tY5^l&G(Yz71eBmy00Wr'
    'k6|ZPCql(8<s?Y&ocA=GtV@8}uFi9M)kYDdp;Mc#Qpc`HcuMGtE{}+~LU-NcfnV*6*+ru;1O=f5rjKe`q`6P?tSZ)1Wq?wmOnRz&7COi@_f;FTO{<;0dG39*'
    'Q}ga5rLsP>_WaGMxASqWVlA}F?4~LlyHS==$aU5w)f{~pe$W~-+r>!EV~M)>b`M&!M*XkTtyO1hUa4MlvV6<)2{I>Gp!OU?trxUk2u7dc3>6e$_+i2Rm3KXS'
    'P9R?`uy#Dsoz+}1w&S_#GyC#Yf7+L}Qlz02q|0dbmQIa4;VZOih3Mo;qeh~V`$%qU&x+4}WLT{9;p}m!bQlE}_Yb;uFzK~sP^aV`Ig-b3*X+3Ck;UTeBTAmA'
    'JM=4Kt=m~MbpCkC7JzBPQT)b;0k?(l@%`J?@0)YmrUCQEllGt9*i=qTdZd;Kiq50E&N2dfQM9<miOKB#iYc#bSAs72lQpr^tlBt&JGV2lYw24b{_XmZ49B`T'
    'qJ05&Z6|^HBFUU>QxP(tuwdC(f32R0t<)RNf>7imo);x#QHD=7%VwxBKQ*wW!OhOK&(WF&bM2z&GHgg2xK4%RWqqZiPQeRavxC-kuSQl>slSFv*}(B=vRJw9'
    '4Qr?(-R+zGucC>J5gJ0;+j70qOxADpGq!>en;3qrGjFG=lM=h6A1$s`;;<5YjWQ?&W|`!F;W5r>I;vCrNt@2vd5fEJ*w_*STalBjG~-j|_cZ*R?k3}>KQr}m'
    '4tg{JF8Q<Q00Ep*q`&R(+T}zDJ?cr}NKxP@|D9j!jdMEYy11?aaD~H%^=jO!6Bl{aru>k5P9zm8)IU>CO1^oG^{=<A-!G>=LRf8zrGdeRYW~PZ@P@5n>^G3^'
    '>PD28QgVT7z^u=hAd{E1zm@CCm$E?$VRrScS)ze<K0523S(Vjk-Q`8o_A8Tr>BTk3h{U;IFUMv!xK_j?)~gW5H`P~U3@b0kONF>(S0UPcn&%Gl0Tf<1m9Y!l'
    '10Sj)3qaOnOa@b!v};GbSSIL-&9<PXl5!HwhKqQPk@?sj8g12-_XJ+PHMWQ+YbB(#ur7p#Y}##Es+*paNjCq7EF((a^cG}EI0xns*kA6qRMj!4wb=nvk<6D#'
    'G9l+7pa5OvK4dNg?13*m=GSVp!Ih5aVg}2#Ri>zl>f(#O_ImgtG461&b5$@HySjf#C7q70>Djg8T`&cg+>!*5kRDQ$5Z9`fFMvDj)|spJqQM>scr5KXE!N*`'
    '%Fa`@@D!U`I<|_n<&!72zeT$-eKO6?8LJ1h|D9V_+jx3oPLw=nPdr)!-I6U6gw2>67C|gn5w%PaEAEvs10{NezuZSF8L;f!WVl<tpb4h+Y$=6`z`Dqh67&(P'
    'qG$;BLGT9ok7T|ir@1n_lHJnQwbyRc$!rjxEz@Ntj>M5-G&9LtA9)#`H(rrl`xTEzp=*(b1KPiWg*I#ZfwQ*6(Qrjs-w$|wx56!YZQq1f+`q!80kDvP<K|T#'
    'j?c=dl#%$uj)qo3F0nLmtf4+Z);L6K<k^zFi8Rg+B3MFvWJ=G`SZp~*^F%AmoooH1UO>lZO$WjfS{c&WaA~kZ(o6j)2Lg45Ehiu#+~c<&YI3@JV;>gh(BBea'
    'FM#ZHiL@Tgr0f1xn^>`IWMp0D$l}7C<&5fM7VHn~n`8sytwtRc36;Z<n(h=7rDatfYxYg)f@WN&R`r4B*SQ7@t8Ny+TwD^{$%LtM<t*cjNLu9*`4muCg96td'
    '+u`F=fd)=7$J#kxvphP}m79O+)(9$mHWNNf4q!va#bi6eZrYO|+U2~6tToi&YRQCn1I}qDj6p>uN3uYuC`Bg9L@O6yVfc85)NUoXmSm&y%A<nCoGbTp`EDUn'
    'X7967S`k6pyr@|feJhpdSbAi=3B1L2d(8L3yKl3;K`PRmD*3etY6TO`VE5?8CT*~ep((V5tx|3)5(k%_hm>ehI=d?0WM|u6Dl~(y)ZMFyiomB7Z8v7EZh5gh'
    '6IIO?fQ+%|Mh&04+-npgNLO*%<rqPm#BD|7@oeGMr_82x51EkLM{p~qm=8Jh=n}_Dc3_kFBE?&w_N%1??t@fGr&7j2CDv7(cbfBRQnFrgvC4bubHNJL+f8Wz'
    '&dhsbM@}?;Q|pgJdS1|-|Gz_F$TMZ-p(@D&>2SfQT*<e&tl4v5vJG1+M~NJ)Oh{rKe$ET?JOtisq+*nE{2IW>S5H25Lz2ED=T%8P@aE^#$1CoBb^1!p)-pJn'
    'ppblExEt}W!)Wj@c|4P7|6k(h-;HvwquO^Nolaea>#N>b))+L(Y0(&<y5bDE9*$1&nmr0wB$^f2ZI#~BFTVE|gu1&D^A&Xa4m7*)B*WrJNg5+=Q!U)8J=7hJ'
    '&wSV0dHXjVZug!karGNPi5hQwBXHerI@@iT32ZBy*74Y!qgH9OQ0{!q;<+ThS3>c=<2a6)rFzQsbEJ+;jfpmPsloxEmuSP)=5$t*6YBw7J5XDvxuzeki<?d9'
    '9!V?T@zkSg<$JWu1|PDEGc1_FRG<Jpqg#)9op*Z(tj<tf>DXR;>65Sqg^;{aL21BR7hq!=;-}Al-&1BYT_h45HanBQ=|=3<+B=w$fa0=s_c%E{*J?KumxFz&'
    'MoS9N?!3J!CoF;st|#)Yqxk1jl~E18MPY7~H<%BP;*q8lnPnC=RiM8U?I@`($X(vtUF&JY^0Ay=Ll4UgFYc!IGJ&7$H~n8c7C~V-WV>;Sd!Iy?o~;m8^NB`H'
    '?<qPz6N{_V4w%##v-L^$_H3029<Hr6okPx{98R<D+m)Mc6hqrF9(JJQz00L4u$$hTQa$sf5Ua0z3sIN5rh#&yr92N3#s>RU1jcADc|!v!sH5QtW{}Se_W6Qh'
    '*hjB#JD5}eZb50rkiU8z-9)G-4<3ygvMab#;%-4RvdE!+?JDa*FT;eNwdR=0hpwWmF1(?<EhT|yV~gUS8g;b~gYKRG=yi0Xe(~m3o$av42RCp><g%hWxh#V='
    'iX^BTQrUL{VyMb%@d2|+;DmMUs;E7NsFY&&Aith*+$UPM4!*T1nvQ3a>}Fb;T)Sl|`kqyKtNUHhep}J08k}wf_8oOw%~=Yw#24DS->42|)mRxH%(r5?3Z5YK'
    'OX4vI7{MaF%oOdEB%PT;H~gOtSWevg%Fqt%thG3g7Dln@zDO0EWU#6ZVbZA%e@8q$jp9m*4kk#5zbdEWO_98`vS56sEhZg;cJCg?VOFuw%kQ1m#Zkc-%UAaa'
    'j6+OV4aspsJqG8|j4TanylTF``kibCHo4|kJeE2xexrF@t2B38_4a8ZmJ5t4>hU2J55qCL7wVgo!;+Vlx%yJDlr|O?_7s_ddlbz+x}Mq$%{OZ>%)>^vA{LHx'
    'Bn(CY*HToAs=#Zb@so78q!YESQ8_|}bk(GZ;TvIAU$HStcUMXD3KJ6Lv=d!#Jrg-+t3Qr_SzzdJ{jCzr8V9(+<?t7JdKTmE%GXjk85xa;lT1fs_-JSmT11*('
    'xBl800y0LxEK{5$F2XoTdiJ+P+Oc^TG%}G&q1qVD;zM0H+@M<i>fTY-q5*m}egQPkYp#M|JL>|8k4jzA>04!DaVKL61z|yItor9JHh`+zN*ln+y~c~PSGXzr'
    '1+OaCDrXo0Qq*6*ruAu9Yt`_+8(q_1{+d<B7?E>Xopji<9**yhNP4Log;k<oAfV`glyx<117pf}^|>k?pYDo71yW_0M`zdF!ckqbw$-tCR;@=9Q^!X2b$i7S'
    'VR`~vF|sOL+g2C_TRyh>vSW{}IQ3RzrT~qqfVcJv`CCwe8#=2U>!8|Vw>|R`?JUgZcJ*jgvvi3}<y;uZRnriL(suAKTAEmwqib-Y9}$HUuqReY^DSo8`dbZ7'
    'Nys+C*ek_8W(A7ltk1X0R!;q6*Q0+u3~TO9ys`>p(L_N+M<1DGr!XJ)#=Bj=PfqwhG3wa8F~EI`YfU;?9VoH@LRFZ^)&07(-x_QERj8h@^i^o4dgrU~IRmoo'
    'C3t81>pON*w8>s`zqwN1uk?PieJdK4N7%a1Ldc?1P@(yH!Y!8O5~3RRp=)-t;W>-iU41>^XXEFwrlRr$@$ZWB7V+c?{@sXj;h;3U&FHS1j71GVp)pGxtqcxp'
    'k5CAAS$Q~0E}T^Yg>8{uGCg0%Afv)vhpRzbS#5^yTFb8PLT3~(<_W8yRI$F1kn8L%>uH0z?W{(M9#`cZ+yJJnFi}CBtfCE_C<|n|O0^9Rr#&X$s(Xc`Nw*uO'
    'XK5&g7kiuQR?MPTLlw#Jg05ER#EwC}zQb>EL$Y&hHLf_15WG9qjaqq?mW0QOd2ZTZTZ&Z3wR^MaqTQbTx0ZjSKzUb5(!wtp!n#K{U!a=}`0plEZhTD>Ah+m3'
    'uEnC_bKS-G-Y!Thm>j+0p%Y=WP#Iy1-iAp1iq3|c`z|@DRG*2vpw`X3duwk>nhU-;vLM3}jGh@CG5K33P;gvgZO0MSU6UM!t9(avjY))|E42T}?e^&7#+hC%'
    '{wS>4j*@4b#TOP#3f8xBl-(w106w~qvdU638ihWl(CqM=fU62@+9CMqyLlDzn_-3;&iJbCZ+h>wRCN+sg9ITm8m&~}1={u0@pzQde#ybQf{KeTo~lR~Qc<(Q'
    'DZ9i}>%J<TuS?Y}7bY`46!J2=E}YKCzJBcH_jPl~<Wb#z*ESz5>x5!lhm}`XF)D<qScn9$%6)uX?K@t)67StJukt?_29~F=vkoR+0N{sOP^YkWY2pQ`3L<J_'
    '`6V%2cAE-Y0_0}Ps^TgYpWNF9o#pzTFH%Z953^QoF`a6kn|1eiF{a18dusU=Da#CVkge)T+v40<{u!N>jE<uTDlQC{YE?+b;awfZrh-4cz03Kx)lgMuajU(l'
    '-q(_y$O__pC#%)+iZdCnmd>_m&irUWT(Y~aNV2e^t98modbr)aF@xSJ^`bLYb^rDY>i-{a5!!8XVCq<FvGM_ixijUERp!Z7YnJh$tE5QQm#m2CgIbL;+|;X*'
    'dcj4^E>`e|1lKe-v|rV0W%dnc%oP}7x?Y19suJLaJtAWVdR@39Sw?5JE$WSOQtVWEC^o#>2Kp%bnaPHl=lYs*;|<jJUQg29Beq2)cP}bq&#u1lWwSf&lkS&i'
    'g+W@ZWg<dvKgH_MD5_xHi{GkiLXmT{c+TN+#>i-{=J^rq?{rMh{sF~(LJ1qqj@Jb>xvkoaq9t!Z)GayVROH^Vu(h(fHtg-bn+Mm*w*qP6WMfg*25iK&!d$}c'
    '#kIX~`l5rc)O!|y-8Ghr(py)VJ%VtcYGBMHAU4S;>v9yQqQMG?6WYRM-!GV+bkQcL^7M-dAwgTpRPWjZmVBr;*@+kZsja<yk4|~UAJ0zF<9N`$Kj%4R=}e&q'
    'S<E*EP8IrXdrF!NC|zGT%sK4xJ5iYme_vx5EM0%#`rH-BTi?!*It%CZd^w-e`zto~P5fXf*vj8gjj<z9`I(xOlp=Nj;eLgwUB&F)!2?T~`;^vGF$rhxd;Pn4'
    '7wI6r<jpp!qbJLa48gm4yF0&e#w7drj(2l;`}RX4M3kiu|2+(|dv!DHnZT5NXWZ?D8arITmR_2A^=u~_MNU>YY+4zy_zJcV@J?3X)iJmzm<~EvP+FS$x|xv5'
    'n~PN|4E?w+EZ7#eXo19TwK9%j^ka08@X&+j+B4pj@}3?i7#R&I6cTyIXp^?}<>OP=CtGEAAq9DNd%s%S#_Ii?RV=PJs$5(;sdsZ_qcHBu-aC=$!eFvAftEfi'
    '9o16GqJgW?+X6_-6M_&?IX_fJTY<iN;-S`O51{kf3Vmy}ucfQgCNGpc9L1U7A-&0V*2|)0PYwi%l~y1@t+Zt+1g@`oL=lY=(WEslIx6au*4NNtcOB&H@_&@e'
    '-C2}S<kFFyTdcU{umX#aHaxxLgG?ov9Bs-wxH>vbRqzUIUJO}hLuuS9!r?jT8LL?+n8so_vGT+6q5Q39uY%WaUY<UE;=OwD8F1=+o+B?sPlS9DXPR=1G*eQ`'
    'Ot!ia1sBMwa<=_2p5wQ7W8F5ggu>DAq?DDnnq$$#8yOV-RArmYyv3&gE4ff)%7^JhIuEC~BiEc=4wKnRUsi%uSq*wIt!BYY(!IO$9;)|dI2tK^gNkWVValor'
    ';D*2`Zb!afIiv^_Dj?dq9<UtItilROg&p&&c%F9)C%6JWVJWKNb4^A_w<!U|H}rbp^tQrpPUq~f;u+|({tjGB@s4~Ku4cu2Y)rF<@8eG#&LC7n-&$oP_KO!I'
    't9uCWRaA~IAD%hW`YJkDKz93PSYfnXiK}CG-+(@DB+9-^Qtq0{nEWV&<TZaQo+CL?^hk#>=k&Cg@k!Uz1XwN_XnJ(O9iv|Z`>k}!G4UwEb5R{|NAb4d-IGKf'
    'W)c08SAH^10ST?CJ%bXl(IB)$%|N%_es~NVX`B9^-yB;d#|mgj!o&j3V}Kq8(DD9$`}o-V=p(OvRMvlNRNOcb<lIncvQSOZQe?P4braxhw<e*}DxUf&ll>a|'
    'I!Po<dKuK5;Iyif?8^R&W2w5G03GPk6j{SsdyZFKqfznMNKg^^wYuL#<t1mU+EOUd+&>&8_aY6%ff0Sj7Fnb@ewA`k{L2=22!m`n;9s=zc~@SbbtWlE|L$-`'
    '^2SW&)NI&;J#ja6<%A1uC((4?RlkAj7)Hj~Qu$VTFzh&7PV<h3?t-tLe7c)oiqn^0a|fhh!o?Xod(?JzNd3M8XBf3*+)bAdWm7}mWofx)ilv6EQq^<&_17<7'
    'R5&3~E?i8mNlE_D<0dEJaYN_iQCLG9U7Tgz+I?$Q+Fx;1TV^e+WnkA?2D80`sRtdW*Jw7+uMu;}sg%ky%0I{*Z5WN58lihiMMr5@)wp|V%skCQ?u}g_A9*cr'
    '7KMO+%Y+Jys%zCKv+zo<NqMBU8cFqw&};b=5NYEq#P3PafIB0D7v_*h^LV!$U!5XdVJHZMOu0_!q11K0;%unTdN>m1uj!2h-wc#ffq6z{???2aHKe3CSYcwy'
    '`CU5^lJKzVwDMPTTBSA~#<FxkH&i+vW8i!5yW!ax#va2%SQu<8hzgYyJ=IdrVwqGEwmO~<E3!ELY}#~-E(}G7>;xamCA!jY&|+Cd;5frH4dg|*(WB4TfXE<p'
    '0j+kpmbhbp$ehZu!fl?XWqjx9x9vbL-t#I|AbZABE&5Y=(Nl@I5wj~jd1fnk*^3*qUNg+5&~~_Yni!AiLNoorb<bxAEbmDx-Cp(P>4Va#SgsZFBlYA4ktt{u'
    '%z{@6)OH85-0E6agak8z5uS{evpGJKV}~eiWy=Nnp$y}g<BYJ{ApwHu7eg#!6U6bii3roguq7PSnRByW!0A+?_G2#kGF(WV@iGB1Ii2bnmkT(Z@)6TrtMWJ_'
    'bqjzZ>93^@vP08&psKy10JA{S0)F89^5pc{$%{9yQQk#%spHkNc{HS8#q>wJ?_1%Uu?QJfX%@VG_4M&;<<0_JO7QCR>8lf9f`ex-o;>}{;PJ`h?>vRR&AjR+'
    'a0f{l0hvo!ml-0TCJXd2a9h-AO-lK&zf{yXv~`jCHH%mhZyqfutWQHC8F30J)OJQh;)igr7##?A|7*15W>gj8S9uuL?;-Y})zJB+O{|1qRW1pMFaEkDgp5FH'
    '(9#%owzKmHZ6>S7K~#6_ugk`?+YeQ3aZo)l>;Vi-z`19F7fo%jL~_tOjCydTQ?2Xcsc!mi4aToo5lD?PuOj~FDUflM4-?T>B3Hgpo84|mCPoTDYQ^1>HXIPF'
    '7>QJK9jJ6ga5|-gV^%_P6D&d}x_4CT);pRiGp}d5WK%e~w;$q()@NA-(jf3C0|N}2Fc*kmU^@5_#TXgpGTF=tg~qGZ<%}*GifS^uNgwFYXuI>C30hSM6)(4{'
    'qfnE~3a;71T5T+iqn^A}LN|&%ZJg(Njt1losvSmX(d(?&e{G?vDazLl=xg2_R&5!&PBanO&!g+CUQd{U<AQQE#8blGlE&XwbCt6SaBEDrCe9rhS`U<OlO~P3'
    '{@K^TSP{!I6Ko8*U=!mTd2n{87Ib|?4otu-gRrTR`Y4uD69tMCqXGTTL5^Q0@g!c@`Zp`Sw`qri*Ky7TD&wz@m31RZ7*qdN&CEM_{`}={1fM;9@$~fM4bVz8'
    'b<srOAx-cJd>RjdLvt2~jx0|SDOEm+W`$7BG{d*a=~L7|oxFMW@<s6J$+PEQoIVX+zk%Uiv+)zqH_kGoVtmruA6fCJAiY2m(i!j!G0?#pWfK%ZBrvV9B5HEU'
    'o=sk~I4Q1rs2onAXHOg)zA%)vZk7v|(%hthiZ}sCK5hu*wO1R<n>yAi|6zSy!o^<qo4OklwoJWm+2K&Vns?u`I6LmHinBA(uC{pJtY$~dOREmZO=%IZ4|;2T'
    'B$6QEvU>3KE+-MC>O;NB$hm{qU!=Q^v5@F<WV1=0y04T{%nrHSSgmlf;xf``#VAh_fP_Al1MpHYMHrDQ<EoI#piElbsPR|j(B57hvo;BHs1kHs<>lN1wTpbV'
    'L$qSY9>L>Ry^8Du?X5En*nFkg*6NG{TV4*(P5Bz2AWv9So*mVff<8xyHx=(Nty60%0_w_m@h169Wd;Sat<`Sru4ROqTVX|V1wA6o#Ga!566i-vf9r{wiP&aY'
    'ZftxBx3nY^ayY{P6vbOY?BoBb=LNhP;5IgOZt8Pmj5V)%?APW9Ym+ZnFN>smku(*hXSL#^j^E7S`k=Imh0B-h5NyZsIB`OUGofIUf+EmVYI(k>BIEO+N|YN#'
    'F<zN^t>Fv_w4=p7DR(;tmZL9f4YvfySGZQ#lFdPwmw4R*k=~6H(la<GTp5X|EOMn;DL{E>8j>bTehJ8UY?1XzgWwLGUb;CZNM4~a3SbB!CYv1{r(Lc1xmxUu'
    'h_NcO-+OabP|DLBng;6wri&bBB-dGIecCI>_svbh$8ISXQG!y^R{60|r3Onyff3zpLLCd|<S?0jB&4xy%OerENYhDWqzRBpubzBrud3Qg*g?@r*gI-oGkNoU'
    '=a`jSp(TAG#xz;6%9-E|D5b-ON+!1eJ#@p*Ae<Aj2y2#|$MboFPg{W_7S(RO-|<$nf*Tk5S;bjUTvQ2Mrj$J@Cnsn6EfGSX3?C2ia4SYYXY<MopK>RQuwS`3'
    '*ssVIOla)8n}Jo?QOR-=PQn1ydAV?86<i%KbnA>-r&U_C41@rRb;>FEv>ro%+-1<K9GLkM(!{6|R9*(5s+y?Naim5bn<&W9vB(m!4RW_c!z#!4l}#&Jshg<C'
    'aya!1F^er_hR@PmnGoxgg6}7X;C5Rlv1>Uk+A0>W3abJ`pfQGZ5&?agI->JN`$2v!sL&7KqG+4FI!E*+-X0C^SJ^uX7hzo+c2YhYT;lc1#nl-zwp&K4n{|s<'
    'SP?fBo@2AwWt&as*48F6Wvr>hv?4AsuH)T`5@le1lQZzUVvyF!;1OPtmeP$2Pij&$X}Zk(V&X);Aq-zHE0S$Un^sAUfNcYWP8Vf_R%tx#`-tu$TrP~;yyk}V'
    'eWL)fWe;P^y%y3>Nm1flpJUV|0MVR=gJ_B$RcL7oT(T<H5~>*sSBk#911P7J0;@FkI9qhp(v^gM`(0HF_aZZ7mI<qyw$Aya%-6r2eeXSG;>$EcEPVOL+9v_)'
    'V9Lc(CQIwJd$uM3^^wwD$GcG(MjpZG2+xu<10K}UPvUr5foY!+i<ziaO}?8Vvi6y(wnI;h7-IyLdW$$CepytlTG^P+^PM=UhHFha7Pfk=AW&ty^*g5(RX8M('
    'gxy>$uI$jM*hmcm+$OhXUaWJg$iVTgz6_RuKb_0^isXKcLNdQqES{zRK|{<EplYj2dez>IaIM;pT!&vU!3|7MgHp05Rq3|xfUwTPU=1wi9na!?tP9HM3@pGj'
    'Jk0V?RAn0u`#v2-XJHatLhS{HsD{%rOE1**N(aTtjiiVSn+R8^7B*@%<bra=tG>AQl%PQQ4DoWbi7Y>Py7Sq|3r}~jBONF~g!_TtF&a>5AD|UW0vB?gEnMRX'
    '2{*T50!sgiwW3j=<Q?O-g#KaHa@}@r^m>_J)>7S~399qV4E*fItk47wULsK5+v|Hf*s9m+Q@m~bb6=IU3o0et=sLsCb6L(87y&1k0I%{xpn0et1iYR`0jj!c'
    'z8F=R224-07_MPjTxMFK7tCkK*wpqu>N;(P6F@9rV-^8%6*b}~pw8Zbv<L@TohP;H9RTQU`rK-^4!6++!9Koued;M8Kb{h!9syGgRCAZJ6(K!G7^=`>V~!fz'
    'IY*_%UB62iuy)wzItpv0Hs$K&%n_LcIcy>u>T`zd(w^Vr=8f{!t54fAh*W)d6wjFBspN60&XAR`8bL)!>xb>VhS%OZ-f*O4_axQ|^0{Gh5t^hH(9IR}q5zn|'
    'Hgn>LUKRQ?3~u13(z2*(D*6*QZ^(DqmT){q9dg$zi5I2#Qw=9m26<0XW7@GJEz}k)9Kr^AGGUYkz&N<Vl!=3~L#l^VB+b|-4B8BUUmIZ9p=q}r?X(_BN?-hf'
    'cf}9mxs2XWYvtd8*&U{E+k(uJM|!1;D6q9ZK}8B9M-4wN@Vz(p?%8Wy2mq$XvNfYOw<cy7wWR=aMLvgVuqmMk5omNAXdSiHf8mQyVIS101hOgtkhf8!yH`dQ'
    'tE>60f2?k!%Ls{XmEKdoo!&9;);dOCZHU;=&99Yh+bD|W_`8^~O^V$(H}5$$mwuY=Iay*bevA^INf@H2aX4_D{3@z5XRUTgU6EPw15WSNDWz*D_<fd*zl6aK'
    'OsKZIie?q5tnT83Ns-juGOb80IS6gc6DE0ku8R$2+1(&I62!2~!lJ(peB37Y-YU_pp|xRG2O`HokII69bUxE*y`DL*0E}<0qMr?6@`b7@95VEexkoIQn0n2i'
    '90SxL)>;=@;B{)gr3o>B`PH6W+bla$y)~MuA}v$QMmor%#f4ron^JD*aTOIUA3U8*5jtgS8`zu0nTao1m5}Y$C3L+N3u!6G$dnM1WRkb7oaf6hlU#d@g0-sP'
    'j1{zEMGVerS8g#>6nimiI}K+>zqgUzN-Zm3v$SJ8D}PHFRm+{_*R1uq@dWETtMD0paP(V*EIs@`?F3TzM;^YGc_aPW3LLEwcD1!OQZ{vIp(QFC<dc<@zDZdp'
    ';TrpV7p@q3tc_IaHqwXgDnmx4cDK^Sp$#S`ffJ<z<=tefMmM9XvSKeAUJx9v%>_}T4O(Df8fp~*NZmtP<}Qlwx$77O%e2F4#P=Icsbs0&v#RwgCtOi)g~<sU'
    '3BtLb#Zv-1oSp&t=93xCF<QWW4_HS)rfTJxlLTQ~xKW*a0zW6|6vV|i3Zi)yPf<;GyTa%n(da#-3cgpBEx5wo`AJGEfO29A$hauI&LOqWQxy$>&s(OHy|T&g'
    'C&-7gsMDA)7BpJG$f8Kn+r8^suzoHq=-RAIKTx!hCDPqU+#AYPreNFH+Ek@C*cETg@0vXLy~P>&|2&I%ZG#5_$p`P2qIVs;SzsW41&Pr~#3n0R<o#_etX9Bs'
    'P}@Z*nyrdryQm6Zt0Mm{*79Aqb~FQC!@J}GE@;$v9$O`luE+|EhXWWOWOr03)8Ydu3HX$_C{9+7Mjy2J49mLmCU$EUebf~}(~&@}P@F~)zc#I}*hhB+*kY(o'
    'D`UI~v$i60{5uF7o4dCzgqjme*6@J0Zqib0Kkq1x5n1TD^Sq)S1ZcQh%TAGq+^u7#TTWQ7-?FpP%YhL$bmCAGuwg64)OAC#P9?u_RVNj~lSQY@HbdP3jM}+q'
    '*NS0aoxrW@8B)SWTxo@7w}r5X0xkRgxxqvqzkKpEc>V0N7bkDNI7RQcT045!YDa_pgXlp!Y!4nD97Ycg!tv4m!NcPRgM<Cy!v~}G@i2ODIBvH`#|Qg|`-5Ys'
    '*E-q{55@;!&2lgV$%g!W3m%8^BhB=m3sZ%%kxunb;Hf0yk1Ci5g)$}aEOOl#Ehj@MnS}`~+G3p};;WM<Po6&wU@czKO5D^YykW4!{FgC19K$WZU-&QcC0Wh}'
    '<_kLW5zm*w-jH73HE^7O$jp*4h><$7!f+aeNid3(7NVX}tOs_B1$jE}AoJ8cXhBH~Y$NoNanX?~x`sZa?7;VSKF0QL0Jd+w`m5ag5C44m)4z_r={H~f?{n|s'
    'r+@8DzWM6MiC6#f`RCsGH(&jqITZez)N6ag$r1{_{`X#T1`UD4fAjT!quS-)eEp{}w*0%CJpg@&am$cJ$eIiE`Jdao)spUtFo9dOZRW<wzRV${GIX=(?JPTe'
    'mybZdwA}YTGRvSezSycMBW~oJ61cZT4JKXM0ShOXl{NN0?gGH90O`fvj<@W@bcfM?6kMjHzl3A;Q|-c|k#`SzgDwr=*ma_dc>gM$BVl|Py0D(=n?j-42xTcY'
    'zn+Z<eFzQk0i*{%`VKOD9xl#D>1DF}oj0F9_lDnm{bwQZP|#rb(;pAL(NF&uboN2tiF~sB=Bqyg6d8Q;^&flx2z@gf-Zx+UB$+he+c#f*?G5l3utIqXxC9vX'
    '&Da0ZoBZ@Il8K8JGiI@x*>X^?eYBhRMw9->cosYip{z)2vDV__-}z{;cnkkz_jdm=)qndoS>2-d+aLGuQ?-?KKEU33z2C<F^!xYdKlrbD@3(#a=@;8G+iQpX'
    'oRYe9Hd<x&K259<X{hX`)!4uNM>p-p(JjwMMw!|v94p%0>+RgTPY`DJ{~zk!?YEIE%@6!nqSJ#3Mixa~GAk>K+3X&rQk5x}QqoAunyR6Q07X&~J1LUIAeE_B'
    '^T9CoVX<or48t&t#bBp*usaxEV0(9VVOQ<09}1ZNqaJ3x^R)lMe%Et}I6+dGHG}rpT`5MKIQR3N@A~^Hzii+iC&|)pE5EEaBP=_u{(t}Hf6wEp;D5T%gU9o#'
    '{gXyqUGhHpg!}S)r40{;z{HOS?M|-}Kjg6`_DPz`6K@ymxgL7>I=>MsC_Dz{$v(^-4{9}z6iK|JRXYfwLwSZ@^Gb2;$bM__B;OZT#Z<eP4F+Vosl?5--_&Um'
    'TA%O)<U53!(S+d}<N}Z3mBrY*?8!j|uCGq1rWPF4O^{iDWCo6B=jzAr{~c<6P^$j%`@clG{2%E2{gX~Ua5z54tlXIiro@&D9xpBpA)i#bzjBJI=71wQ$x=o#'
    'gCO&!+ys9q%}KC{9u`~uxv9KmbY6kF_#a=>@CH~k8=HUyfZ_RehKe77PZU2XB40$ndzXm(0gn{!#G4*z81Ebop{$(S5poa6N3JPSQ_Hn7CzRT{F=$_(ciYi<'
    '-6^KbA~ubo$+k=d$4{bIa|6Rq%&#r7tZt#fRUd771s-t$=TfC!uF_-IeC06GGWz^h=BK)y&y!_WY6ryM4)VY}T<?pw5nXZ>zWVXTmFg3J_WCe0^0^df_FD-~'
    '(HhjBz|9kF!DNKWY*Lw9F5@(<;{e&PH<?!6(&x9)FQQx>Z>bLG&Ux<K^82%4`r>fsL{izQVG;hBRI*7&m^IwE#DP(UU*^TOHZ~;1FYtr>X!FU&hQFgv>3|=-'
    '_yP|&U9Hj4-ad2`d9y1x?SIzCv;9x|aJbv1wj>0XS(?rswds?N9)B8*v*h%<2?bi=eCO~+kx6XwWc!huL;1W9yf@|!#QnyPe5H#iVDb*@OrqMe0aFJ%{4;kE'
    '``75Jz~9v{^G|<JNf7?#2G=w`e)-(k)e9gIcnr9~a=Z(vxd0g(PK<0UJLIK-ITA0)VZ+QbEKo%sCa~4a@;j<KJQ4C3M(cgygomK$f;&MLmq32<(}LfDcnM!t'
    'Dp(KlpTj9tM=l)bz>*NMzW0v(HuxpHrK)SkJs_%ZOAaq<``ZUQe|&gi{ChwO9bSx3!82rv(vYOV^@!8M&xV~gg#>9%DD=GX<|<JuKwaH(WW<<}0GLr~yN6hG'
    '`Ud59uL?~`+@}nI`lqh~hmUO-_>0-~^{sc<c88bflk)cJhrhx*lz3Y*HNdI;;XlD6yg>hPgrI!D(f;sfKv0pP9rwpQ3W%Iee)w0zXy@onO=HMUCvaw?n;*ab'
    'NA&Gq$i@8X2N0~dvh%<G_aHLCz5V0&|6S)w+aBOW9bNzU-Tyo#{M(w_%>@DK?LuriA{@5eFDA306o%O#R6_TgCHsm%D0xE=iiW8IX-fY(cX-}aqbu5m&LN;f'
    '&w<w$@&EH5|IPpVm;cM({@FkJ+duuE5W@Za-PIgfcp$~`-+up(|K>mb>EHY}fAL@bkAL=G{_4;E=AZvb3|n$@qv(u_)uFQJs*m9vsMHknqqvJpEY{U|Vwe8a'
    'Y<N3s59=$>B<G=kKIfBEI23_!nc^mGn3ib*MIaKcVK>cmR;Qy$=V~39w+v&ZIq>jeFw3s;;Lq*{xm9vM*K*sl>F7vZy%wpqOKX6Be*^vgYX6JTWM?$&64Gva'
    'Ag<dC%pSci=nnVjYtq85+7kBlHBJ_^L|f_<*70<7O01dJ25n@Tf-3)wr+e6YC%-_DKnYt4wC&qAzDi-Awg*$!i<`b0L%%5XXIuXSF6q^qgKyq!AEZaehp&%L'
    'b6>F45k+zVM!?LCQ$!g>$FVs7I`&`z^0h!87u8~3m(<ej-H?F_{!HQTK03VB_zyQV4O<QnV`Kbi3l0L!QH?y}9w89u#d!0H|9JBWng%YM$LI2V-h>vg!d8CM'
    'Uo6@6Qy_x<b(TJuphkCXo``u=VbaRhzTuDG{pT47GXGmrb|BF4HREiYcd`CJO}I>~KAJOfN`7L^7bJ1AA;nEWtXl))IlaR~ur-JW1zQ>2DJs%n754Uw(jQ%q'
    'rl|Do_5?a*mLWJvHdQP(bQ|Hu16z?BNhL&A`3FihY>J4o*Obol1Ta+8`O@6DBR9`ebT)xYnzWEGIM)tBZn-vd@*T~eQ)5FuktxGU5g8F~W#j_YH^zK05AAxW'
    '_jUO^(%x#&f6FKJ;X}8h-lPJBsOZ^9YQ3BGpppb&7j1V0B{6?t=4VTG8i<kElvof`0<&3NT(H{*rG4Z%S1iOh4NX#2WkZ&7?PrD0CGUoSz3Nv>Dm4e9v|284'
    'zl!I}?m0Ut=-m~b$dVIXN<##=_Q&u3_hFO)w{<BD)}5~M99nA6qPz0UQT5v9#k_F23Vt41g8Ciy$#D1pA*W`=t-{pm&C9Q5@`RKx?WqZCZ{yCa+a`lAQ(F9q'
    '$)BuoNhnm@8PDPZ<73)coxP)W^bq8FZe7G1Q7O%F4oiqc6{%PbLLhjLmWy9l1I9|D@&)KYuPdh4U_GTR*}HaGOM}tyva}&nO<>MyG95mnd!1dX{e3iG%V@?1'
    'Au$f32j5*RNocJtNUiL;goy8AN~z^Y$cMws%v*EUC_<({Z!yue?G|ke(-duK1GFb$AFBl5eQ9{Sl<O0kOLiO>m;B`2m*T?vPcV*iG|3wAcpCfE*mml2((h7e'
    '8ura~_s19M^y*!lD@^<m{0sF=fki;!emtvS^&}{gzLqL=!#n8EG+jOg)wIsFXx5F;Kj2*|K4_ay+!T=PW=0V%cnO%VkT-(Wu+<>M7lNIose31=yer~q@gBBR'
    'nvZPos`f7*rYE_Z2M0|~rn$MS^Ay@#yCQ>U>ANe+1&Ndw-S@XBnuTV9^#J&A(>`puSs19!mht0bb2(b4$5wu)^=*13;#Q5y(31IYIJX0BS6Vvut&d$N+<IiE'
    'Uv^roYcF4UZU=K8N<)41oZhLAULcPIlVi+2O$}9U*ywOWnfZ_#CRc<JPE^P<Sw{bs5{;XKz&r=>H%U&Smi7#$9Tu$wOb9;vVobshA99PVP6W{>r6Zd)d||0O'
    'N?DJh94X@9PdP=n2uTeUsN7RXFs|sNRj=yMBuO$%y{;<k7S|#xBEBU(=XA=tJ>=yOh_FhYuPFle_S*yPFkR3lbPNAagnFVGQ~c00m!N^v9pNgP0r|`NkP>is'
    '-974<Tzl5WD7UnqWiv{YfUZ9N!}7Z4iK}Wj0?In*{atDrq=lfOVNCC!_hsUgxIjoL>jZ%h4tI{f+ArtRg6*=X_x9??@Bc?sOn>*=k*#d)t9_LFxHV1reQyEK'
    '$|w$Sn5i{~49xJ1h#QIS_K5NXN8o~R=rPPe>e7f{xI*Y;xS_!)#dSv=^fFhbn0<5m7v#aK?`Okz?FqUX`ze3;0QAS4LCVZ(MRkTc4UPP~plBoAD`*gSyOZjo'
    'u=RhJG!<n#roh;L6d%8w>t(=by25<|sqB+H&EFY_lSMZaZG}qg%~)`v;SR3E-joDqGIZxiL!}I=9f*Ql&=;P&x&w{OyAEh?k2rLl76(7ur?X7XIKQ0IZFJW{'
    'czMcqla!LzhY#s9-|?Of8!@$fQ(UWWYuS<1D_rA;)1jm}SJh!$Alg1>#KS7G4N^~kC~1K_E@8e*9OOy(mAhc9h1Yoz7a5Prfvl*LbM}CZu4fPG(Tx}pe)w}S'
    '#j`n}3N=#NYAkaN&->Bas~K7W$TxcKTl4V$eudP}pq0YJ{EC<=C;IhQRdF6DXt=}%Oi~HqzF_81u8Ur!{4g;d!c9)k{GRRg@5zqrmOp2VsOE-A%hXC-KVq-2'
    'U`AQ-#e1t#5k=<4@!VRje`WAx$PE-mo#IaCuuih%1%{v|P0yRJ6`G1N8#y~76!{u*G-Brx$7kbCeH-dh%RKR+2YUH=Z_1yog|pps4i@yLIJb6Iv(9M3L9<wk'
    'yyk}(*IA9W5@%K`GOfJPfICB#TgKCGQpIXU^(HxFl(AU3k@I+nccOpq$5iE5s3SO5o}upOXbm3SJc+J>y3MX#ynNsgXK0IKn1wZijS#)VPzPQE%&b$ppDk?('
    'pA=$Gi0)F}0J-90?KY+oLfszBH-{snOxI9?w53uW9G21#&=vf0*kAeyl#5vbUB<eo2vA^O+!yRVJM6hNNPtgaz?vJclqpLJm+OZzSn@E`yEt&MdQ4vLHd!T*'
    ';3B?*qH}|iVA(?U#bzMq$nGwCK)vXKAp2KwIlB+wVMN`68#-m8LH12g{gu-zU|K-96MGV`KtUNZQSf>ZTPxauSFi=6$4eOpqnkxfK}xB9$Y7ridc#JAy&fzy'
    '$0yOF%_Lf1j~;DB4<j^P<vu0b0dih(A-B;zxuSwsvD8ukuMS6`>?)*O?>+1c1f;}_a&$g9fQ+m20*@B3Uf445X)AMJzY`2nfIaa?o4Ex#bdoa{=7WTX(ctz{'
    '1U*|6pB9SZNpQ;1*1%`^auSCUC@J6q#Sc2bPQg7A{KmN$9Xpef1WI%@#G7w)ae=W#ji}OWVQCAnpu|vO#E7||2-?{-Yj$X63TLxFn2J}T#ZTkBVECV$KOT`t'
    'ogw<SNzJt^^4QO5pVPafs5>>4C<7zYSW=C=m5S^PmKN6nB~8mIw8J1c0`rJE#p=2IWW>UuIiJ$EI)4H31iT9z0JC5l;Vj(mD8^4VsG`EyR;tc)l_6Wc<G{;&'
    'm*4sEk%4)j5;%&@lA8TZXrs({P{?-i=sxiVYZ-DLTr8u*xHSF?o?U_{HXA^H3DL#kPLX6rWP#MQV`O^F?S!i83?%zD6CriQ33c0>{ZATpyv5Dv3{3jXq_EiU'
    'vW=&&6RUF0_sj2zZkqrZcEkZbvB(iyST5y+UE08OgoZ*&bD|VW1!xRE^1Yl~M)`C!ynHzedPKue`lx7?G9p^8vyZfKU_&)L5NCyD!&_{4)1QonXb`?9$E+b-'
    'x0#=m$)r-RO`$87vlK&OlUncn7|t-7$0|-P5OP9r3);oXq23dU6I*{I?1#d0qji{R`W%&@_)7;{8{ftWDfBUs@^T!slOEc7ZAU0zy)dzO&3CSPowum1Ck}~3'
    'aspRXjD=3Wm#x1eCrh7Myu}oW(<}8z-l!753Z+NMt8C4fG}WU_H+e_o{Tgy#*yWahyBVslL?1ble3u0m72=qX`ma#hI9oW~u?sG5VVT07O>8`7Qx<V|#P$L|'
    'UUzG6#<-%brcDK;@WiOLlUfq1$df0c%H!HSh2G}I)??D&$vanzx4H57>8GDPNs{^Rrlu7~c?mNAxM~C|Bh}maL61lhIY*O}HYu5~hy#vk<cq(cX_wOZ^MX(H'
    'WV?$*oobQBG6@EE^M=pT_Bn|py`Vv@q(%w_oai|x*&}bfE$f$V9?u68e%r6&0JfujyRggmZO9*oeRF@sKU(!ilQ!YsYP4+aR3E+|_=C~pwgSibqW``@If?jG'
    't`)^rNeD~*p&aE7)9=5-@<k2l7yZG2Xe3DOL!f`NF^sa;zGhQT268pb2VYO)-h{G8L*?x*(IyW1Dri1)q>MLgjietOnUb`9sN(ta(feI}QH|2wuTPJ+cTNwF'
    'IV06a+1%u&^=)6nAHVzUIMVYXS8SOX%K&D0o2767Tls7_@ozM!4Y!P-XI%7qgRTTr(ci3E2_D@=M1bC*_mfwO)Q73Tn2GaYaZVzK{3Pd7a^_y^DK0FRdsYJ;'
    'M7gu2z7h;x@13-uGqTw}_&>L}aru*N#9*af2}wwoqkb#z4mN!u%2Ua{AWgAxJX}smauZzI;fhr1Q=Y1}#j>qkd~`*tZ^X$0Yo3nsL#=twC^Yd;`_W~;J(TH@'
    '@Q3LS|8{uA;pURmk0TvxA8G+y!Z$R@W+yg5>-Lo=u|cC?fRo56-^!SPGG-(EdExe7$%u$t)>S)m>kBbh@v>VtG-<9GGTb;Z>!n;>F5e1g756S{bDh;t&yC@Y'
    'Nc3o9QC7$yy6*&1Yr8C~qvYcjs_wvGE&CbTAbxGUVhERptr&%9j0<U_COHa<@Z(l^IOc<=3S}!Fu+-S=N}_|tomb~P&LK2iqVFAfA)nXSyw&fb<p-r~(ij<Y'
    'j&2;y2#$*;BN6;l=R~jtmhDljMaD={1pyMF+P2?3m&jcL)@~KG0rb+>1Q>BiD@jeEjMRoHgAp_29;pY^VikO?x@yOQB~#}Z?OZz`bTv`vQkg~u+%6eaz`i97'
    '=wy>4CJ#&Q_qz6hl|E|il0wET=5>@OfGpq^8%LhpR|_D%az$-Jv$aGYd@qqI@vSxis#hdsh_`z$W%G(w!s42+!EpEB<=%lLmAzswa;QN3$;s0bEimwiO)*u`'
    'RpA3PF`89FH5Hv!Cxncm1?Pwi^Kv&m-Fv-z_~sPYnY~kJZFB1}UPXV9$Coi7{mRt>Rf96!OhSP{dWKa)&_253a<S~g{(CCdn@ncoDYMB08Tm*3VHXo-0sR;;'
    'yyLsk<Spfmi+0|;*hYwNB*AYH{Sod(w2KzAg;Pfe2i~Dxx5zx^4}tl}TqFd`1Cw*I_i}sx_;qZP=IV9adp`zYgG!=(mbP(N$8ySpS2FqS?1$f?hd*_1?&(Ly'
    '80s;akxMi6pvM15e`?a3sx<D(GhCnz)aqK-U?K8bZXUu3=Dk`slHkjd9_mp1+~vV3%+Fe}vvhg-Td9&qI13JdKFn%q?|pw-DQs+af}mCm3ov*~HC(lcPGw-m'
    '>5E#o-Mgl6B)muSFNqnfeEc$SZ5?!b8RW;!??_&9QNNKBA+;Y&Rh{^aDy6cNXnkZh<j)Lem5w1rfMC&j9=!wD63T<Q9$mzD`0YHU&+h2IbF$qHVr`x{bg}F$'
    'YoFRgCZ@^I0q~cP0CGocah-8?qRXrPT-=d1fffHPI&)wC`2D{ei&qi#2mLJ(A_REvKcTGMKl~dEWB!vQhws|S1kyk`UxK7p7z{h?q;@$Nomc#!Ka@dxJSwR}'
    '$y9uSnRPlN+)0giHoaJT8k5zS^#g(QhN_(o7AQXzG?IlWX8ov~PS%{bkam+$7CMS|Wj}|1cyf5~f?S%3d{v__d$%Twc&LeSJInIez8Pb&F%3`c9VUO7HwrUB'
    '|C*=nc2b3_{4`~}I|$Jg+~0mFGDK(*`eEL4ct6?y@DJy?IZq6fslOcXH$=S~^v~~w{dP5*_AT~XHai#pfBWb*E5Uv1%qA1GSX|6tl6xlNSb{l^Mx(**`(9@@'
    '9VrMLqVtgr#TGV?s;p9xk{3gQ{LxbK(^k{No;Y>T?<)20*JQOQ2c2W(#JAn+8pe2-_%e8)w1e?$Qs7xCsD|(MCqyQq-#9cRW7wb&6;aB?pQ@xKKNZ}h{SJRY'
    'kI}z^N@)qSo*>T!MS$IN9zs#Qr{$~K^;`IF1$BhOX(nMp1VWvT-nvrNbc8u^MhF8#f9k)f06tnHEV29`F{dQI5nE5g)i4e}9jh3-^CzTYhIh^r<gyon`rCA@'
    'd4;1<AtEtuB@P57bcf)2MbgL|DWamYNO!~eyhc-1dj~>OPmI*im~F2z&LLY;#R)pV2)0>)ZaNxon6ij%l{7<ZZ0E#k+*?*>T+Eum_xcj+$$zRV^GSnREGmfZ'
    '&g(z_bYmXf+3M%(cUs+?RXK?90A-_SZ7n)|wfEz9|M#2d4(l`T5iSKs#E!v66(xl8PN0|T+$+Kf8TRxijIU}yL(N0^J-;77<SK`HRWet8_``^oD^Wf9F`M<3'
    'NEtrO=)*zktVX+-m2xSSJf)9aB3XsIIn~sV-Bk$HaP+Q{3z?O5)7h(G5uL!=Y}!Fv><E>X?I}IS$e>0n9A<2|H$FNzu@#(0<(^3Q<N&5Q4)kQMVM??*S#s<s'
    '^gZ$}X*RpK=)bSTHQc;7-8KBr<4RJy>b=J!TAN;v<9rqFX(z4$M9Ohip&sLT&S3J>cGl_lRRh&0B>u+M@?&osQ8YcV8?CVCs?ia&EW)I>2`4>fNkp$osy3{k'
    'BC~9-oD!z^$P`X;XJPpP_!cDn-lLU|sQNy<JbH5kPyTS$n=OpC3TlSMN{GrunLlxP%eSVL!t-HI0-wD^Oy^h)525B3s<8%A#IwdDS6u9#*Lrl)dX@O-c>Cq+'
    '?dW`#-KMrM{NG}va$>E!=K-#hL@0Q*Op1a|7$_?{PMo?vX-8%Hel<GJs!UF^qQ!5A9UVr9w}*-B5%QKqBaCak9#7G&7^xCEV9Wi6Jx5`}nxcwEKdTU?)w%Cs'
    'HWQ?(E?N~5I0{o1Sc_kcW~8Y?Ra)bZw)gYs;qE6p$Gh97yV2?P^Zi|#tEkfNMtcXRyDxW-qod=!*W1V6L|^WHgD0B6rcQUiK4r#D{v<P~e1X4ik$;hQgk3Fv'
    'kKPSohUK|vbD;J8YV_t{?+@SXR=9yGR$awPlVpXS?;X6@{W^jp)xS=m|Iy(A4a2L9dQt6nla*(J&BRf~8M8kb%ygJhlYzKve-jJm4l;VN`^EN~{nKb834ysw'
    'ZxIuScmuAFct+m30j4oDh`oc8-Q!d2$DxctCxR>1dcWG<e*-dG<(F0Xe;B!_#Uj50rSo--=;U5=g;~cXtjc!aCCIgF+BS{TchR(+y|uXV&JBbw63RO=>{8&O'
    'H+S}9iob#GBpzv?Jt9=W|AuyELGZul;d{Od(r>i)`t|OMJ=jS3Rpt!0&Ie}?PImWqcTS^zS7#~u;`s12pTa+SwR^lP-E90a+CF$eeU(2sxlqGQM09v;o{8gp'
    'H>Q#_w8k%iAwN8R0Vn$TH_^`a$*vqsXyqVszBt9-8`17Ql-LA${skPGXn*hZ-f6UnnMo_5c}r>+J&-7Yf2kBbW?<Nhx9r)2H%Bk<;L)&8c26T62kU|3stiG<'
    'q~C4)lDa~VIT`0hE3i?4{k!fL^onxdJNgv`4v<{Ij=P>?tK}KWZJxrmadH@;F_8!(rD{N*Tj&qoBv-uwUNi#CJryn?Tc^&DZk%J~^H<8>^Ojt^6^q+X_Evbi'
    '{GDwP-MAmLM^vH338K{m>^TkLg})-jp!LK%Jg@9UdZd04M1evz6#*THjAdnUwXU7RXPoducMO<2G`k88f(UdZbMz1{<yi*>H*-5THa?OjmMz>btJ=aZmk>Ja'
    'qWq~#?Cd`!5wzAUPLrvK$02g~MY}&BQ>@kfm(7BIU$oELAZe(R#!qoalx}3{qfAC&q;SDv&Tl1%cHbw?F84c+<fPZlExQ_$KJuoMK?sk8!*JQ1!kXd@pPPHX'
    'ui3CY&aOs9;bhg$NlzNl-H;_}R^l{-m%)wEV1}j|iAsgvA<nWq7h2ZXsHz1L|Kx?izq}z5OT#K>dsc|l4|5z8e@IEVoEa1{`;e(DcRx<%<$NJ?uo0!pw%l{L'
    'R{4fM**Sc3a9VkoRJXr+X$*-DaesMy_~r;HxBFGlxki&8a-L^WU4=~sJe($1*lKdw7HZ$9`>TDVny^;1cCYW;H4(=so-3HlKyZxMF+(+r_oALUfBcqZVviww'
    'u~G-g1t~lg3S3VglxLy2?38Nn`lF9f<P~$(xVk!YbL^q(ohs9A$5sJ%LKu7pZPbB5v5<mqnO>wC4QGRaJTs|eCOvd1;klE0Q!I|1i_1ECCGN1trL3VuoaXvd'
    'p(&*VxgK&OJ`nUq_9Nsc^k+3HAX{;U-6BSykJD+JF^{Tl7zVv}ew&x<(c->Vhc9vK<C;A<5JE&UP)MA5E{~9ewtK=x!`J+0HS(cb_*D+gCGQ0249q?>#f%S)'
    'Tg*H(4dMbd9Er|p*bQctqu^T@^$b-6GKcpFM^&YqbIjAQ@*`2PeuQe35*xy2^bS*GqDK}z!DC;+EFgqYmcO{D2w9Kg$tULv?8W@Bmi2mXD{Gro^l&)Y-Te~A'
    '{i4ViN?}TzzDXgz=cI8BxkEbjZY1FxxHxuJB$`?SCC?F@ftAbpQEdX8Ww*kOquw`=WOoX;r}IfutEso{V-I&^HLNhXuZjE3u#dl-XXVE3oZ|c<t{H!(UU%)z'
    'cjTNyuP%*o?EH4xEA64P9McJ+%fo%Gvyt5ETJjck(O$`YK%La(;*$AY+AiCqUKOrMK(j{{Z34y;kkrT$0PKm_0g^(!<RI;CI~deOc024;v;bU7!%-zMhaOrT'
    '&$1L5_;fsIqh$q6q{=Oc@>Z9*t5GBz3}&qZ_?1!>O|H})_Ad}nQ@jK0x9h|Uq*{RdMPmOW-*LmMijEeWR0@_6n}`o8Fn3zz;X|zf;Zz9wR(pwt;7D?fLZ9fT'
    ';TDZqJl}mag<HEhX>l7=7Q>NXHe!6FPWoD8bt@DIpJb8F`>%LDF%_>@^o@GoESe@SI~7G2<GRTbkH}A?#DHG)#^%}4nZfr{=k}0BWIQiv-cgEQ8W$;{%JNjN'
    'h`R5BuLGA-tB!FcKRT*&C66KPCi@%98Hp)0TYi*~9G)Y}%Zv7Ms>Ec}%{8=|w^7vzR1Fbh>vc38bw{|NNFRc8srp^sYN9DINrxq>hqeP=)1jBL<dhf--wRNa'
    'Bw`iU<En)dVv(pye)BY2^he8K4*C;CHInomPM8udHd>EjAJv#g)$%RPEV=e44JK#im^92^Ri8jLykGXEc}|RSgOd9a@DBc`iu<}j$AULxga2XOIY`a;Uy+)-'
    'Ben^2M-BW>QtCA<6q!A(QuG<GxQ0vCW}fsWY?~A8L=^HsiXas!YfH0{3iIfd2Wu#>NR&{BEB7>`AHvTSuDw#`%(7e4-e6Fn#oq`sZ+g9M#7Xdd;f^a~&eTn-'
    'Gn@__DuO>T-vP`ke_54h`-eMl`+xltI0rA_7MC>t<{*DFlNe6!^_5Qh8n*N0(C$vxR3n+Ed1<TGJs`JnXfq5(SvEk$9+%Fdc4q>QWXC*%&C+dSlU>ckt|jrb'
    'U$*;0Z|f`o46^BKDYcPY_<}7qqSzFr4m=V|D>!$$l^kr!Tgj(e_uRud_p?p5i&r#d>ZVz7)9AXC%n$G})s#aqQV*&c^Q<2P)ejY?Y&CL@4eLYha#`YYxqc-x'
    'v~R(a)Jh~=Q+g#Q?UBl~_>&SZ(|S>!VNTt(Hn}`s0~W9I>ru9Lv!yV}VsLHXRvMT+#*0AjgcN0xe$n>QSsSXunJW2^F&{imQ5(>lQ7ILLm9WSUh1hFb8^&Hw'
    'dIN!n998_PVFr+IL61gT+j4k^WA<(pN{X;wa8cb7b;IlN3W+RtI+E(W&;;HJ&WP9DZBGo3EZ97~b607OpcEHp55{-i<q85|edE*ayfz+Q>JiI{G18_v;|@74'
    'xju4UOcaZiBzEms`{rnlD!HV9g(W6-y*F*+feX7z5nrFI!g{G<09H4<y#}u9ZH0gcwSMnKnX`9cU3X|f<-0h<#|<w=IY~nsTrOmII(=sj=t~x1!>l7`Oamgo'
    'naGhEaHJ+G6l+WkDp(`=mUa7>m7qfU3I0;(MQi1G@b`B=Be&(yd=y~aMz!ZSkiA3h58dr-4=HJ!vGE#GD42{d4{XFa-z?sn9Va5_f8Z!7CBveja*KuJb@)IL'
    '$A+Br`z{1ow2Z{(nfSw#0=K?6V!-n4?zC!3#OcJRd`v&GNe9J<yxWJlEnM1@BVZrmSJg#qr(s*N4&BF95(t~?E@{^zYXil0>a9lX721f84qnDp!ps7~o~47{'
    'O>fZH4EHAQ5V?$sIew*j{xQ}QPiUQkzH<o*&1N<CEG6vF9aY>wScBg3xr1Jy8Q!U%2N0*cJ*SYI5^=~5!zo_^R&T<}TsK_6GXXr*-|0o0#Uzd_RUSfqifo(q'
    'yA3yMtb7A|z^??UnrC-V5zUJwfj=2m&BmE1Um|y|N1(|c&>L5>0<D&wM-68T*z72)3Cb0(*;l8p)5D$Pbo=yl_w~^!qv7VaFjx64dYdZ7Oj$h?Soeo-cHbC}'
    '5&8_iaD7%Qwl2DnovL$?SM2y#-X1lT9{$n6E`m&`@=gRiVEss=w4%z7qY>a0O8S;-!zXAD)I7n!X0wd~NZnwDmU6Ag)fB0aZ#>CSPc*x(P}*Og{c%+j*u5s}'
    'wc^%1`h9u8cF-CeO*}8$(E+&z@hElKdjiOrYJ7)j(yMqu4RD|?iQs66lS_3&dyz?J!fYSU#5~nnRMTk<_6&w<3gnqC-Yw<QNZxtH^^kXaj^vCx0|ZFd$0)Aq'
    ')<@SwR`^qALoCb!dNCHciC*)H@fb1coI(SXq!orPiGmFPN?E30PMfkK!Vz!wHC9U)emNx6d3aG)dM4aBr0ipKk14nnc)KQek9+#3oYu*e<c(^Hv`VT`5>*l_'
    'k*cch;{8h&a4$3;TO<4)ZI3`1+!oyj1om=4pvu@T{>$-QF}F>brs#<1`cs&EcNFA-KTLnOIDNRy*DckFcKU9<r2U)rq~9LW2=&`IVaN{sS<(s96p_A0G{k$w'
    'jm-*@!m;11C^xk!wsR^i#CI-jWc{hsKnNt1?kmx9g%`-;<v`%_xYzC=7J!A?c`Y;Scs1oLJPT_KTXHIa7K7%X(px<&fEAh-yH24y)&L}2nizffs(1O6#q(8e'
    'B?*tv22c716tBbKQ$=SU_lQBLDs^Tw**sUlH4DQ|3G-%daUc-o`)t|$<o+L(t%*0C@sTeAMY;ViY1zP0F?DZ6debT*73JBWd_>PAeRqelYILqfZuV}9ljp*i'
    'jpLsTYm1<{oaHDPrnv@J(Rcd$phWK(vOKws;I{A{{LE@Vk=&;!ZRBJ{=znvgWyHk=OBO&vCB+um5U^&1wH>0G82A8qOhD&wGm)EDc=nt<ykyTPI+;a2EPCxM'
    '`Lf_*lY8TWZdSrhIUygM#b>Bi3b-#r$a<G1$tqn*Lvpvp7-!PJ=JnhQwgSi0;9X-dZ^~Cw`v=l-)#<FH!mgD0JXvR_VwUXi8XKyxV0F#ekaPfohh<%Jg+J^z'
    'h-oaD16Ohn^HukA2{HC+u`iQ811WPQXSGi6NEB0mk3cId#IdR$^)==;a5U2ac9ALD*@FUUgG=PjB^f~LD6X0qG>xnVTK3of{>ShCyd(4sR_}-ZbtnbV>Zs*9'
    'KYsTYLp-lF^m|@N@?3e0X*fIcD+xCPRnP9=#lVv60@}&W<M!UF_)4t&MX~s1SY`g;0nf!9*-Jl(Hs?9m#)?K^-m!BX97%aH&ywQAg?ofj9zwh;IM0Fu>O0^m'
    '|9_{ZAf6%S&C#mkq9-x<T9$E1g;FYZ`<dcJjj!-_j3iqhrlk6pn04yM?|$2lOGHDd#P|fW$;quB)m-K!ry(jwf~-cHNx5|R6EGZ>>5#~cz>f6-Hze9YL$TE#'
    'y?h8IeJyvuGvaFqouyKoPP=iEB?A?upjv4K^^lEHSN$pZ4z6NP_PLUPUD?XY$>JWFuC%8sv!UU%C<HsMP9<d1H6Fty!^RF%z>BP}GTD5z8Xg-j%-CgbkzBza'
    'awcH!H2N6`-WfgcyM7mY5?`4oSA_b`M`Y#WVY9oq)>2$iJbwDOeeKrmE`U?*BT5!*^35*kjZ=w!N*h371&y~4x%KCD;AYi?m5DW}`Ji{%?%XOW2XD%Y_F#Yu'
    '8waiZ-iPtLe5triHA~R+xyh!T*YRRn<CLr~+v=KJ#`y;AN{sK=TKshvW__4R1xS1a4<>#ue_gl771GmBH2E%@#zgfwVelDFy|UzBF(M;fu+53Ck)~FrT8niq'
    '>Assgeqry)V#3vvc@A38?yw~q^Rk*JCMl%pGQ&qkl4?9&)+-SaY1M%ZaE|5t{hfOx=LlqR=jN}ObFSk`mH?AwmuLRW4i;XRzvs)O#_$DS#Lz19u+)hEa>feN'
    '`*F3u1NSGJ4L|t$R4>%#m{zVK16z>t#uyboX@s2T_5VsV&VLtt<ogjkASi6^e3PhtYo0CVw;i+bF8zaqtexSr`hWer(f#>c)Ol85@JTHtQ7lb8UKeF6d|M^-'
    'caE&Wuk`O4TaeK_RY`tVIoobe0z7my#)K^xKyef&&HAH_jn?1Yo*n%7{lDl(cg6WeQImTJ6mT%QgrB@<^QD{lU3<vl+}+eS@EOPBz)gNhUSpI2ia-B-q6cd4'
    '3x(gzIQYgQgIUqo+OD8p&uV4Mz}msfuu%cG0)y|7wrv&VhJ)4TSbn@EsI!2ZSxsD30XBKi2_h($D}_Hs$6~^}X`&nMJn*FpEDS>b#1bZ(HTa*uL!HdM^d3Z9'
    '?{LA^iz<*Xfh{`fuB3}ZU9yz&;}E)|&J2}!xQ$(m&Ke<K6(bI`yaifvvccJ}JA+<(;=aA?)H<WVfC9D|n9D-hf|!|Jw-Me}8dG&3Ws_=8E^nG!LM{-_r|3pu'
    'ify)5Ft_*^{ZL~w=10bh7P#~&d`Afr@ipsB+1`SQZ^U|37Net|7YxCn;8oW!At^Y}0pj)Hi{1V7cz657H?ehTC~gNwr$t>APXK>^um(!(=H#m<UNipM7R{D@'
    '#Bl3RDH9LXfe7<uj#DzP>-MYlf*s0rZn}?)mN{0anZMHpfUxN4s#+ytla4*QL(Dl_c{!`HQq~--%!$lc0A9ADGGx%cKJT^}^**EyVSkm0xmwJC%7Sq)7LLt?'
    'Sg^?gVwtq&tJ@V$zWg>eNqZ?4#W!07;*S@z?IGL5Jvcq?3qk=H%>I_zQ*yPg+6`+u`G9WNGy%D8m=_DXp^Hnw&cl&rV`#0|JSLE~S<g8OH+6(~+eJDcPOZ_f'
    '#uGQ8ux=+Op>TO!tS{ooLQ|WQ_&1;Qw`iGrr?<LfRp9R$&gU9YPO@x6GXP8Wh{gg3f&~VWvc6*q9-V-4h;31!J$#7gt)M(Z^LQQWotIe%Fe{kk{#+>RIxL7w'
    'YuVWzYDop2c7h3ZH?{<)9m!p=TFSK2h#YKSrWe%xmPdEE&V0^N$z#~$&v2aLeW0m4l42z}G9)DhPajVK|2x#`iF2!PvU1U-yvMU?)0e-urJcm5`M7vWvmsHO'
    'f&3eq2*oBLk-z3iuqNH57lZa?agYoykFVs3H#@m}Ae6?YfIKB>e_&IwvO2e^3jGY_84<>te<#(ltuOaUo%@a=>YzQ&dfoIsqT!<j^)rJ``E^0m=yxSrQao1b'
    'hQXSb`Y-$2-B!a+HC36f+$5HNRe-D(#e9xK3}r-9X2apa&K^j*qh?{|TAUY*k02Mxtx~Im7g<eJQa(i-$Sc!xS6mw8-RhAYO~8**H5K<$Mfgq4gOhj>Tc*u%'
    'aw(C~;laF^CG8A^u1Jd!^OBO-kfyOk?lq#s*$|fC0c^C{SkuSV#<vgZ(StG+opv@gf1kIr-jm1XXD7Q6(3&39KHwW}Vli{@mqp|Gb$?ur$3y?1+q?-mf_2uv'
    '1I4@q0w0CuGaj_3xN!CXtxY&@!2sREKP-zud*WjT<w1mCdieL3yT=D4<QyFypQby9$H%)nr+bG7ocw6B_xQ8U(<hzJFM6MCJn3%rwm$!?*M8dBxcKz5$DeIJ'
    '>1=j4+wJc8XOEsd={@cCKL7lD^Yc$TpKWb)Haed@enP3DhO_JOtt5>a6LCbMv{~DFQhQA1;STZ~_-w1TS$h=E=RlGUk9H4szDkd_cfQ<yNog!9F@_O!Zq~-P'
    '(<?e){)4qE<o01koHr;<P5S4>Py845Q`oXc$9rFGpYEn7d+?vVgT2$e?ftz!-o{CRUYtF!(~VXLg#RMYCAiN~1O_f?hDqvUeASs{)6q4VS5UmcVzt*);1thX'
    'u1tbez>}IB9yo0eY<G^nIem3_kUl>=JUux*-abOvc|lJ<!4O?%ZxVdk_*Qz!*kqV1%MtKv<I%XXiS3H8VMY3S3!XBTBE3W174g3tE%p^&o8jcH?a_Zv=)b%4'
    'U&;t*QJ|=e4QxYP{_&v}U=OjA+YGkrE~XNSk0$*a#P&L(WfJGKEL!VSqqS=^#bP9SXVM>Y{$H9x<Wn&=Bi|Z5=gzJPniBQ!E89bn{iQXGIKnBA!i)h?Nf5FU'
    '%J&9ZCsM(+POO!5LhX~FE7M7-A>aex?XghnOwmgE!{9D46gF{60<A_tAx)JhN<rl9yhRx|l?a<OMd*9^>2|{GQB!k^91Q!zGYxg<bt-5GK{FDL2j=PLV_pk6'
    'GQS|tZYYNdpS=zRCg<8r4mZ7Ydbn6_f@lL1unNDfXeo8OZP)=-foH`Wc2KEYUf3y=M1qawn+jx5AIXz$e?mT7`n!FektZb9f(dQ&eKjwG{nS&=YPfzj8nh|$'
    'f#w)fC0xT%tMJb%O<Lk3<Tv$Q=m_E-E2AIPNWYYkOCPRA7=F?po5ll;mi5yoEmhp+*VqxL&g^yvl7Kz}v6)XW9I;Sau{8qg7F2R@J{nDdO=^#SAuxawtbknI'
    'OR1*d0!m4OxR{W@pg3pTIVYY@`-5Br#Y^R4?n>Y*CS~Xq*d{s<lfDqhJtkjfU6hOV^)z}&iR9%UIArH>A=ZJrq!fTE^^w}7fD;2~W*<W-#K(mah9j@cq3|v!'
    '8)fa`Fm&@eA6uVG!0tuVW0ZX`c)s8bd=Vp~gbGZID2>iR^cnQ(X!7LhOgd8;+L+iXSuKu`hw1C3GEn4h%vO?!Y6l@pgfuEesqP!R2+^b-5r_`uZD%VA8ZfOL'
    'nmjTPouv|2ac|e-#9(O^Cx1~8w#_-gj?kyjeVz*+g{L$Hh1e|#Z4mld$VS1B$>xoJ_z8YEClbG533)b@mFlMGM0`D--m;kNATfxCw*R3#%@<}XAL%MsTEj1z'
    '0#re^7j-son3u-=oK^-30nzJBL3p!8__rx=_MqkI*=1}6+Y9wY6+Rd7YncKaDJgKXs6xsdsUw*ub>af7CyKmj);G6Wx$NaK3(Feb8%}$dnCq4Rk7kpev?-ql'
    '+N_ujX}xKrwDiN(rq9MeQBahVts<L5ao_Y`u3CDRxH%1R2c*u4s_;=n(R-Q)fH~&koQFg31S8;=`WJoJCFi#coGzBSXVI{CgBa^5K1b70^kw@JF@*L|rRZV6'
    'xB)~Sq4K(7j;UJZ2Bd3}cd|eH*i95_0<%eF=Z2c$axncsGY$UI?<uaQl-jUShguK7DikT+*@L~|O?%K+7?cRBG}DDLxI0YP=^B7cCjiGXuDR6KOj}ZGXX$v9'
    '_1{-uH#y;Ej^8*XTZ|vZ2}hjuIkvNsm_&_(F3*>fxsePUcvn&fygLm`*Fe&j{M;2RTI+xWkFU^I0!V`%0!OcFNR$HDkSfF(wHJR__nqGpVnIEEQTLaNrAtdL'
    '_%%QBzNT<_tRG`~(L7AQ^IIe#`+pCJ_Qd%{`YWGUr+{=e<^UzD6g2yAP3BPD5YQF!ayY_j$;r@BBzgtmLw9g65oFGUuZUbrvSv%i5o^~zon>^8D5-^c48rGB'
    'rvPs;eIB|F<dqV9{ArI(frjfO+G8P43a27}0mh+&fgiIEL2Rx(-9_Bds-ZpNt!(hak&n4U1Eu`0+47*y3E88URU~4T_4Y-G9FG<`&KNmEg#ys{IJ7to>c3T-'
    '2ZwLSS&pPOg~_9F<&DPptcj%vTHE1W3xE$^QQIM<SN5!ng+;f%)1sI;^;15lE|0M4B$bjAhpSw+GIn^p`E+Zxl-S&H2kYYdSUmE%HL)Z^3-He-e&5Qi>AVEK'
    'w-|Wb0lBJ&@;z4LD4dJdF{3c29E^w&BGeUmtq@rar0$?0DRM{#^UqH90=iYfi4B#3TLTrRaWIb0_6uaZDKRs)Upk3iYpo?cwR1vY(WU_wk<2gJ^kz=__OzO('
    'W@Ch7nTa{0rpjHM%&b}3kEnt)U-?6VZ*t5%3TG~FUj{TJgo-zl2TM@4q4OlZ6}UxLw6_W^Tx#_U?_k>@?{aYtnjS5C*!bw63nb0I4?vTfiZZ&vpE_SEjc*oC'
    'ZhoQI-`8zN&?1KI8{m6I|Je*uJ((;wC*NpDe(BmiZT!m8PUOS{t|hz?oqm-rQeb_}5Pg!2qo>x&*3oAN!VBqF=`JZ%y`2BlH6M&F`yCTCL;jVM0EMbL;y1J&'
    '=F$C?a%RnyfSF4-jspHkBL^H{NhE54HK&5KAT|LpSTWuLCph-z0xki<rlN0Z<J(UBfy6Z&;U)c-%a+>r4N^5Jfi@?O<f)X>{c3I<L1c86D2Os<1(l_MVhhMl'
    'eid@NfV%38)ITcz$8MN{10CrPj3Z6>IN0UzimtdSdQ1xJagnnqoDn0+1p7E&cAB~25dYos$wE#7c6jjM#}OTy9G|obFRn2W1j403`j#^<vg3>`X4!J{nzal>'
    'L2s8U8PaOx=&+V)CM<2Bhyx%9m`@w}Q7K%auttQzH2LxSe}S<TKYsUrA;QCNM;x>Yqe8FXb%K?{99FqI4{h>4{41tOHcA@fUjJzyCaC}8cmH{Y(ZBxzDmi(z'
    'y+#-eT#rw42_uPf3{EEkEdjS6NTIe`hMeGFTB@Y5BT*tFphBcV(UjX)tP1KQykKOC_doo$6J7U*S6o$2=G1AKUj6v}zivlwuYUL|KMeebzZ^$5Km6H9VO{)V'
    'WA`6|3Dtu>2xkv+T8+9!OPLQ7%=nFqHxE0FYJxaXYN|D1ykJ9N@aY83y|2C6fp5?vk9UZO>Y_!0u8v4gMJxhR3RIynFGzGF+6M;dJ(;-tG_(0Ve~+Cj>sq3t'
    'DyK(t1^LGj8q6skuls6cXJ7w%_4R4_<<1jd>E2SWW+5MqQs$-1!`WjKd0^G(9v5HO5BdWTE6oK}!pxP(VdlcAmq4cCTecy2*p>0S^SC@g%8J*qJ7hOK3W_i8'
    'rF74}b3!<pjG*<ax^Pffj{Vj{3<UYT)bM!Y4ji5T${?NdH;XUR+~>k;RNaXv*%6&9INau*y{41q6CmDv7A=#alhq_D1Ts&CEQ?&XNh&NOD$d8uhXNN46bIv-'
    'tpe0hvn(Na9$=LCza&BYXCnTcX$s@1te|ao2azBA`K&+aO0V)Ak{S8vfvX~G$@rx?9KGAC3}Lc~Li(esc}i(+VHL}RKYq;+hF>B;$Koka?$84^r`#)Cjj)k_'
    'Kl9~w2D)$xU3fV=+&R`w<GjCLN(XT@tNn3*`~~@wET2{)3GE4RQa8Pv<(yEq$y`rw2_AryWO<t8HRE)?%CO7<|8lKT&I39<JKy<xgEUs|l(0F)%-Y(+bza0#'
    'XR?l-{TPM<To@XQW!{>bEuSAYlkrfQ2kN{BkLyU!bJv362FF}ZMzb+J(8!XiZYlgLBmXL(<ymiP3}mpoq<LI?5Hbn~dd5KH%rm#7G6{YbCq-)!mnv%0O?w6o'
    'DU`%1;9*echZSCMhaHTjU!do-`V~0xi}5PJs^;#U7e*Mgc*;xB+et6!LbOTp*rgc*F5>4r5m&J3B?(S<^lnH!fUeTi!~xAY9(`F>cE}svimjNn87&yZJ2D2F'
    '_}`S=f`n`gZ~EPSdu`b14Awfc%}23iHvN&X)FLkP%QGOn5R47!N5-&Gju;ai?Ck7=y!HLR21#u)><wzs&X3>y+i^6!f`6t_p#l-52xgDO82Ia-M;U_fy}Rmx'
    'lt&d#`XG}rEFYi4D#~~NtRG$e_}#x7M&m1dk9GYy4DiSA|A#({GO)9Hluawz0kP-51>vWC8~yNC5xFfxZGZjm+a%ik1=Y-P%|=SYhHLKPaCTTAX6`hS78lQL'
    '!Q}J7RDC%-kPrqvol9lz@&tZ4jlg3303!r41nz19UG5hRRkoQ&$x0u72oJxgIww%UaU8nCEK~JJkPBjGI@y-Q#Q2k_2{R-PIOxPu%*2QH5PIV`wvt>7#M;v`'
    'EEzg$A=?*iuBLa1w$v43RsDX*#<|GfV!5@`H~Nz6gkzdq>@Bf?ToycNgr3(5lycR{S!vI%yhlHBPp-TNxtfwK_`!S5=xM%EHy+W?&@?9V2*ddi*A-DVO5G(?'
    'zpD}K2|{19KjXvbQN6ZxF^^vMpD)ty5LZ{IM_DFUXknZ#U&C`xl?TIsbJPUOsk}#nd}x4CR7C`QKkZE#F*^IjrnCGuMdyCw%bSx0#A|$YmwpP$1YVBgxt__o'
    'IlH83UgG6K`w1oxP5^9*LH|5%Rm0Bhs3Q?&7Kb-T5uY=%0Vp-17%5BR(CTy|V@%4Ol5w;x+=4d^i8z%E1PE(@#$w#^ygoosCjGGih`=uC@x#B%yiqBTBAnl8'
    'Wrx&L)~HdcS&;CiJ+j-Whbp?ZsY9~V41@^VH8hB$Iab_)sx|i}-TLbym_y3!Qi>^vG9JE+0GP&0a!#iC0T#Y?GrjhTyF9hb5q}iqsNGxutPbR9B#Z+mr6t#}'
    'h(j*&{nACPGR)=+;WSH?UrU<W)zfla#9z`}V9w9CQ7`MXF;BXvOtG$zr*T;_se19^tj&h}&Rq`Yp}UC22*BK}EVvTboPXu@=mxi0@1{SRWoEyDFyBYV<^B-U'
    'WT1ajTSbTdFhh!@&Xyn@gA^cqcEmfOkZzB=g02CmEi?J!JMAHcoKkAY)cUQE4Fo07!h28pND8;X+~ytWQu_0$I%QTU&1;pjy*eUf7mf)r(wa#NA58k^gn-QA'
    'S%WAJorP9X^inGJ5vwqPf6R9uNWub7pd*5bkQ&%s*IkLnPk{tx488y`ff+}ymF$L&2FPIjQSAgeKD@;2SJ(l2Wa8u9ox|f7j2Ryn4Ux#4jNS!}u+TwD{|fIZ'
    'Drq?>AT5556f7VEG<J<IxZk<=Ob=PX&BBrsp5HY)mqm&p%$CF-1P^EXoPe==oa!yE)nbkUXHqiFmKr3C`9!9T;|p@6Nw|uioH!j5Nl<~8=)@3&Po-0O?3WNK'
    '>0LhWoqZfsoX85Poah;jF3jhQgF+?i`AN$Wzr|asKpJT|N^Uo+U!@7q5;8yPMPQ(za^^iOrIe2;<}+x1N;h{GaW>E*9CdECkZHyQ9k3N6gg>Apw_QHyuqM5a'
    'rI=Q0*L+y6TB%04moi$o0Vabsj+S$TP+>Vt*;OBwt$QzOobLf^Ca6~H%qtbwi22DF*QAmQ%JmrrGsH=O@N=@X#oZJ4)?5><qDZi8pq|%|-aJJCf&+n5y$Fn!'
    'rQh>8D_;69`j{KxrZ+UXY!Bi4ov&=tN|YdgHAK>Oix)C5%)BWyjbQfiyGSDbkSH?x1q!P}ml+nXe!3^uP|+PH;mJ1wbF;pN@?GqH(yC2Z9*Eas#2>`VCA>H;'
    'U!#XqW|poRWo~?7$760Nc7$hhjqd_;?lRK!+kg_64tlfcwSxRaNIs~_JtBQ6K<DGGC+Wi4Vo-~{SeRHMLdY&rf==~2GQ<-3!MQy3G6!lES0m>a8Wn(YP#1ai'
    '@!`W-HbNJV-1CP}x=YWOZvU3o1$FtYNXzVqZ1l>wP2v>x8YW}};W85t8fBUCgk#0)#olT88yA&FbQ><cT5~HP&6V7&{C0(VHF$^SE;VCHm$xMJ718{RyIMmF'
    '267i0EtQrPK1DXDJ`XV>$!c1)kDaH&T%pvPlw0&C*QA5?*o`IL3x!!`>eB`ZXquxw6|xKPLCHJ)T;paUCO~8*VFJ_>3vDSt>ANtuu$3omJQJL#9E1_JH!#ay'
    'AVCWi6Tsh|4|>;_28LCvNT)EIW@C}Hl-euNL-FONE~4y2Kjd@gnn>|In@EX`t-;VTY|Ewj>qZE{(GP1BW>bmR*%q;xco;bnKp_c_onDx0V_mO}z-Jn#hq7L='
    'i|)zO)v@lQMs;AC^bTtNivlamOh&O#kTKQ;EAjVC#m60lAQY$SHtvhx$MfVqQlU--MSnXjB(*O2c~z`r6^@j_>bVs}<!E>dQ(x5xz++PL7Q?LL8js%PiIxa('
    'vf~562t=;9RJnj=N-%;Kydj5O?ZAq(>&-B0Ul1{z4VhUWzda#od}K-8A2O$ggukM0>cfP<U^3>iZB^J$+=A}lc=ceTa#_&nf7!ObUmvk*I7*|5reC;6H`{|t'
    'U^u2%*Hud$C=po0Yyi<q#|<JVg;CP>M2aP2D%zk>DNeZpK5CSd%wY9_sWkz+v;+;p%!S7EGU`=rHW}caPS2k_7H|QLIHF8!5@)2YaAqcfT1W&E+&`>2KZnUP'
    'M5H){(l8q1+%@>9>C)9JOqt$%KD0g(N0$*Lt<Z|%nd^Sxl%*TR8x-AE6cON+GpB&Sgod}dw;;U{ACuf`{}g)S&CUo|50sWhH+{_M?amRfA?gn1GW<TyfRr&j'
    '7LCe0y%#-lrAS+Y2@Ma7*<jp~V@$60(wLH*vQB245}A_em;-5`6L&GoGcn39Rf?;Di&P>xuxU{f_}qkqHO6T=zU4eNX{;r!3b53BcIAByJkg|0)^YL{pCH*g'
    '?+@FP+rnk9v)cSfQ7zywQZ|d_^K+lQYpU|eJg3;f{UPW}X2hbFxQpqYl^A_yTPfH5R1+)E7O4t#a3c2QH^xFpr?4CH=iDNosStU5_<YwzVk?33g9;hA7J=~='
    'K=U!^n!SRI?X7B2${i%MPZ1-^fh|zQQ{AZ`R(!F#)-{KZ$W=#F4Wj6|*{VhakB2=Tj?Y||K2^JOeyIlBCG9E9B7OrKoo<0Nu99<Ll@6~ol`koeO<{$o8uogO'
    '@IWD*c)?X+-%WQ$1Iz#yziUs%+i)x~QF9Dc9*;Q5<m)!vD(|;mz*BhH-GK{mG()XqJiUT?u114yTs_NgI7ExyhV66=GXzD?NAHg?jWd3Rj;XKPZ+p*yS?yuV'
    '{cGfb=@0UMpuhMMq(8e7_n?qs2F!*+JUzXGj8Pc34zT^m7*^aFqOHyo)AHuOY54i}xX%b_{9EDdGK2=>Aa#BL3-9<EqDZ-Yv{!;4llZlm=nh(JN#pGJnAkT-'
    'IfW0&o{9RBN_&!2n)6%a02NE|6wBZgm!T=n4Xz&Fb<}D%Xne!?>5#i3dMnnWuKFpQ1xPp<3Q4iu4ciA{Mp`8M-oI(AR;O&9+a6|Av^mT#e_@@WHxN{nNkm=A'
    'LbdQ6;l`slKd$|{HyLG>%G2u8>Z7FEMP$GRJf6Y9d73=qlFhC9lPdfVyRh}?r+(vjz3|ztaBjQG=9B8?6W|TAtM-^Qo%{sqKU3%7Hlgmomr4xeTYR*0xPN$@'
    'K7V<<_40W88~rA&Mtlp(hlUVTu(s+(6*@+rpT0WYJ$aQr-#geo{wDfO<UBh(J$aKD1VyGP)hLx3T}o5owy476-P7ZA_v_Q$<Ad#e?%}KLy@Pc72yTh5)04f|'
    'NBg_Eqo%bh+f*~e2<bu1AygGDiu!MN&K_5{EEXe@L+NIKOjat7Yt_fKq`Fb7Zq}0G`dB(E^UO6?aoOEQImsCPELP&vP^oBB{vQ^2et7(1_c+}-JUBVsJ~;I@'
    'z(y7Rzu5qzx8fQ`cLRQ_#J2+67$(!D+H)fry*1OR1iB)%Q2TqYF^xt_`)A>x47sD+_T=N{fpa<zP7+i~(2ixe&A|)xt$4!`)N%t}3eqy%^6YW1@Q;ZLOo}{&'
    'CVt_^L~kVusl-)}TdrESyQ0^f3jirDdpTXEdSTkcn#{4NE1BavRcdM>`=+GgQN@U8!ZJ2&0m}y5a(|2mBx8~Vf=b_tW>E$^J0s+SpUxgSS;;QBiUG0Ic&Pd@'
    'uCN#xW03&_`%7JMP=4H8WZ#Lm^A?=wdX`PecnbRuMC-wAgz@S!;YQ|5+<7oM)XXSMS!I=T#yzqgxk(|e(J|bTi>>I5e^!L}S;=4}2666(@pkX_FD_tT)T29k'
    ';rY4!E}P%(X?7!n0%FuIGb|YD0&_*=l5(!oXH6^2_~H3fv>wQ+BWJX4IY`|#Y?qKg>0<43<Ua)y=UH^opAdFjp<EM0m+~BIE1Phg`JN3c1+>^^yk^n14K}S%'
    '8uI^K8;;hH9=<ofy1jA6KY$i=dTW!OEl<<kN<1Rx#%$ehsbo@g8Da*A;U7+k)b~_m4=R}xSq;~R%6?OF3s8YooUi*AQDNN*JN%eeBDcrYdNo0G{9)$qqxJ<l'
    'A(&a!&5?RRP}pFvY|Qx19WgGwgFNiw47*agUQ_|nZK#*hZ%ffscy3j>Q?1bA+hkP!@YPqqo&aCn(3*3I|1++eGq0Za=8HVh$)y=f8Iz-U=D5-Zo&cE8KFXK)'
    'T$8Pe@3;E%)`OQO0(E1zC+6ln(I+RpY4DS2`_f*srs)(P!-Pu|GXJ7X3HW!_<R73nI>>Klu>UiOT&5&b1rNz%DYSOvMG_N9^FKGBEA9v?xl&0DS!>*<mE8O+'
    'A?V`8tB!J7EwoBnRY8VUGY`7TdF{@1J=BD)?V4i%>_34#;qLN_s41crCz#_-wxP_Y!D{m+xMIB^FV*DfE={HKsLCzE_HUo}vqtkG?u;MZ(X%<70BxXQ{&Fid'
    '*w6kB@G5rv*Bf5-3u{*vB=4>IN3V8w_tja|DVjQCOlq?s`iniRn5|dA0NCcI^_KDHC&mw-QrbhTf@c^XWo)aapjk?GBL?f!_GLY?&ZQW&>NJjK1X{kIHC*6+'
    'QvKI>HthCvzWaTe%KPoS?dN+)*082?+=t}9iN08T3i8{_H9U$-<DflMe@6N|z~=jvcu~PQrv{{PFkLr2%dO)`(U}o$S4_J;?A%6}FA@=l)tLc(ATIjvq35HE'
    '3t9>j+_Q+qziwlSJwts2^3@ID2!1AK5#O6{kL|M$mL_taXt;`e^pz-%B&nt8BByx@^qms>$|1FSTsZcuzbhvS^{tlk>BVd?aK6tkWjxAg2xR9)lZk|8r+N^@'
    'xH_X3hN^JzHon_0yt{69t~i-c{Ql{a^a&+b=_3z-!y>JOX;P?AJlABK15sV$9>T;LsB9#Aq=ikrkKbjvm#dl$7;-4=xC+-K(Ghmq6j!op#6OGG8}y>Pd4eXp'
    '>&(CyJBoT)0U3B;MU-4H!j-|tZFeaMS5HyjzUjTk=nE&5!@Qt`#x6Xr8|(mG0XFxFqw?X8o4)2E!JM-hu4Rwhd?j;ERWn5Lf7dR@Gy81Rnb;ribv=QRny}9<'
    '&TwgtADTXYF46p)3Y`&&F`<cd;c|Jq2=!v$k9rW<H7AhQ^GI`j#9O(PTSGTtve@Z3S`%7YjHH(OR&yG84d=c;9u$Og&Lh{k*)m)NM$rxWn0ST^`*#M+nVU;I'
    'cama2HU+UX!Z9p)#~D#%nX*T`vph(HaQt;h`W2$?+yE}}WnZ1+?d8O@6e{;E&i9O=!TUGhbyR+-x-()k*II2u!Dr{g`EO6~CQeUjox#_2%HU}ja2jMP?g%Gj'
    'iKQw`_Cpt(N6#NBaqM9<ZOD=G@<(e^?iWfwC2v;VgzfY%|BYnTs+uynxEboYwX370+|Veopn0s9phe*F(pPe?@>bB<#g*3=rrj0dUv>~u?XAyYX+8x#g$6tQ'
    'PHTos36qZBxm_p(@0oEf?QnjD%Mc;+_k;%ssm5SqK{pxC^3{7-s_qCv0|-NmnUomCD{4`vELT$2E^+||n0MtGobfuG&k6@;?j<gvRd!lm&=?DM`o<+lDMn>m'
    'vGoPK_d;BJGVLSYEr;zW4K6I|HS#Ma0&61&uqGqPrBKPi?7i&|y9__A&K4yYaw}mj<&p8iPR6Cs<joItHGaK&y8UANbi2&ixIAYz<w-GpDCp;xU-~xV28*}4'
    'V<@Q3!)P_hHVP1WoJ)xFO;!?4L&*uH90y+8OSBypTzG@tQGsg~+G_I6+A_B)QfEWUsK8f-V!7n3piV0VB@axKvJN~=fh&2H>2fyv(qox}?OQ|z0(wH16zHEW'
    'QQ=@VQD@$d-SmhtZa}}XB>F{R4=!RhHDGT6(<$s_uAO-vWk|1W<zE6`;Zj0Jx*A{ne1u?*(oKdub%RsOT&Qi#piVO#?v(@q_bw|6Rg3Re@ld-!pE<lHv)LF#'
    'a+`ffM{-B;5tMH&H?zr{xA{FBd;<cIYgTmh96%?5ypKk>%^hV2<vtc27-a-6%FLMD&WrW*6L67b+&t&poZ%Ey_M1RXdfiD?=l&xKuA4>$>JasCdv7y9$VqIK'
    ';!=GJT`A1P0yL>_MdM_ZcMNrAUetQ18bCeh-NT2THJo(=<eUfp&UwP=Z}v7te@s>%IIKCN5X{3)zl%49({Gcq<XaQm>h6q%XjrzG?tQ1W1^88E+rxULPS{BR'
    ')?j&m$k%j2x-#H6XcG%TD6Yu879)nERyMcg^D28`HW{pZkr@&A&GEh~1_Pa#4Utk1sU`d<9Opb*`vuty<{<>G1x(+!wUTTcN`++uETF`T!Q7S-orMVgixJyO'
    'rmiq2niM>wC0KP6dj1&STDzCE2l-cj&KlU8z*~_1B&q8?7#a)_67LnG?6w;sumhe&SM8e~+{R`73G+NGb|btPQ<YpwG#}!mWL88q`tyd=Rx(U}sTLsnVRFA^'
    'DL6+lL=oTm4Y~_Z9JCSiRcGQ-M03bekuMtTJ!6Rql(i;gVsskLHM}|#rg?pg?zqI=t9(#&^%YRonyRWbJO*;|=nR(Rj6s@XP#nPSfz~E2U>^v_c41O3xFGNa'
    '6-3CPWXCAPRJFp>0h^t_bHvoSeP&b^Ej|qTT$B|hJ|s@=iQI~ZJD1~sNuntE9FC$gg<h_J5UU8hoQzeyc7wf@ZW#Tey&2FfC!RLA>!CiaysC{m+k5U=*G5Ck'
    'MvR`M>bMmzfw%JymQBJ5M^9~o1|~{$5hm$;&V2E(D(JS7{ad+74?ft*dmx5Nj>g4noRG1$pYN?%MQX(0Y>}kNWkjiVw;OfFk2W8%tH^Xjzcv;b3>TT?9NgSe'
    ';6?Qo{1BUI2n7l25nN-zOhU++DucvymG)_THvcav(;FGRq=g%~j{!hGFV`8t3@z1N+FBNh`=Oih%QCfVpl7ZYHqc~HAzQd6S5n>w8egExFQTa*aTYP?$6$mw'
    'KO8NE)`>EB1qqiXd8X+0pz-nH&};kYhfO*{KMzNd6T8gqm8?QxN=btTz1}FY{EMT`ot_6davNKIe|8<@MmLl^EsG@&#gpU5Nr?z>|Cn+=Ffbf;ZXV^cdfi`8'
    'F}Y2`n~=s<x(!_nOU|f~kK=SO>5Yf1DOnoQy_#DKrT=jgszrC$-Jm?wzc8y?#-V;(fVZFKKC(#{hrM?d8}c0fXjx8@1;P*>n{!yPx42b^RpsRHb@`N4yicQt'
    'f@n+s01Jm%R2EcghoXEbHIkHa_bUX1LtXa@`2L{4?-d9cldF)Yc!++j^QH<4P=uI6;lCs>^8%J&Sd_v+ID}MK7~u+We=pk8s{uj$)QEtqBLmX;ybmm34m;-~'
    'Fw&O-8F`vu3PKQYAM(fJC%m4?{WK9r&ftMqGaMctaNcLAq9O?DWwWqU!E}#}r;rm2uVE>3NwW;SYSQ<~Wn%;8@0ZHe$!oM_k{SiElq6RWM(y0Iab_uepa#CF'
    'aYKew@<EpEUZ!HL!S{MF1zfJsvbmD9jv~pi&#J{Q%IcLR;K*e`Tp}H;hSy(N0=rt)h{_%-BeW}N5UCXZw$iT`2fM&8HC~io5398K^+0wTA{>p1T1NJA{lD&&'
    '+C>_=Prj7^{sDOrU^W~dDwO^F4BUDV2<%P9AYd8lXO)!hym_%Ls9(2H)<c_aL@|}DKF1&cmX;59j-z)Y@`LpxUyB$9G>;8=cN9NhL9G)HAj0eycCdl{q4*m>'
    'iXO&+5$gtjH^D=Qd=U?_h7^%C(lPA4H9zUsc+w-b7D&gS_rCV+Yy=EQTo<oa(`|#Z-T_tuFT(gA{A~TaKU~icDPcCmyevoGoW43dINCmaC4as;e7!4wzB$=F'
    'e!hLOOF@|ovOv!d4)G1b+BY`syVpni&$oBJ+&y^F*uKQTdIyzQd*tTy<xYLV*-@~9{TJ!}-t*(_<8M;zPXlA^Wu%`(@!5mlyUUfRe^URw+@m?c>c{FiNpVaV'
    '0RWdZr3@8^UX5_l9c5Gk@}jx6iI!?v6>k=~!@Uke0Vonc#a#*ZAh;BhApAu_8`qo=t|DyP-}?i%tJ0E7Pq@7z<v`udJJfkMV*K9UK$7b2@#l|x0$@hM2P**7'
    '-<fsWo_+~lqDNg;lTvg+RRTGoDzV)}=rw5fk^QAxiLfR4QF`*t$<E=y7ke+!fh0ZHe(k+|b$D``9_??ResOsG+T+EcKeh2_EIN26M?Ag|Zv1HPC_Ok#_YPj{'
    'e%**Sq2gCxq_5sQPY=KNVt?;om!F*;Zy%gs{ji(;H&Xh=-pTg!{oV9*cYpWw?&<M2{Pt-3#S3_P^x|t%;`QN+-Tn0B@XhhgZo2bocjwDSjC2@F=MN3l;T)G3'
    '&WjU;gDa66u8ayPT8b0fz%7R5;AWwRbYPr^ueZNW|KZ{DlZLZ?bP?+doa!yldj*O$V_HMp`dSy|-0Kes4s^tk)LCx~Kd@U=qb_@#uPb*n3)KQuc=C8b#rf}@'
    'He^>rQG2Ihb_`U?WA!OH@Sq1IJgEd?tBW|2eJfv##_Ym@H_FSSH+1Jtuh1W%modX*vHJHPM<=hNr?t<007=yv7+@QMuOSz4IJ6RIPd{teUs8amoK8Ut2iz69'
    '!zkyW6CV|!mpRHq&^e?5g=l)ctl-AJZvSdDnb$N-i1E4bS(U?U`U=C*bb5naMYng;@4#%f$L;ezi1oLXKGU;4DP2&pSaCkl;^h97BrUYKOE^3s5g7EH6?2VX'
    '7$VQ2<QDO)&o-rqL~TwmVAjuS!WDaQNJ!TG)FSes;Dw3xyN2-|6ogxW={-B^euDq4@t@Ey{77kg=SqF%@V;Ff-x=?4kWm-Ix31A@Nz3c*RUxdj_O$GAQiyhG'
    '#;1qJJFikY_}lo`{@%$c9k2MA+(*)({ApQMg|%8>9If7#oO6>;SVeQ9Qu@Wx6Ls8UEg2kmxb)fs44jv6!%X%w=VYLx)wB7TO}O}QaO)i@r(M;?Igbh|IPR?t'
    '<+DWvmJ6uQIzN55o|^PJsMCMQxg9+i0jZAN6jf6v51SjUa%F+r$HhTuvN_m4@8Z_5vBK^1q;ZpI;<okLuwTjgWxAzijvkVtb~a@P$5k^SN{Ei4(c<?5)S$Ho'
    'DY}JJhHULhtm`PYR68GaZwbIPqZs93r&j@riC=F6^}HM=Ho>7&WXeU*KavYfYkh)7mW;2sDbgG0K9ex=z!=s9W;!uZgeJQRYW+JV>PUzX)MHp@e#18nKg0`W'
    'U8B|Q@;hJsj4!Z~HHB$~*%^{MaIM2v3J7yGR`FD9J;VZ5e%V<6&F}rDiBu)4w-WdIG&6tqws%|CifceSMSB^g^+Hc1rYgzJJw2>qX%VmJ0Zk%QE*~*85>1F7'
    'b0rUz|JZ4GL}X_Ns1CBTYh%O-LZGi`=j*R?g}(l}SZe>p$*b+--4}jQeTvG4{TJ|IxR7~)<(@CAN(q8-nULGD0>js7u;a(s<G(0>JubQKDcURW&<$BFdg<l8'
    '2lD$DsWb{UbZOVd7$MfyeOjfyrMPodllQLQonAFIH#cb0Aj7{<V?izkXdEFErYor;$K{$%B)@>9`|Dj3-220zm4y76vCf4?2`Q4xA3}_hv7gAy#(M`l`)^+C'
    '9(@z%?qeyxh>9xpt!UjaFs6~UD>t{T0U&k@!tgkanpboQWj^vL@fDfgGB5s0VHxPw>r>Rav^tOT*ow|z;?OaU<x6=fzY_MjOWCA_XoNDCyI-HCCvU#^V()9x'
    '(ZQLt@9==)GdP0I!>_rd?A;W$G($Ji&v`fx_;fTHK#Be^pY(146L2)aoDkEiHb*BTzDOi=5qGKHX%yVdTyi^q>rtu#3eI(T0TxY(z!%XZI-P3NdH<d=QL$$A'
    '-BrJHCE_AsLiw^wlH|nM%MNm%@#QC<xNHF3_#Jcizkk1k*%J<49n@&78KD?P&;X4ZT6kt>1TTj{|MF@I%RVAM=kwkL22+VNp%71TQQQT>jKfCS3mtRFtE-mi'
    'bP|sYM{ArLC-#Z0_sRSBrQaNUwX>7%>^$E);5LoOX=XPkYl0*UQn?Du`1e;R5QD5Q^75t;mfsvahaJ6(-GQebOE;UeB^GZ1lj^XCnxng>{<6?LM~MN)7jb3G'
    'A|T8wFzP#wq8jF{3cCVk$bASi<2plmooiC9twrLFHb_u8j$fpRI)2tfSA=nK_*w7^H68SmVo(?lxrxMknmKwMJ4IJEEE!>!Tfln;)#XmLwNC67^{&T%pw#*2'
    'O2mJGFs>Nm-T^9bc>XJpjG~9_VTP!V_CeMgTvXix6J*ctl*}I8?vI9-=@*mHHU3d~yp=qQ=5Eced(d3>MUTLy;jgSo!6|oV54bIKHsilwZ||PIN)HaVk6)ft'
    'fw<A2&K_VO{lT04eK-sB-wpm_K7VH00SZ2Y?m=^ev`fv=U)YxOG<}7bdYB~?8^})}_J!M6+pKDTsTru^@CAOBe(~mD2in;i_NQt7`^wFz-}PpP8%}rNP{TR@'
    'eeRP?Q8|lY$g#89mBMl(f8%dT_vX?YpKyneYQ>hO#A_Evdpq>U;g)0@Emo%YV@p*8Sa3upCFx$vJ?7ZZQUx$;9E`dK1j1*LLI>7>^7bav%0|^<2e$~clgj3Y'
    '|4e}t3&%h<anfi!-qI|u@s2aARCrzLwfDi{m#_d26hbL+2UjtN&fL(;Y*Wr$u41CnT&6^oyss_(3F=DH>+8;q440QBXCdYre0w||+`?rGm*hM61LSVJ>4b(L'
    'fQi+;jr&7PA$5vcrf93Sxv@_AlFdgvPhpK4!_~RgTg&-}(iT=xz$2YHUH+Ara>WaB`C2yZHaeetviaEi^;t|e0uFe6G|tvL*`$NY1nlT~Hik*MNeS(w#>fz`'
    'x}!-AF<&vEL8m=vPfqEF^Ohf{TqvAtm4~=2+iDq)9!QhO?s3c10bVmOStRKf-Q)6t5B|qs@ah~d&yOyvl{awvn^6ks{u4JUh5^z5$6Rb|m|>+eh0}$zWkE6U'
    '{E;e_MQR@qqbOV>Ct8d!XLkQ>-KmW^&9@+R8;LU~aF@1G8IUg}?9D+S3}4g9<yx}Mw3bwl`hjqz+)G_Zlt{|3A7L~9Tps@-PKuuh-ev9!e)@|zDS&xW%ig-1'
    '@!e^vbip;L7oaf-NJZTl3ZdMdGeal{Rkj7A*W+JRWt1{t<gPLD>5&82gR&al7V;C`SOgz^Sd&Mj-yihPJMhlK8N+OA9po{2X1F<Lv&IsyS$B7^g}TT|F*|gR'
    'qOb@57<CB!Z51e3;*0(5m&Bv*939!ezy8|%z5l{KRtpsVuo8gzWeUE2z4KN2M|;3F9Ujqxpu#zs&&?%5eqOh!I5k*EFLOsT3~GHjwiE-EaSgb17G6p+v&W86'
    '=Gh`JJ_K>X-@%mZu&u@Z&+%YVwvx1xBz80INs#gfWL&>ze)XKRs>y-eXq-yE+o8!~AU2MJYd&f%(~pRx%kkw1rq&(x*!!Q1C^jvi!LC!(*;)=rjkmmr%&AM('
    '%c*$~IoYA=mi2f}>V^?eJ$5*Q9d|y0!3;*Y!lb|A43=%<{CJISJQzqJ(QO?;w{uw!6H(vzNiHI;6qS<JNme}Wob9KTi^>pnI3g{IM`@DOu6pm)p|n}wQo&I|'
    'A16K1C&$gTwAT8B*qgXoB$NhDKZn2U+Q^#iwO?<necqCP*IEx_-;*c|xUBFEV(wD6NP;y}){WMrL}s0O8zhV2IgM={C(rqQWkD7O-(yD^Zg(9W+vrg)%eFoZ'
    'v0BBr&#cWEA)WKD6)ZQxI3aQsJ9*XU@Pz(IO4_;;b|U9?n1794k{mBj6b#?i-9YpJay3m&mCH8e<Cf1O(mPfG={99J5OY#c`Napx&wW$z5w9^brmagM%!;}F'
    '93MI&A%X~Lv8_vhl3fQGEU0EQ+K64<cfz*db{KUgsSshBhtQtGaB)*behVeRJrmd=i>fHuHU6dvE}mBUA?x=uXZ<u(!4{Om$%1Z{Ds*pb)$qCX<`)Ho^A*-R'
    'a04kRF_VTaeM9G3qM%@Y=Q$JKJ5Q%N+zUX=6sYU35dyU&-fb~X)?_xG+CJy@tZNTkjWx0#bbE+qr;1XgX7<W;;Z%(C@r4<C)vX}6H+aF^_h#9Gxb$J#e^j<m'
    '=-+&+@|`ae@EHheHK=_@n^9sc^Q29mZkT0od!yQUp2_SjVU55NIm0g{E<#wGqSmCGbISVa^mPgoksj~<;hVkV-E?Pr=hZHg%y=^{T%x#1JrqIY!A1^pba*ht'
    '^RWwW4Q&;;Uwh}H(c5UF%~13HaJ&5sWvy;4E-HFm(~u~8@?4I&Lp8{WDTHHLXHEphh>DaoWGLcdcri&5Vkwdij2R!F&h$dXQ*dpN+bdUt<J;I4T$y)(!su&)'
    'bz4vAHMHN)YvcYHsQP8>yL>Y@$^}6PfzcZ)#5$(#fG;q3d2|^Z1*fI#T{Go6?V-HVp4`5W)QYzib~R`)g-6y7#?hcPA<8z@FI?6|pXv0N2f+w8l`^zY-e}9r'
    'S81S3ktpztWK)vEBFz$cTZob2t9KzzXAab!KvvhT30<{6=KKV>B2}Da{B?74gHQQ7T$b4Ye$ZxFW393PzS48*+8T7-KEWj1k`VzaT%!PlxLRN>go&ksyk=aB'
    'nO@%?E|5P#*hELR+@ajaNBnC6dWd@#$D_gR<!JaQ4kj8TjrEZfktMRmvu?kAIUHqZ_ox`yVcg3X?fwA0+u)J}ve*XUuA9^sab=4UR6?uks%=Gc@_rhtc@5>f'
    'xRsN*I`6uT(m0=0X0t2_Wjo74dp(!z!6g7RLqKYyFI+m{(12%3{N7@(+EauU!{1>D7v4<j{AN$6MkBnt8hFg2c)9UsK`g%IBY<h@&N>>zB2@_-7l_yWA&Ssb'
    'p=c|yV1&2#_r-<<<OXct>*@lwio5`Q*2+FtO=yv;TFlklS^m2oS&83d=XC1RG@nKv8er7s_b~T(Yzr%A#Qz{=d}@2&``LR807PR^oRNTBYonId+k`=AS>#gb'
    '`1aA}qqH;Ie6$g}AQQ|zq5rWmOOI+$g52D^Z;gkfO^cxovVXg{g2t|YNl4q~X+b1ZClErp$mEE6VhkXektUl=k90BJKtn?Q;Y5sP(>1=TOjmuwUtt;qnB9{r'
    '30*`HI~t0)fRFQTJcaZ@T%Cn1ku+rzktMNkbh*vOsbFOo{Be)A4Eva<4EetyM>s8r>I!NtiZ#$D*1EembCLh~eKHui_q7Rwb8qU2_risfS?BFno5CgJFPs4+'
    '2}8rUfVH%st*XC{1{)@im(nn^U%ol`Y^WV{r6pNbnY$RPY|8uP#nVM2>+^@?tum`lijdq|s85U^rV;Vl*+jzv&Tmr<;{y=bs`k&X_HA5;EIqKpa*=ma3g<&Y'
    'MaCcg3d1)<GLxOtjSH1<Ttz6xBKie7oS@)GR&jnwG9Fh?ks(}3X5u+2@#T0{-=I{5@5%N{1xDyw)ZD5@Tj<wtYhPTqr`NMV<$X1JpM)Y!?v#+%^$RCSF-|%L'
    '<8afNEmJ-ybh{hRQp`7B2kvmxo>p#a@T<bfAQ;Ngi7O#a&@3_8OPoAYDz`E?AY}sLD{O)|Pm|uHaEawzHT4dWJQHX<;npz8_d}`JTxTivoi~ZqfYl?G8nujz'
    'dDJEABmObZ-?F7$tgm)Q;bCAmIdh-J{EX<o3ESmXjF*3b%!!|HJiZ-wFZ#3Vt+h{Uo1fM;)_g9q<RGp}YWZd&YZrF0YC4vuow6|QNO)QJNm7`{>1cR)Im^$Y'
    'swuL2vMpR&Ty_WNS&?FbO21po(-b>rD7kE4$`q4R5ZjgF*C`T?X_kx5PFAon^f2Qy#~Zj=V7w$S+=hdqK3~MS$vhGBwGl7kz4+<6Wx*cR%oj`9EY(A)|1`I6'
    'rc=h%lt$K61J!bvs${j(bdCtV(@A?wIo4s1HyT@?)*czb6P>TJM&s!Q^uD%bR0)m7M(y!a*a&Dkr%p5)urF$xRAks6{;J(*Y}R0}Y?&M}P;+go_9Siv)APAH'
    'jGu~+d1yvg@SFR{mdi9H%Q>sEA}sYT<-U0TG56(;iDFNEng~xew`x!0P+&7%U&9;r2d5wSpUKqz>~sb`cZ0B<k{rmDWaN2@?u>;zG5iW>U>MwA_pzXs3dVoc'
    'w&gNJxRkg_$BayX<1pp$4xOMrlaUCJL!n=-f+m2fS)d5WVe}XzIUCS;(~Ab}TM%PHAmfhJ0b|CvIgpgAv_Nz`voxe;Ej_0c_v|{B+SuiFUBJAfwJ*)(<B|=*'
    'VaufR)_Lb5lH|X4&}8T<gL%i(&h*E(KaWR_c3rOs+!8R4MY6^P*7!)^n0zjVH5%S#XS2B+GpCfOxkEEf%5|gpU${RVJ$|d0%oyjG)KT0ba}`uOezCS$`>eKQ'
    'n2MY%T|$OFPF=eK5por=)a|M}A28z#i(T+{sz(&e5DN^!^EZ3@FVek}!~Jc{)y1#p&IBv_waV1|G{hNeFQ@jj6UC<(MOn16prCerJ!qe!lC`x^iXul?ZiD|@'
    '8;vvktJ|X)oQ$Sz6z-5i$%n&hGCmRx+@wWBq|x1}Pz<U#AH63AhF`KvZKc`#v{sGqe_QgujamyYe-b$w!mXjNKJYYw&ugED3WHelw%0>OKu1;^US}zfsav6*'
    'R3omk8r42c><IC@8eH<dL50(!gheV&DOZ=?6a^~t%AJHNo6+DMb<6)MyH;PrrBVLmdMJZQabTFD4mRxJmvKP!^7954MK8m!8-~!4q^zTxyv1#&QWpH_yfK0_'
    'Om{%EGTby=$ycLmBNO1IP$Cg5WxeY;-g}W);Bnw_fUVBERjJcyE?dz@aM_|B{jY%6#_O1{i=5!RnC@u*vcN}6V(d;~0$$E2s4k<B0SOSOZA3`A-u9;^iL@dO'
    'eA&LdM4SNvB(r{IZZaersw!U88X{fE8}6+;ETW@_UTTD)h84F84Su36;&>fHs|2iN&CQlr3*+@!Hd)6I)4@8EqmmbEBz!7cCp<69_u}j?ZFuNaPM_F^2_CPT'
    'XjG~=7dBgsyxd5Rt=H%v*p9Uz5DrJ3=gso)9_Xef*#aU~@k<T{(}2bVD-TxQ>18R{ca@cGM<(fdVOLdQ;$Y)PcUb>Cz6W!9f0aB(18Q(j!TVxyV6HC2Pp8uV'
    '0CoQFs*2tf5<f&mUrdO5e^vcG6!E$pej@cbaRO<WcQYYuMLxqSw?eGgv*>+;pOQ+svQ^zmsy+*4>SpzB<^3PLPka?GHu1E!p-%TjHfRMVW`Tm(8Mv<-SO6jA'
    '%%h^CaBmi0?{2C@|E6y~3}hy(Mh|&6IjM#TCAMK;12B%?z;1F*Ve>@LO!#S{DTxd3`DtRANnOsk&BY15D+~}+b>a*$9OW%k?!`3OC`5^CLgM`v6$w7xz<lnA'
    'pQ-Uz@FVnAOX65es`xXLR8evq<(7vE1f9-Y&)E+NM)1y?@6m7aygDOxGnKFpXA<yZvfKBAw12SNy|qctmZz3{3keUdKn%pwry9|oLRXC87{s=I?;PZfUteLM'
    'V-+*qSrJFRApFXM+4T{+E6B6xdX`P2bK{~-9&8BODS2ZeDzESwWWGX2G^tSjtqjMt_GHq&1?g_1Rt15s+T61LZsETh^e=Lr7){xLFNoAUd7L=1So=JyU{$?#'
    'C0To1OQN4go4vKqZ2=e-GBo{P_pr@h_a-BzXVpiEQ}YZc)YGI|d0d6sF-~JlZdCXi3L5!N=U`ptBsYbUK}EbZdGfP$pzR!Jn{fp2LbI2<ggFuHC)FzX`XFX1'
    'gR)HE(7H#}9!B2-H1X9UD;z?nM}d?0F&diu5iUEr*aJ^7^HZPLu(n29xu3RXK?lV|{!{99&Vx#sPw|9S`P>E-GCz?nrkYhhLIG1`Jmkg<XkSG6Qqa8=RVgK!'
    'lx0Tjg_;y&lU_lS64H^}SJ4u-VcYkfp1JOw8&bzO-WKM^C^LUrVbMOig};xE51;SSW;`=5_#X_CLv=#ccyLTHaIQ+<oeg`#%l@#(Ck1T`*>rR*STQJ@8dQh+'
    ';}&^MFr!aw*VSxzJ8R=aH)UVoFemL{h9SFnQMIf)8y1p3&BIY=0!7aJdp#bH@%ifI#qeC-3)TP8zg@DW$9fv^@7wk@Q68X<bmZZBi|ySUx8_ZrB%FZjewLvN'
    'wr@siKD6nxtUCY`(S54=+wL_Jd4ExAD5a(7{#h9(n*JQy;V4UH`(QyYBJKj+2s&Vy2*G`nb*-`YKJM%@--tuyjA{(#5`F0@!#trnI(mReMyZ6{Jv!?ppPxA('
    'z0Mx0p9P^9Oh>)KtMq|On)cWq&PRFUC5hkl5FsR6XItAk%rwvmxGDR?i{7L*p3+9qLO345KO^UWR{boP14(Cbyr{3^gx8GXeEF4Dr*YW2N04#kKczr-)R|pV'
    'GE#Y7JL+D%kcX5KMu$GekkE$t0N<tftn#$_RLh+Ar`Kee#}_sB9H}6lMz67Hmh74<;<B~rC<QWST9NC43e`Bb5%6$M6a1z#7-cXO&$Jfvd+fb6os}nNibI0C'
    '-7GZ^T>O08t>qj7LwwxDN1rtJ=<9TE&!!{sP9IIqa?ugFWKv~yyqXQafd;?Y|6(-R84bIAWIx-3m%X8|qPUc=N8R4wtPpsxM^%Vp{EF>ipQ$=%Zx84A6d20W'
    '5&k#SlCtK&6%h=CH)J-{ns@YVa^~h+_U2#aHf7}hv_0-S=glrd-}-^YRUzB+!g{RkBUm;RQz|zM=j#Q75#O@lSbW+c{bQl6tTUHAxHgbExY+^@hJ>KF%2_-8'
    '-gyN(H`ATbU^LO9Kza6tMv00!y|<6{;#IeD$+oDL?sNHe0Z(egS-b?DF9SeN$T%TbuvKX%g^ueTQ4>x=I)MIJLcz;F(Q8FN0uC<KXDM)wL~s_YP_TS%aTcA9'
    '1vBhzGv$$-LHXAis@@rc-@l&(SkEBjs2vK;C^_n2u06iQQn{|;D!0&#<vHDHy&+1a%@{$nu|}3;$0K0kQ+7G1a(>wAl2+u7U9k}=v|<AoC0-Z<v31QH=h%#|'
    'aSgOyGI_YwCV4_CP*5Is!IrA<O|FPgHoIP3>(DzMeigqp>y__-%O&8a>#@sFEaDevLwtgg^(UG{CCT@i2yXz%>b`a5sH3b)jyj5F27_;pj(5M<-+TG$G(Fxu'
    'd9!~?xko&aP&yEsu2dtdk<Ev6uprH#s`!T$_YAZuBfF)v{8@$nQQOpH^e(Rp<>;Qy1*H<<(yCzfKqxigNfV0JWV~VwhFbi7wu&F3Es_OL61A$f-I~K4dk=W|'
    'TZ@7*9aC6&Mv*|_rkhx##h*%^2u8VYJ}e9q-&YIxNVyCpaT3VOs$PG0u0%%Hz4iN@2V%;+4%@6AjK`0eJxa%?=&kolO>)2FBX3T2)6*{+@rL~U#s2o!_^W_|'
    'k<A9V{0yj;kd1+c-Z2_N#B3{p(5PvH2aV0GjY5)!;skRWHCALGMonNcO`ZrToK1l_CiAX+0A-3Fb}_6Jwf2(sK~g4?r}O5MHCRsU!Z~ms?>9CG?L?a5U@=Xt'
    '!zsiYu|({l7a!$e3<QYcDYdE7=uE<amoPkZx<<>4<OYwqdla&Ve>B$~Z)|MTTR5<bxc9z4ZQK>6XKq}?mZ!WUbXafNM!89NYRBRM%KG(OxA83D&JB~r{B3lL'
    'e1Xvelbp;+K)E8!1KQkV+t)z2=?P|@&U#%cok@5Rwb~z2A>bS7m6NT*I-#&O(n~W&eaP?URThUX8mzHt$a~9OD?Ip2X!4~9yhI+#Ezp_=5PJlrr|KT76Z5CM'
    'b76h<P$kUBKH*U958HKkNkUyF-I%6@gUawuIygMteSUcOWqPuE`sOG-*?W1gefs7YRc-O3t<RrsZe472w>Gw(e75oFqxPpy&o7>QzS(O(zIgQ6r=7<aTb-@P'
    'pZ7YCo_^Zf*!-;5efnslyYcy_pPhf!eTvAxXTx;w;N*0Be?L8Zb9(gV6zWBm(PSH$VZxHj%%j_q`urh>X}gK3_})HyaQx<=j;N?Yqtb4VQ)VH|8KD6O;>&a7'
    'ja=eDl1D8*=5>LOl0do{>;e2wG9WYY3x<C4xohNxi|C3-#v1M}IDG*xar5Wrh6s{mwelU;)Cyue(JCe(k$IF$!!jF26rT`X%!VB{B|(iCe;>8aM>oA^Zqj?('
    'J)k8Nk&GGP+ZcuJ0iL0MYI~6HxQl}!{X*0Wbah5ZjI#%Krjv7gVG_GA9>&>&6Y8i_>X{S&jtgB_O{>9`_7sJP+dO-qza1Xc0LG$7=#vsD6$qF)dq5=g>;Zdn'
    'Nx?wgLMNs>RoK>&zA5@jN(+>_7*EdBj3%jtB`YF`rDfo)xgLAPOZv@_b1KoBt#$T*4Q*(*Cmd><a7|_Vt{@zG{Uyfh?15R%3_otRB=2LOq}vQxXRt%mL#$iv'
    'QLtHAHGOyXK;kqXU?-VFJ9{wa1(HUwnzILRc+MWkVD0wGjiILbXFI)A7&B=saiL+YxG5P0_{#Odu5oKAb0@EJZ<}VgD2AOq(A;RcSp*m)Uni!$!zu0RciwX3'
    'F<fMuQtT;%Z^Os5!D`d%v6r#gLPV6{E3pD41Rr2`TBtgOJ`mlvCvwQV!y;C4Au;GPf>~Kg!5rLSdV717!LBmkJAXgcIa!Xso$H!P#*C#5Vj(m~;dycg^Z&PY'
    '{XCA;Q2eV{X(l<M30pvsB}k|23Lz*EcU`g?C3}0DVUtYQop3}^3Pec*qM)GQ1iIn`#kOn;ik$X)%s0XBJ$r1w@i+-WSFCigJ<s;@v){kpV>OME@^S2yysqAb'
    '(G?)Lv0nPH@d|jwLysw<4xCJCe?r$-lm5Tu{vhu${R7*R>LpL@xTb-@08oJmQOy|3w^d{=E<mq(j|>|s1_1Jg5^AetW>w?hVqhWpz*o7(9BI|4T}{K~WO~GU'
    'D?|ZkmafD?skGTgkLkhiH~t@jwAehHAk%}T*d9QTNxLlz3gAQe9RDcyg8S%IY8|?K;Xq@B+bn!bmwKFh0noXU+`%gWg8pS=yBO>u-}@!Ncj18IVWYo$o!a+~'
    'm=@b#(o^y7r`;doMX~);UT693<9)IH^%C&#VzHJ>0EHMnH_xBpygx~$!d*Q4lZ(sU?<+CCUKZP*q0*2c9ons7#WrxXU0k4Q4ejcIWuituK+%6|!iH&a^zjE!'
    'KxgM`#J2je$65!F#jF%(<=^1B%?v%%ZcoH^_I2#?;5<cSb*RX-Z<{1h?<Y(wMCJd3>DpbtL$)LbFUnvYu6m5XFP`>4d-}=wv*hg4FXfqjydMD%KFlSP<rCnn'
    'rC2*3+Qj1Bk7=>}4pJu`@5QXxzPXe=i|yNy$(S#;zlpJ8-;>yb<4$8betOLVqh+2Y$X?Va2pHT~Bojr>Y#bW89%_A?B)-|b7J=O;=Q1O}L1vSMDy%s~y)<=c'
    'AxK-5Y6=vHZ(-ssgc{w#D=@o@UyY_W&8TyO`6r!pF3%SP;hwPS0_$3S3h~WGd_+%h`goUHb&!5-H=ycBRKJ|v5HN#5_rjW~l@x6Ei*2MoJ}XFFf%M^zd*Z|H'
    'fgez-34K10qM02_DeZ5Zf}Cnfl@w)fPRH}zUo*)Q#lxEwp_A#}8*w(uM^Fb{pK`<T_52kxtSmK}lL2GdHt_PpeM!(j(=4xoHa+VtN$R{<%BZ5si6U))YFE}H'
    'k6^3~sLLoFsW3{VW0`{yMXicJ<$Y|}4eCL8Wh&02lO!pYA7ZgK&T3i11!K;_)Von|hf#0G#u@7OMhfw*oLr5j`D7I^zG#&6J$tWYpu+>!LZKDUsasG6Vbb26'
    '9u<3VD(caIAZ>#|0U;8Qlc~WWfojq=q^4cR$fSb%LP@<2`%^LQ?Ta1dxg~eKYTRf|{z)igbZ~Ov#)&q3N>-iMWZ-P=z^ui4BGN2-!RkBs9V|K#t_s&z^;LNM'
    'G3zH!jJpkDppAQ~OU^0VYTfjh?INld%4~OTbt%W`pplp5U&n_NEc~PG(cA?BYyZBZDR@ol&uQN0-!1?|tNtj_KRh(xMv=5n({UMr;cCAXtygt~!2dE%<yi`f'
    'KNKy#TY#2PVkOQXAx{(XRW{bfKoUt-6|u5~Yu?jc_G7)j%%(5UV}nD>euIcu$f|ss-CaTAApTfBXVvwq_FVqVCe!3k_a4PE;%H<BrGZO>bzAV~To1(HILoHG'
    'Tid~P&}v~%WwGY7MF(o8K>&r^x!aWijYgziY)cbX1GSt<#O3qdud%p|EdmaPdER4jjywkE=m@KD6fg=$r5bk)WmHmhrn3SexQ+>4I7y<QpT?^kG~SjE-{v^g'
    '#pCh_K1v~@Y_SMvIBYE(-NSht1`C^6E253AFn}q9vbVW8X>75oO+7|h{-L^F^3?0RfOP&c8+7Fr%lo=RCuEYh2efLsm<@G|I(W*BnbJv>`%kU}EpHC>BiQpe'
    '%FAMZDC}a<bvA4VLS)vAAlRyFS!H01iL+PF8PQrrbY1xSAMh@$D_L^3`+I&}B0AVMA~X(ty&Q=fsf~P(QIc^TU^atGwLj3GEQ^P?QUx!zKX|{BPp&d;>`?y2'
    'I0VdnlrK{8G%GPdTiez*myw!EK+r7;p3wx(agyK@GD+Z0O_Hbt?1LM2IB#tJ1r3O@RR'
)

def unpack_runtime(folder):
    content = zlib.decompress(base64.b85decode(PAYLOAD))
    if hashlib.sha256(content).hexdigest() != PAYLOAD_SHA256:
        raise ValueError('Sai checksum runtime OCR nhúng')
    for name, code in json.loads(content).items():
        (folder / name).write_text(code, encoding='utf8')

In [ ]:
def log_message(log, line):
    print(line, end='', flush=True)
    log.write(line)
    log.flush()


def process_activity(pid):
    proc = Path('/proc') / str(pid)
    try:
        fields = (proc / 'stat').read_text().rsplit(')', 1)[-1].split()
        cpu = (int(fields[11]) + int(fields[12])) / os.sysconf('SC_CLK_TCK')
        rss = int(fields[21]) * os.sysconf('SC_PAGE_SIZE') / 1024 ** 3
        return f'PID {pid}, state={fields[0]}, CPU tích lũy {cpu:.1f}s, RAM {rss:.2f} GiB'
    except (OSError, ValueError, IndexError):
        return f'PID {pid}'


def run_logged(command, env, log, *, stage, timeout_minutes=None, ready_prefix=None, inference_minutes=None):
    lines = queue.Queue()
    start = time.monotonic()
    deadline = start + timeout_minutes * 60 if timeout_minutes is not None else None
    heartbeat = start + 30
    process = subprocess.Popen(list(map(str, command)), env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf8',
                               errors='replace', bufsize=1, start_new_session=True)

    def read_output():
        try:
            with process.stdout as stdout:
                for line in stdout:
                    lines.put(line)
        finally:
            lines.put(None)

    threading.Thread(target=read_output, daemon=True).start()
    output_done = False
    try:
        while not output_done or process.poll() is None:
            now = time.monotonic()
            try:
                line = lines.get(timeout=1)
            except queue.Empty:
                if now >= heartbeat:
                    log_message(log, f'{stage}: {now - start:.0f}s, {process_activity(process.pid)}.\n')
                    heartbeat = now + 30
            else:
                if line is None:
                    output_done = True
                else:
                    log_message(log, line)
                    heartbeat = now + 30
                    if ready_prefix is not None and line.startswith(ready_prefix):
                        start = time.monotonic()
                        timeout_minutes = inference_minutes
                        deadline = start + inference_minutes * 60
                        stage = 'Inference'
            if deadline is not None and now >= deadline and process.poll() is None:
                message = f'{stage}: quá {timeout_minutes:g} phút. {process_activity(process.pid)}.'
                log_message(log, message + '\n')
                raise TimeoutError(message)
        if process.returncode:
            raise subprocess.CalledProcessError(process.returncode, command)
    finally:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                pass
            except ProcessLookupError:
                pass
            finally:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass

In [ ]:
from collections import Counter

WEIGHT_DIRS = {
    'hunyuan': ('hunyuan',),
    'paddle': ('paddle_vl', 'paddle_layout'),
    'ppocr': ('ppocr_det', 'ppocr_rec'),
    'tonggu': ('tonggu/ocr', 'tonggu/layout'),
}


def find_models(root, wanted):
    required = [item for engine in wanted for item in WEIGHT_DIRS[engine]]
    if root:
        roots = [Path(root), Path(root) / 'models']
    else:
        roots = [path.parent for path in INPUT_ROOT.rglob('models_manifest.json')]
        first = Path(required[0])
        roots += [path.parents[len(first.parts) - 1] for path in INPUT_ROOT.rglob(first.name)
                  if path.is_dir() and path.parts[-len(first.parts):] == first.parts]
    matches = sorted({path.resolve() for path in roots if all((path / name).is_dir() for name in required)})
    if len(matches) != 1:
        raise ValueError(f'Tìm thấy {len(matches)} input phù hợp. Điền MODELS_DIR. Cần {required}')
    return matches[0]


def make_model_configs(models, state, folder):
    base = dict(max_new_tokens=MAX_NEW_TOKENS, repetition_penalty=REPETITION_PENALTY,
                page_timeout=PAGE_TIMEOUT_SECONDS, device='cuda:0')
    results = {}
    for engine in engines:
        args = dict(base, engine=engine)
        if engine == 'hunyuan':
            args.update(model=str(models / 'hunyuan'), dtype='bf16', attention_backend='sdpa',
                        model_min_pixels=262144, model_max_pixels=4194304)
        elif engine == 'ppocr':
            args['paddle_args'] = dict(device='gpu:0', text_detection_model_dir=str(models / 'ppocr_det'),
                                      text_recognition_model_dir=str(models / 'ppocr_rec'))
        elif engine == 'paddle':
            args.update(paddle_version='v1.6', model_min_pixels=200704, model_max_pixels=1048576,
                        paddle_args=dict(device='gpu:0', vl_rec_model_dir=str(models / 'paddle_vl'),
                                         layout_detection_model_dir=str(models / 'paddle_layout'),
                                         use_doc_orientation_classify=False, use_doc_unwarping=False))
        else:
            weights = models / 'tonggu'
            layout = sorted((weights / 'layout').glob('*.pth'))
            if len(layout) != 1 or not (weights / 'ocr/config.json').is_file():
                raise ValueError(f'Thiếu hoặc có nhiều checkpoint TongGu: {weights}')
            config = folder / 'tonggu_paths.json'
            config.write_text(json.dumps({
                'layout_inference': {'detector_checkpoint_path': str(layout[0])},
                'ocr': {'transformers': {'model_path': str(weights / 'ocr')}},
            }, indent=2), encoding='utf8')
            args.update(tonggu_repo=state['tonggu_repo'], tonggu_config=str(config),
                        model_min_pixels=1024, model_max_pixels=1048576)
        results[engine] = args
    return results


models_root = find_models(MODELS_DIR, engines)
environment = dict(os.environ)
for key in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONUSERBASE'):
    environment.pop(key, None)
environment.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1',
                   HF_HUB_DISABLE_TELEMETRY='1', PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK='True',
                   USE_TF='0', USE_FLAX='0', PIP_NO_INDEX='1', TOKENIZERS_PARALLELISM='false',
                   PYTHONUTF8='1', PYTHONNOUSERSITE='1')
if GPU_ID is not None:
    environment['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)
components = [component for component, members in [('paddle', {'paddle', 'ppocr'}), ('tonggu', {'tonggu'})]
              if members.intersection(engines)]
job = dict(tasks=tasks, output=str(run_dir), packages=PACKAGES_DIR, components=components)

with tempfile.TemporaryDirectory(prefix='vtm_retry_') as temporary, (run_dir / 'run.log').open('w', encoding='utf8') as log:
    scratch = Path(temporary)
    unpack_runtime(scratch)
    job_path = run_dir / 'plan.json'
    job_path.write_text(json.dumps(job, ensure_ascii=False, indent=2), encoding='utf8')
    if components:
        run_logged([sys.executable, '-u', scratch / 'setup_runtime.py', job_path],
                   environment, log, stage='Chuẩn bị runtime OCR', timeout_minutes=SETUP_TIMEOUT_MINUTES)
        state = read_json(run_dir / 'runtime.json')
    else:
        state = {}
    job['model_configs'] = make_model_configs(models_root, state, run_dir)
    job_path.write_text(json.dumps(job, ensure_ascii=False, indent=2), encoding='utf8')
    counts = Counter(task['engine'] for task in tasks)
    failures = {}
    for engine in engines:
        executable = (sys.executable if engine == 'hunyuan' else
                      state['tonggu_python'] if engine == 'tonggu' else state['paddle_python'])
        try:
            run_logged([executable, '-u', scratch / 'retry_worker.py', job_path, engine],
                       environment, log, stage=f'Nạp {engine}', timeout_minutes=LOAD_TIMEOUT_MINUTES,
                       ready_prefix='OCR_MODEL_READY',
                       inference_minutes=counts[engine] * (PAGE_TIMEOUT_SECONDS + 15) / 60)
        except (subprocess.CalledProcessError, TimeoutError) as error:
            failures[engine] = str(error)
            log_message(log, f'{engine}: {error}\n')
    if failures:
        (run_dir / 'worker_failures.json').write_text(json.dumps(failures, ensure_ascii=False, indent=2), encoding='utf8')

In [ ]:
from html import escape
from zipfile import ZIP_DEFLATED, ZipFile

from IPython.display import FileLink, HTML


def panel(title, record):
    text = record.get('text') if record else None
    value = 'Thiếu bản ghi' if text is None else ('Chuỗi rỗng' if not text.strip() else text)
    details = ''
    if record:
        flags = ', '.join(record.get('quality_issues', [])) or '—'
        details = f"<p>{len(text or '')} ký tự · hit_max={record.get('hit_max')} · {escape(flags)}</p>"
    return (f'<div style="flex:1;min-width:280px"><strong>{title}</strong>{details}'
            '<pre style="white-space:pre-wrap;max-height:480px;overflow:auto">'
            f'{escape(value)}</pre></div>')


completed = 0
for task in tasks:
    output = run_dir / task['id'] / task['engine']
    candidate = output / 'candidate.json'
    if not candidate.is_file():
        print(f"{task['id']} / {task['engine']} chưa có ứng viên. Xem run.log hoặc error.json.")
        continue
    completed += 1
    if len(tasks) <= 12:
        print(f"{task['id']} · {task['engine']} · {task['reason']}")
        before, after = read_json(output / 'before.json'), read_json(candidate)
        display(HTML('<div style="display:flex;flex-wrap:wrap;gap:16px">'
                     + panel('Checkpoint gốc', before) + panel('Ứng viên chạy lại', after) + '</div>'))
print(f'Có {completed}/{len(tasks)} ứng viên để duyệt.')
archive = run_dir / 'ocr_page_retry.zip'
with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as bundle:
    for path in sorted(run_dir.rglob('*')):
        if path.is_file() and path != archive and not path.name.endswith('.tmp'):
            bundle.write(path, path.relative_to(run_dir))
display(FileLink(str(archive)))